# CKINN versus M1 and M2 on real data, version 2
**Clinical Knowledge-Informed Neural Networks (CKINN) for type 2 diabetes risk: NAGALA (Japan), Rich Healthcare Group (China) and NHANES (USA).**

This notebook carries out `protocol_v2.md`. Version 1 (`CKINN_real_data_study.ipynb`) and its results are the record; every change is an amendment listed in protocol section 2, decided after the version 1 results were seen. `AUDIT.md` gives the reasons.

**Research-integrity rules followed here**
1. Every number in a table, figure or report is computed by a cell from stored outputs; none is typed. The integrity cell near the end checks that no markdown cell holds a result-like number and that every number in the report traces to a stored table.
2. Nothing is tuned to favour CKINN. M1, M2 and CKINN get the same tuning budget; lambda follows a rule declared before any v2 result.
3. No evaluation row is used for fitting, tuning, threshold choice or lambda choice. Development rows are used only for tuning; thresholds come from an inner validation split of each training fold.
4. NULL and negative results are reported as found. Each analysis is labelled P1 (pre-specified in v1), P2 (pre-specified in v2), E (exploratory) or PH (post hoc).
5. Outputs are research risk estimates, not diagnoses. No clinical claim is made.
6. Row-level data and predictions never leave the local cache; the outputs archive holds aggregate tables, figures and reports only.

**How to run.** Set `CKINN_REAL_MODE` to `smoke` (pipeline check on subsamples, under 10 minutes on four cores) or `full` (the reported analysis), `CKINN_DATA_DIR` to the folder with the three data files and `CKINN_OUT_DIR` to an output folder. On Colab, mount Drive first. Work is checkpointed per model fit; re-running the notebook resumes.

In [ ]:
# CELL 01 | setup: imports, versions, run mode, paths, worker processes
import os, sys, json, time, hashlib, platform, re, warnings, zipfile, dataclasses, datetime as _dt
from pathlib import Path
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import numpy as np
import pandas as pd
import torch
import sklearn, xgboost, lightgbm, scipy, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True, warn_only=True)

IN_COLAB = "google.colab" in sys.modules
MODE = os.environ.get("CKINN_REAL_MODE", "full")
assert MODE in ("smoke", "full"), MODE
if IN_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
DATA_DIR = Path(os.environ.get("CKINN_DATA_DIR", "/content/drive/MyDrive/CKINN_Real/data" if IN_COLAB else "data"))
OUT = Path(os.environ.get("CKINN_OUT_DIR", f"outputs_v2_{MODE}")).resolve()
CACHE = OUT / "cache"                       # row-level: frames, task checkpoints, predictions (never archived)
for _d in ("tables", "figures", "report", "latex"):
    (OUT / _d).mkdir(parents=True, exist_ok=True)
(CACHE / "preds").mkdir(parents=True, exist_ok=True)
LIB = Path("ckv2_lib").resolve()
(LIB / "ckv2").mkdir(parents=True, exist_ok=True)
(LIB / "tests").mkdir(parents=True, exist_ok=True)
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
N_WORKERS = int(os.environ.get("CKINN_WORKERS", str(os.cpu_count() or 1)))
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "torch": torch.__version__,
            "sklearn": sklearn.__version__, "xgboost": xgboost.__version__, "lightgbm": lightgbm.__version__, "scipy": scipy.__version__,
            "matplotlib": matplotlib.__version__, "platform": platform.platform()}
print(" | ".join(f"{k} {v}" for k, v in VERSIONS.items()))
print(f"mode={MODE} | data={DATA_DIR} | out={OUT} | workers={N_WORKERS}")
T_START = time.perf_counter()

## Protocol v2 (frozen before any v2 result)
The full protocol is `protocol_v2.md`. In short: two prediction tasks (incident diabetes in NAGALA and RICH; undiagnosed diabetes in NHANES, label corrected by amendment A11), headline predictor tier T2 without glucose or HbA1c, repeated stratified 5-fold cross-validation with 5 repeats and paired network seeds, an equal tuning budget for M1, M2 and CKINN, and two hypothesis families (CKINN versus M2, primary; CKINN versus M1), each with Holm correction across its primary endpoints. H3 and H4 are redefined (amendments A8, A9) and H5 uses sensitivity at matched specificity (A10). Labels: P1 pre-specified in v1; P2 pre-specified in v2; E exploratory; PH post hoc.

The next cells write the analysis library (`ckv2`) to disk so that the model fits can run in parallel worker processes, then test it.

In [ ]:
%%writefile ckv2_lib/ckv2/__init__.py
"""CKINN v2 analysis library (written by the notebook, kept in the repository)."""


In [ ]:
%%writefile ckv2_lib/ckv2/config.py
"""Frozen v2 configuration, sign tables, feature lists and deterministic seeds (protocol_v2.md, sections 3 to 9)."""
import dataclasses
import hashlib
import json
import zlib
from dataclasses import dataclass, field
from typing import Any, Dict, Tuple

SEED = 20261120


@dataclass(frozen=True)
class Config:
    mode: str
    seed: int = SEED
    # development / evaluation split and tuning (v1 split kept; protocol 3, 5)
    dev_frac: float = 0.20
    dev_max_rows: int = 20000
    tune_folds: int = 3
    tune_mlp_configs: int = 32          # M1 and M2: default + 31 random from the 72-point v1 space
    tune_ck_base: int = 24              # CKINN starts from M2's best among its first 24 configurations
    lam_grid: Tuple[float, ...] = (0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0)
    lam_reps: int = 2                   # lambda scored on 3-fold CV repeated twice (6 fits)
    tune_tree_configs: int = 16
    # evaluation design (protocol 6)
    n_splits: int = 5
    n_repeats: int = 5
    n_seeds: int = 5
    # networks (v1 settings, protocol 4)
    max_epochs: int = 40
    patience: int = 4
    batch: int = 1024
    val_frac: float = 0.15
    aug_prob: float = 0.5
    p_panel_drop: float = 0.3
    p_feat_drop: float = 0.1
    viol_tol: float = 1e-3
    winsor: Tuple[float, float] = (0.5, 99.5)
    # thresholds and tests (protocol 6 to 8)
    spec_target: float = 0.90
    sens_target: float = 0.80
    fpr_matched: float = 0.10
    ni_margin: float = 0.01
    alpha: float = 0.05
    n_boot: int = 2000
    n_boot_desc: int = 500
    n_boot_band: int = 200
    min_group_events: int = 20
    # missingness (protocol 8)
    miss_mechs: Tuple[str, ...] = ("mcar", "mar", "mnar")
    miss_rates: Tuple[float, ...] = (0.1, 0.3, 0.5)
    primary_miss: Tuple[str, float] = ("mcar", 0.3)
    # censoring-aware analyses
    horizon: float = 3.0
    nagala_horizon2: float = 5.0
    # competitors
    rf_trees: int = 200
    spline_knots: int = 5
    mi_m: int = 5
    mi_iter: int = 10
    # explanations
    n_explain: int = 1000
    ig_steps: int = 32
    n_pd_rows: int = 500
    sparse_quantile: float = 0.95
    # transport and site analyses
    recal_n: int = 1000
    weight_clip: Tuple[float, float] = (0.1, 10.0)
    site_min_events: int = 100
    # decision curves
    dca_b: Tuple[float, float, float] = (0.005, 0.10, 0.005)
    dca_a: Tuple[float, float, float] = (0.01, 0.20, 0.01)
    nb_points_b: Tuple[float, ...] = (0.02, 0.05)
    nb_points_a: Tuple[float, ...] = (0.05, 0.10)
    # smoke-mode subsampling (0 = all rows)
    max_rows: Dict[str, int] = field(default_factory=lambda: {"NAGALA": 0, "RICH": 0, "NHANES": 0})

    def to_json(self) -> str:
        return json.dumps(dataclasses.asdict(self), sort_keys=True, default=str)


def make_config(mode: str) -> Config:
    if mode == "full":
        return Config(mode="full")
    if mode == "smoke":
        return Config(mode="smoke", tune_mlp_configs=3, tune_ck_base=2, lam_grid=(0.3, 3.0), lam_reps=1, tune_tree_configs=2,
                      n_splits=3, n_repeats=2, n_seeds=2, max_epochs=6, patience=2, n_boot=200, n_boot_desc=100, n_boot_band=50,
                      rf_trees=30, mi_m=2, mi_iter=3, n_explain=200, ig_steps=8, n_pd_rows=150, recal_n=300, site_min_events=20,
                      dev_max_rows=3000, miss_rates=(0.3,), min_group_events=10,
                      max_rows={"NAGALA": 4000, "RICH": 20000, "NHANES": 6000})
    raise ValueError(mode)


# ---- clinical sign table (v1, unchanged) and the sensitivity tables of protocol 8(e) ----
SIGNS_T2 = {"age_years": 1, "bmi_kgm2": 1, "waist_cm": 1, "sbp_mmHg": 1, "family_history_dm": 1, "tg_mmol": 1,
            "hdl_mmol": -1, "exercise_regular": -1}
SIGNS_T3 = dict(SIGNS_T2, fpg_mmol=1, hba1c_pct=1)
SIGNS_CORE = {"age_years": 1, "bmi_kgm2": 1, "waist_cm": 1, "sbp_mmHg": 1, "tg_mmol": 1}
SIGNS_EXT = dict(SIGNS_T2, dbp_mmHg=1, alt_u_l=1, ggt_u_l=1)

LOW_COST = ["age_years", "sex_female", "bmi_kgm2", "waist_cm", "sbp_mmHg", "dbp_mmHg", "smoker", "alcohol_level",
            "exercise_regular", "family_history_dm"]
PANEL = ["tg_mmol", "hdl_mmol", "ldl_mmol", "tc_mmol", "alt_u_l", "ast_u_l", "ggt_u_l"]
GLYC = ["fpg_mmol", "hba1c_pct"]
ALL_FEATURES = LOW_COST + PANEL + GLYC
BINARY = ("sex_female", "smoker", "exercise_regular", "family_history_dm")
LOG_COLS = ("tg_mmol", "alt_u_l", "ast_u_l", "ggt_u_l")
GROUP_OF = {**{c: "demo" for c in ("age_years", "sex_female", "smoker", "alcohol_level", "family_history_dm", "exercise_regular")},
            **{c: "clin" for c in ("bmi_kgm2", "waist_cm", "sbp_mmHg", "dbp_mmHg")},
            **{c: "panel" for c in PANEL}, **{c: "glyc" for c in GLYC}}
GROUPS = ("demo", "clin", "panel", "glyc")
# Post hoc features (protocol_v2.md section 10, PH): history questions in NHANES, the TG/HDL ratio, a cohort indicator for pooled
# training. They are outside ALL_FEATURES, so no pre-specified feature set or the configuration hash changes.
PH_FEATURES = ("prediabetes_told", "hypertension_told", "tg_hdl_ratio", "cohort_rich")
GROUP_OF.update({"prediabetes_told": "demo", "hypertension_told": "demo", "cohort_rich": "demo", "tg_hdl_ratio": "panel"})
BINARY = BINARY + ("prediabetes_told", "hypertension_told", "cohort_rich")
LOG_COLS = LOG_COLS + ("tg_hdl_ratio",)
RANGES = {"age_years": (18, 100), "bmi_kgm2": (12, 70), "waist_cm": (40, 190), "sbp_mmHg": (60, 260), "dbp_mmHg": (30, 160),
          "tg_mmol": (0.05, 60), "hdl_mmol": (0.1, 10), "ldl_mmol": (0.2, 15), "tc_mmol": (1.0, 20), "alt_u_l": (1, 2000),
          "ast_u_l": (1, 2000), "ggt_u_l": (1, 5000), "fpg_mmol": (2.0, 30), "hba1c_pct": (3.0, 18)}

NN_MODELS = ("M1", "M2", "CKINN")
MAIN_MODELS = ("LR", "cLR", "GAM", "RF", "XGB", "XGB_mono", "LGBM", "LGBM_mono", "M1", "M2", "CKINN")
RUN0_EXTRA = ("MI_LR", "LR_aug", "XGB_aug")
MONO_MODELS = ("cLR", "XGB_mono", "LGBM_mono", "CKINN")

MLP_SPACE = {"hidden": [(64, 32), (128, 64), (256, 128)], "dropout": [0.1, 0.2, 0.3], "lr": [1e-3, 3e-3],
             "weight_decay": [1e-4, 1e-3], "enc_dim": [16, 32]}
MLP_DEFAULT = {"hidden": (128, 64), "dropout": 0.2, "lr": 1e-3, "weight_decay": 1e-4, "enc_dim": 32}
TREE_SPACE = {"XGB": {"max_depth": [2, 3, 4, 6], "learning_rate": [0.03, 0.05, 0.1], "n_estimators": [150, 300, 500]},
              "LGBM": {"num_leaves": [7, 15, 31], "learning_rate": [0.03, 0.05, 0.1], "n_estimators": [150, 300, 500]}}
TREE_DEFAULT = {"XGB": {"max_depth": 3, "learning_rate": 0.05, "n_estimators": 300},
                "LGBM": {"num_leaves": 15, "learning_rate": 0.05, "n_estimators": 300}}
C_GRID = [0.01, 0.1, 1.0, 10.0]
RF_GRID = [{"max_depth": d, "min_samples_leaf": l} for d in (6, 10, 14) for l in (10, 20, 50)]


def seed_for(*keys: Any) -> int:
    """Deterministic seed from a tuple of keys (identical to v1)."""
    return zlib.crc32("|".join(map(str, keys)).encode()) % (2 ** 31 - 1)


def config_hash(cfg: Config) -> str:
    frozen = {"config": json.loads(cfg.to_json()), "signs_T2": SIGNS_T2, "signs_T3": SIGNS_T3, "signs_core": SIGNS_CORE,
              "signs_ext": SIGNS_EXT, "main_models": MAIN_MODELS, "run0_extra": RUN0_EXTRA, "mlp_space": MLP_SPACE,
              "tree_space": TREE_SPACE, "c_grid": C_GRID, "rf_grid": RF_GRID,
              "hypotheses": ["H1 AUROC non-inferiority 0.01", "H2 Brier and |slope-1|", "H3 transport gap vs target CV and |CITL|",
                             "H4 across-seed attribution stability", "H5 TPR at FPR 0.10 gaps by sex and age band"],
              "families": ["A: CKINN-M2", "B: CKINN-M1"]}
    return hashlib.sha256(json.dumps(frozen, sort_keys=True, default=str).encode()).hexdigest()


In [ ]:
%%writefile ckv2_lib/ckv2/data.py
"""Loading, harmonisation and splitting of the three cohorts (protocol_v2.md, section 3).

Differences from v1 (each one is an amendment in protocol_v2.md section 2): RICH ALT and AST zeros set to missing (A12);
NHANES label corrected (A11) with the v1 label kept in `label_v1` for the sensitivity analysis.
"""
import hashlib
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

from .config import ALL_FEATURES, GLYC, LOW_COST, PANEL, seed_for


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def find_file(roots: Sequence[Path], patterns: Sequence[str]) -> Path:
    hits = []
    for root in roots:
        if root.exists():
            for pat in patterns:
                hits += sorted(root.glob(pat))
    if not hits:
        raise FileNotFoundError(f"none of {patterns} found in {[str(r) for r in roots]}")
    if len({sha256_file(h) for h in hits}) > 1:
        raise RuntimeError(f"several different files match {patterns}: {[str(h) for h in hits]}; keep one")
    return hits[0]


def read_cached(path: Path, cache_dir: Path) -> pd.DataFrame:
    """xlsx parsing is slow; a csv copy is cached in the (never archived) cache folder."""
    if path.suffix == ".csv":
        return pd.read_csv(path, low_memory=False)
    cache = cache_dir / f"_raw_{sha256_file(path)[:16]}.csv"
    if not cache.exists():
        pd.read_excel(path).to_csv(cache, index=False)
    return pd.read_csv(cache, low_memory=False)


def empty_frame(n: int) -> pd.DataFrame:
    return pd.DataFrame({c: np.full(n, np.nan) for c in ALL_FEATURES})


def load_nagala(raw: pd.DataFrame) -> pd.DataFrame:
    n = raw.copy()
    n.columns = ["id", "sex", "age", "etoh", "fatty_liver", "bmi", "waist", "wc_hi", "bmi25", "phenotype", "fu_days", "alt", "ast", "wt",
                 "exercise", "ggt", "hdl_mg", "hdl", "tc_mg", "tc", "tg_mg", "tg", "a1c", "a1c_mmol", "alcohol", "smoking",
                 "fpg_mg", "fpg", "sbp", "dbp", "incident"]
    assert len(n) == 15464 and n["incident"].sum() == 373, "unexpected NAGALA file"
    d = empty_frame(len(n))
    d["age_years"], d["sex_female"] = n.age, 1 - n.sex                       # file codes 1 = men, 0 = women (AUDIT B2)
    d["bmi_kgm2"], d["waist_cm"], d["sbp_mmHg"], d["dbp_mmHg"] = n.bmi, n.waist, n.sbp, n.dbp
    d["smoker"] = (n.smoking == 3).astype(float)                            # 1 never, 2 past, 3 current
    d["alcohol_level"] = n.alcohol - 1                                       # 1 none ... 4 heavy -> 0..3
    d["exercise_regular"] = n.exercise
    d["tg_mmol"], d["hdl_mmol"], d["tc_mmol"] = n.tg.where(n.tg > 0), n.hdl.where(n.hdl > 0), n.tc
    d["alt_u_l"], d["ast_u_l"], d["ggt_u_l"] = n.alt.where(n.alt > 0), n.ast.where(n.ast > 0), n.ggt
    d["fpg_mmol"], d["hba1c_pct"] = n.fpg, n.a1c
    d["tg_hdl_ratio"] = d.tg_mmol / d.hdl_mmol                          # post hoc feature (PH)
    d["cohort"], d["site"], d["task"] = "NAGALA", "0", "B"
    d["label"] = n.incident.astype(float)
    d["follow_years"] = n.fu_days / 365.25
    d["group_extra"] = "all"
    return d


def load_rich(raw: pd.DataFrame) -> pd.DataFrame:
    r = raw.copy()
    r.columns = ["id", "age", "sex", "site", "ht", "wt", "bmi", "sbp", "dbp", "fpg", "tc", "tg", "hdl", "ldl", "alt", "ast", "bun", "ccr",
                 "fpg_final", "dx_followup", "event", "fu_years", "smoking", "drinking", "family_history"]
    assert len(r) == 211833 and r["event"].sum() == 4174, "unexpected Rich file"
    d = empty_frame(len(r))
    d["age_years"], d["sex_female"] = r.age, (r.sex == 2).astype(float)
    d["bmi_kgm2"], d["sbp_mmHg"], d["dbp_mmHg"] = r.bmi, r.sbp, r.dbp
    d["smoker"] = np.where(r.smoking.isna(), np.nan, (r.smoking == 1).astype(float))   # 1 current, 2 ever, 3 never
    d["family_history_dm"] = r.family_history
    for c_out, c_in in (("tg_mmol", "tg"), ("hdl_mmol", "hdl"), ("ldl_mmol", "ldl"), ("alt_u_l", "alt"), ("ast_u_l", "ast")):
        d[c_out] = r[c_in].where(r[c_in] > 0)                                # exact zeros = not measured (A12)
    d["tc_mmol"], d["fpg_mmol"] = r.tc, r.fpg
    d["tg_hdl_ratio"] = d.tg_mmol / d.hdl_mmol                          # post hoc feature (PH)
    d["fpg_final"] = r.fpg_final                                        # final-visit FPG: post hoc progression analysis only, never a predictor
    d["cohort"], d["site"], d["task"] = "RICH", r.site.astype(int).astype(str), "B"
    d["label"] = r.event.astype(float)
    d["follow_years"] = r.fu_years
    d["group_extra"] = d["site"]
    return d


def load_nhanes(raw: pd.DataFrame) -> pd.DataFrame:
    h = raw.copy()
    h.columns = [c.split("__")[-1] if c != "SEQN" else c for c in h.columns]
    h = h.loc[:, ~h.columns.duplicated(keep="first")]
    h = h[(h.SDDSRVYR >= 1) & (h.RIDAGEYR >= 20)].reset_index(drop=True)       # adults, cycles 1999-2018 (code -1 excluded)
    d = empty_frame(len(h))
    d["age_years"], d["sex_female"] = h.RIDAGEYR, (h.RIAGENDR == 2).astype(float)
    d["bmi_kgm2"], d["waist_cm"] = h.BMXBMI, h.BMXWAIST
    d["sbp_mmHg"] = h.VNAVEBPXSY.fillna(h.BPXSY1)
    d["dbp_mmHg"] = h[["BPXDI1", "BPXDI2", "BPXDI3"]].replace(0, np.nan).mean(axis=1)
    drinks = h.ALQ130.where(h.ALQ130 < 100)
    d["alcohol_level"] = np.where(h.ALQ101 == 2, 0.0, np.where(drinks.isna(), np.nan, np.where(drinks <= 1, 1.0, np.where(drinks <= 2, 2.0, 3.0))))
    d["exercise_regular"] = np.where(h.PAQ650 == 1, 1.0, np.where(h.PAQ650 == 2, 0.0, np.nan))
    d["family_history_dm"] = np.where(h.MCQ300C == 1, 1.0, np.where(h.MCQ300C == 2, 0.0, np.nan))
    d["tg_mmol"], d["hdl_mmol"], d["ldl_mmol"], d["tc_mmol"] = h.LBXTR / 88.57, h.LBDHDD / 38.67, h.LBDLDL / 38.67, h.LBXTC / 38.67
    d["alt_u_l"], d["ast_u_l"], d["ggt_u_l"] = h.LBXSATSI, h.LBXSASSI, h.LBXSGTSI
    fpg_all = h.LBXGLU / 18.016
    d["fpg_mmol"], d["hba1c_pct"] = fpg_all, h.LBXGH
    d["tg_hdl_ratio"] = d.tg_mmol / d.hdl_mmol                          # post hoc features (PH): ratio and history questions
    d["prediabetes_told"] = np.where(h.DIQ160 == 1, 1.0, np.where(h.DIQ160 == 2, 0.0, np.nan))
    d["hypertension_told"] = np.where(h.BPQ020 == 1, 1.0, np.where(h.BPQ020 == 2, 0.0, np.nan))
    # v1 label (kept for the sensitivity analysis)
    known = (h.DIQ010 == 1)
    lab_v1 = np.where(fpg_all.notna() | h.LBXGH.notna(), ((fpg_all.fillna(-1) >= 7.0) | (h.LBXGH.fillna(-1) >= 6.5)).astype(float), np.nan)
    d["label_v1"] = np.where(known, np.nan, lab_v1)
    # v2 label (A11): no diagnosis, no glucose-lowering medication, diagnosis question answered, FPG only after >= 8 h fasting
    med = (h.DIQ050 == 1) | (h.DIQ070 == 1)
    unknown = h.DIQ010.isin([7, 9]) | h.DIQ010.isna()
    fpg_fast = fpg_all.where(h.PHAFSTHR >= 8)
    lab = np.where(fpg_fast.notna() | h.LBXGH.notna(), ((fpg_fast.fillna(-1) >= 7.0) | (h.LBXGH.fillna(-1) >= 6.5)).astype(float), np.nan)
    d["label"] = np.where(known | med | unknown, np.nan, lab)
    d["cohort"], d["site"], d["task"] = "NHANES", h.SDDSRVYR.astype(int).astype(str), "A"
    d["follow_years"] = np.nan
    d["group_extra"] = h.RIDRETH1.map({1: "Mexican American", 2: "Other Hispanic", 3: "Non-Hispanic White", 4: "Non-Hispanic Black",
                                       5: "Other/multiracial"}).fillna("unknown")
    # survey design (A19): combined MEC weight for 10 cycles (WTMEC4YR for 1999-2002), strata and PSU
    w = np.where(h.SDDSRVYR <= 2, h.WTMEC4YR * 2 / 10, h.WTMEC2YR / 10)
    d["svy_w"], d["svy_strata"], d["svy_psu"] = w, h.SDMVSTRA.astype(float), h.SDMVPSU.astype(float)
    return d


def task_frame(d: pd.DataFrame, label_col: str = "label") -> pd.DataFrame:
    t = d[d[label_col].notna()].copy()
    t["raw_row"] = t.index.to_numpy()                       # row identity in the harmonised frame (used to keep splits disjoint)
    t = t.reset_index(drop=True)
    t["age_band"] = np.where(t.age_years < 40, "20-39", np.where(t.age_years < 60, "40-59", "60+"))
    t["sex"] = np.where(t.sex_female == 1, "female", "male")
    t["y"] = t[label_col].astype(int)
    return t


def dev_eval_split(t: pd.DataFrame, cohort_key: str, dev_frac: float) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """The v1 split: stratified, seed_for('dev', cohort). The development rows are never evaluated."""
    idx_dev, idx_eval = train_test_split(np.arange(len(t)), train_size=dev_frac, stratify=t.y, random_state=seed_for("dev", cohort_key))
    return t.iloc[np.sort(idx_dev)].reset_index(drop=True), t.iloc[np.sort(idx_eval)].reset_index(drop=True)


def subsample(t: pd.DataFrame, n: int, key: str) -> pd.DataFrame:
    if not n or len(t) <= n:
        return t
    keep = train_test_split(np.arange(len(t)), train_size=n, stratify=t.y, random_state=seed_for("smoke", key))[0]
    return t.iloc[np.sort(keep)].reset_index(drop=True)


def cap_rows(d: pd.DataFrame, n: int, key: str) -> pd.DataFrame:
    if len(d) <= n:
        return d
    keep = train_test_split(np.arange(len(d)), train_size=n, stratify=d.y, random_state=seed_for("devcap", key))[0]
    return d.iloc[np.sort(keep)].reset_index(drop=True)


def features_for(t: pd.DataFrame, tier: str) -> List[str]:
    cols = LOW_COST + (PANEL if tier in ("T2", "T3") else []) + (GLYC if tier == "T3" else [])
    if tier == "T3" and t.task.iloc[0] == "A":
        raise ValueError("glycaemic predictors define the Task A label: tier T3 is not allowed for detection")
    return [c for c in cols if t[c].notna().mean() > 0.0]


FORBIDDEN = {"A": set(GLYC) | {"known_dm", "label", "label_v1", "y"}, "B": {"label", "y", "follow_years", "event"}}


def assert_no_leakage(features: Sequence[str], task: str, tier: str) -> None:
    bad = set(features) & (FORBIDDEN[task] | (set(GLYC) if tier != "T3" else set()))
    if bad:
        raise AssertionError(f"leakage: forbidden predictor(s) {sorted(bad)} for task {task}, tier {tier}")
    if len(set(features)) != len(features):
        raise AssertionError("duplicated predictors")


def repeated_folds(y: np.ndarray, cohort_key: str, n_splits: int, n_repeats: int) -> List[List[Tuple[np.ndarray, np.ndarray]]]:
    """Repeated stratified K-fold. Repeat 0 uses the v1 seed so that its folds equal v1's for NAGALA and RICH."""
    out = []
    for r in range(n_repeats):
        rs = seed_for("cv", cohort_key) if r == 0 else seed_for("cv", cohort_key, r)
        skf = StratifiedKFold(n_splits, shuffle=True, random_state=rs)
        out.append([(np.sort(i), np.sort(j)) for i, j in skf.split(np.zeros(len(y)), y)])
    return out


In [ ]:
%%writefile ckv2_lib/ckv2/prep.py
"""Preprocessing fitted on training rows only (v1 behaviour, protocol_v2.md section 4)."""
import warnings
from typing import Sequence, Tuple

import numpy as np
import pandas as pd

from .config import BINARY, LOG_COLS


class Prep:
    """Winsorise (0.5/99.5 percentiles), log1p for skewed labs, z-score; X has 0 where missing (= training mean), M is 1 where observed."""

    def __init__(self, features: Sequence[str], winsor: Tuple[float, float] = (0.5, 99.5)):
        self.features = list(features)
        self.winsor = winsor
        self.is_bin = np.array([f in BINARY for f in self.features])
        self.is_log = np.array([f in LOG_COLS for f in self.features])

    def _raw(self, df: pd.DataFrame) -> np.ndarray:
        return df[self.features].to_numpy(float)

    def _warp(self, R, lo, hi):
        T = np.clip(R, lo, hi)
        T[:, self.is_log] = np.log1p(np.maximum(T[:, self.is_log], 0))
        return T

    def _stats(self, R, w=None):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            lo, hi = np.nanpercentile(R, self.winsor[0], axis=0), np.nanpercentile(R, self.winsor[1], axis=0)
            lo = np.where(self.is_bin | np.isnan(lo), -np.inf, lo)
            hi = np.where(self.is_bin | np.isnan(hi), np.inf, hi)
            T = self._warp(R, lo, hi)
            mu, sd = np.nanmean(T, axis=0), np.nanstd(T, axis=0)
        mu = np.where(self.is_bin | ~np.isfinite(mu), 0.0, mu)
        sd = np.where(self.is_bin | ~np.isfinite(sd) | (sd < 1e-8), 1.0, sd)
        return lo, hi, mu, sd

    def fit(self, df: pd.DataFrame) -> "Prep":
        self.lo, self.hi, self.mu, self.sd = self._stats(self._raw(df))
        return self

    def transform(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        R = self._raw(df)
        M = ~np.isnan(R)
        Z = (self._warp(R, self.lo, self.hi) - self.mu) / self.sd
        return np.where(M, Z, 0.0).astype(np.float32), M.astype(np.float32)

    def z_of_raw(self, feature: str, values: np.ndarray) -> np.ndarray:
        """z-scale value of raw values of one feature (used for partial-dependence grids)."""
        j = self.features.index(feature)
        v = np.clip(np.asarray(values, float), self.lo[j], self.hi[j])
        if self.is_log[j]:
            v = np.log1p(np.maximum(v, 0))
        return ((v - self.mu[j]) / self.sd[j]).astype(np.float32)


In [ ]:
%%writefile ckv2_lib/ckv2/missing.py
"""Test-time missingness injection (v1 mechanisms at more rates, A16) and the augmentation used by LR_aug and XGB_aug."""
from typing import List, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.special import expit

from .config import GROUP_OF


def panel_cols(feats: Sequence[str]) -> List[str]:
    return [c for c in feats if GROUP_OF[c] == "panel"]


def _intercept_for_rate(score: np.ndarray, rate: float) -> float:
    return float(brentq(lambda a: expit(a + score).mean() - rate, -30, 30))


def corrupt(df: pd.DataFrame, feats: Sequence[str], mech: str, rate: float, seed: int) -> pd.DataFrame:
    """Copy of df with laboratory-panel predictors removed.
    mcar: each row loses the whole panel with probability `rate`. mar: same, more often for younger and leaner people.
    mnar: each lab value removed on its own, more often when the value is low."""
    rng = np.random.default_rng(seed)
    out, cols, n = df.copy(), panel_cols(feats), len(df)
    if not cols or rate <= 0:
        return out
    if mech in ("mcar", "mar"):
        if mech == "mcar":
            drop = rng.random(n) < rate
        else:
            za = ((df.age_years - df.age_years.mean()) / df.age_years.std()).fillna(0).to_numpy()
            zb = ((df.bmi_kgm2 - df.bmi_kgm2.mean()) / df.bmi_kgm2.std()).fillna(0).to_numpy()
            s = -0.8 * (za + zb)
            drop = rng.random(n) < expit(_intercept_for_rate(s, rate) + s)
        out.loc[drop, cols] = np.nan
    elif mech == "mnar":
        for c in cols:
            v = df[c].to_numpy(float)
            z = np.nan_to_num((v - np.nanmean(v)) / (np.nanstd(v) + 1e-9))
            s = -1.0 * z
            out.loc[rng.random(n) < expit(_intercept_for_rate(s, rate) + s), c] = np.nan
    else:
        raise ValueError(mech)
    return out


def augment_frame(df: pd.DataFrame, y: np.ndarray, feats: Sequence[str], seed: int, aug_prob: float, p_panel: float,
                  p_feat: float) -> Tuple[pd.DataFrame, np.ndarray]:
    """v1 augmentation for tree and linear baselines: append a corrupted copy of a share of the training rows."""
    rng = np.random.default_rng(seed)
    pick = np.flatnonzero(rng.random(len(df)) < aug_prob)
    cp = corrupt(df.iloc[pick], feats, "mcar", p_panel, seed + 1)
    for c in feats:
        if GROUP_OF[c] == "clin":
            cp.loc[rng.random(len(cp)) < p_feat, c] = np.nan
    return pd.concat([df, cp], ignore_index=True), np.r_[y, y[pick]]


def scenarios(cfg) -> List[Tuple[str, float]]:
    return [(m, r) for m in cfg.miss_mechs for r in cfg.miss_rates]


def scen_tag(mech: str, rate: float) -> str:
    return f"{mech}{int(round(rate * 100))}"


In [ ]:
%%writefile ckv2_lib/ckv2/nets.py
"""Neural networks M1, M2 and CKINN: architecture, clinical penalty, augmentation, training and attributions (protocol_v2.md, section 4)."""
from typing import Dict, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn

from .config import BINARY, GROUP_OF, GROUPS


def set_seed(seed: int) -> None:
    np.random.seed(seed % (2 ** 32 - 1))
    torch.manual_seed(seed)


class Net(nn.Module):
    """Group encoders (inputs value*mask and mask) -> trunk -> one logit. use_mask=False feeds a mask of ones (M1).
    The sign table is stored in buffers read by the penalty; it adds no trainable parameter."""

    def __init__(self, features: Sequence[str], hidden=(128, 64), enc_dim: int = 32, dropout: float = 0.2,
                 signs: Optional[Dict[str, int]] = None, use_mask: bool = True):
        super().__init__()
        self.features, self.use_mask = list(features), use_mask
        groups = {g: [i for i, f in enumerate(self.features) if GROUP_OF[f] == g] for g in GROUPS}
        self.groups = {g: ix for g, ix in groups.items() if ix}
        self.enc = nn.ModuleDict({g: nn.Sequential(nn.Linear(2 * len(ix), enc_dim), nn.ReLU()) for g, ix in self.groups.items()})
        layers, d = [], enc_dim * len(self.groups)
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        self.trunk, self.head = nn.Sequential(*layers), nn.Linear(d, 1)
        signs = signs or {}
        self.signs = {f: int(s) for f, s in signs.items() if f in self.features and s != 0}
        self.register_buffer("sign_vec", torch.tensor([float(self.signs.get(f, 0)) for f in self.features]))
        self.register_buffer("is_binary", torch.tensor([f in BINARY for f in self.features], dtype=torch.bool))

    def forward(self, X: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
        if not self.use_mask:
            M = torch.ones_like(M)
        h = torch.cat([self.enc[g](torch.cat([X[:, ix] * M[:, ix], M[:, ix]], 1)) for g, ix in self.groups.items()], 1)
        return self.head(self.trunk(h)).squeeze(1)


def clinical_penalty(model: nn.Module, X: torch.Tensor, M: torch.Tensor, signs: torch.Tensor, is_binary: torch.Tensor,
                     create_graph: bool = False, tol: float = 1e-3) -> Tuple[torch.Tensor, float]:
    """Squared hinge on sign-adjusted derivatives (continuous constrained columns, autograd) and on finite differences
    (binary constrained columns), observed entries only, averaged over observed constrained entries. Whole batch (A5).
    Returns (penalty, violation rate). Runs in eval mode so that dropout does not enter the derivative."""
    was = model.training
    model.eval()
    try:
        cont = ((signs != 0) & ~is_binary).to(X.dtype)
        bin_cols = torch.nonzero((signs != 0) & is_binary).flatten().tolist()
        Xg = X.detach().clone().requires_grad_(True)
        z = model(Xg, M)
        (grad,) = torch.autograd.grad(z.sum(), Xg, create_graph=create_graph, retain_graph=True)
        adj, obs = signs * grad, M * cont
        hinge = (torch.relu(-adj) ** 2 * obs).sum()
        viol, count = ((adj.detach() < -tol).to(X.dtype) * obs).sum(), obs.sum()
        if bin_cols:
            k = X.shape[0]
            flips = []
            for j in bin_cols:
                Xf = X.detach().clone()
                Xf[:, j] = 1.0 - Xf[:, j]
                flips.append(Xf)
            zf = model(torch.cat(flips), M.repeat(len(flips), 1))
            for i, j in enumerate(bin_cols):
                toward_one = 1.0 - 2.0 * X[:, j].detach()
                fd, ob = signs[j] * toward_one * (zf[i * k:(i + 1) * k] - z), M[:, j]
                hinge = hinge + (torch.relu(-fd) ** 2 * ob).sum()
                viol, count = viol + ((fd.detach() < -tol).to(X.dtype) * ob).sum(), count + ob.sum()
    finally:
        model.train(was)
    total = count.clamp_min(1.0)
    return hinge / total, float(viol / total)


def augment(xb, mb, gen, panel_cols, other_cols, aug_prob, p_panel, p_feat):
    """Training-time missingness augmentation (v1): a share of rows lose the whole lab panel and scattered clinical values."""
    n = xb.shape[0]
    rows = (torch.rand(n, generator=gen) < aug_prob).float().unsqueeze(1)
    drop = torch.zeros_like(mb)
    if panel_cols:
        drop[:, panel_cols] = (torch.rand(n, 1, generator=gen) < p_panel).float().expand(-1, len(panel_cols))
    if other_cols:
        drop[:, other_cols] = torch.maximum(drop[:, other_cols], (torch.rand(n, len(other_cols), generator=gen) < p_feat).float())
    mb2 = mb * (1 - rows * drop)
    return xb * mb2, mb2


def weighted_val_score(y: np.ndarray, p: np.ndarray, w: Optional[np.ndarray] = None) -> float:
    """Log loss + Brier (the v1 validation score), optionally weighted."""
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    w = np.ones(len(y)) if w is None else np.asarray(w, float)
    ll = -(w * (y * np.log(p) + (1 - y) * np.log(1 - p))).sum() / w.sum()
    return float(ll + (w * (p - y) ** 2).sum() / w.sum())


def predict_logit(net: nn.Module, X: np.ndarray, M: np.ndarray, batch: int = 65536) -> np.ndarray:
    net.eval()
    out = []
    with torch.no_grad():
        for s in range(0, len(X), batch):
            out.append(net(torch.as_tensor(X[s:s + batch]), torch.as_tensor(M[s:s + batch])).numpy())
    return np.concatenate(out).astype(np.float64) if out else np.zeros(0)


def train_net(net: Net, tr, va, hp: dict, cfg, seed: int, lam: float = 0.0, aug: bool = False,
              sign_override: Optional[torch.Tensor] = None) -> dict:
    """AdamW, gradient clipping at 1, early stopping on validation log loss + Brier; restores the best weights.
    tr and va are (X, M, y, w). The penalty uses the net's sign buffer unless sign_override is given."""
    Xtr, Mtr, ytr, wtr = [torch.as_tensor(np.asarray(a, np.float32)) for a in tr]
    Xva, Mva, yva, wva = va
    set_seed(seed)
    gen = torch.Generator().manual_seed(seed + 1)
    opt = torch.optim.AdamW(net.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    panel_cols = [i for i, f in enumerate(net.features) if GROUP_OF[f] in ("panel", "glyc")]
    other_cols = [i for i, f in enumerate(net.features) if GROUP_OF[f] == "clin"]
    signs = net.sign_vec if sign_override is None else sign_override
    use_pen = lam > 0 and bool((signs != 0).any())
    best, best_state, bad, hist = np.inf, None, 0, {"val": [], "pen": [], "viol": []}
    n = len(ytr)
    for epoch in range(cfg.max_epochs):
        net.train()
        perm = torch.randperm(n, generator=gen)
        pen_sum, viol_sum = 0.0, 0.0
        for s in range(0, n, cfg.batch):
            ix = perm[s:s + cfg.batch]
            xb, mb, yb, wb = Xtr[ix], Mtr[ix], ytr[ix], wtr[ix]
            if aug:
                xb, mb = augment(xb, mb, gen, panel_cols, other_cols, cfg.aug_prob, cfg.p_panel_drop, cfg.p_feat_drop)
            loss = (nn.functional.binary_cross_entropy_with_logits(net(xb, mb), yb, reduction="none") * wb).sum() / wb.sum().clamp_min(1e-12)
            if use_pen:
                pen, vr = clinical_penalty(net, xb, mb, signs, net.is_binary, create_graph=True, tol=cfg.viol_tol)
                loss = loss + lam * pen
                pen_sum += float(pen.detach()) * len(ix)
                viol_sum += vr * len(ix)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        pv = 1 / (1 + np.exp(-predict_logit(net, Xva, Mva)))
        sc = weighted_val_score(yva, pv, wva)
        hist["val"].append(sc)
        hist["pen"].append(pen_sum / n)
        hist["viol"].append(viol_sum / n)
        if sc < best - 1e-5:
            best, bad, best_state = sc, 0, {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg.patience:
                break
    net.load_state_dict(best_state)
    net.eval()
    hist["epochs"], hist["best"] = len(hist["val"]), best
    return hist


def _logit_grad(net: nn.Module, X: torch.Tensor, M: torch.Tensor):
    Xg = X.clone().requires_grad_(True)
    z = net(Xg, M)
    (g,) = torch.autograd.grad(z.sum(), Xg)
    return z.detach(), g.detach()


def integrated_gradients(net: nn.Module, X: np.ndarray, M: np.ndarray, steps: int) -> np.ndarray:
    """IG of the logit with a zero baseline on the z-scale (the training mean), mask held fixed; midpoint rule."""
    net.eval()
    Xt, Mt = torch.as_tensor(X), torch.as_tensor(M)
    tot = torch.zeros_like(Xt)
    for s in range(steps):
        _, g = _logit_grad(net, Xt * ((s + 0.5) / steps), Mt)
        tot += g
    return (Xt * tot / steps).numpy()


def violation_rows(net: nn.Module, X: np.ndarray, M: np.ndarray, signs: Dict[str, int], features: Sequence[str], tol: float,
                   attributions: Optional[np.ndarray] = None) -> Dict[str, np.ndarray]:
    """Per row (v1 manipulation-check statistics): wrong-sign derivative count and observed constrained count; integrated-gradient
    direction consistency on rows where the constrained value is away from the mean."""
    net.eval()
    Xt, Mt = torch.as_tensor(X), torch.as_tensor(M)
    sg = torch.tensor([float(signs.get(f, 0)) for f in features])
    isb = torch.tensor([f in BINARY for f in features], dtype=torch.bool)
    ctrl, cont = sg != 0, (sg != 0) & ~isb
    z0, g0 = _logit_grad(net, Xt, Mt)
    obs = Mt * cont.float()
    viol = ((sg * g0 < -tol).float() * obs).sum(1)
    vobs = obs.sum(1)
    for j in torch.nonzero(ctrl & isb).flatten().tolist():
        Xf = Xt.clone()
        Xf[:, j] = 1 - Xf[:, j]
        with torch.no_grad():
            zf = net(Xf, Mt)
        fd = sg[j] * (1 - 2 * Xt[:, j]) * (zf - z0)
        viol += (fd < -tol).float() * Mt[:, j]
        vobs += Mt[:, j]
    out = {"viol": viol.numpy(), "vobs": vobs.numpy()}
    if attributions is not None:
        A = torch.as_tensor(attributions)
        tested = ctrl.float() * Mt * ((~isb).float() * (Xt.abs() > 0.25).float() + isb.float() * (Xt == 1).float())
        cons = ((torch.sign(A) * sg * torch.sign(Xt)) >= 0).float() * tested
        out["ig_cons"], out["ig_n"] = cons.sum(1).numpy(), tested.sum(1).numpy()
    return out


In [ ]:
%%writefile ckv2_lib/ckv2/models.py
"""Every model behind one interface (protocol_v2.md, section 4). fit(train frame, labels, weights) -> predict(frame) -> probabilities.

Pairing (A1): the inner 85/15 split, the row order and the network seed depend only on `seed`, never on the model name, so models
fitted in the same fold with the same seed see identical data, and CKINN with lambda 0 equals M2.
"""
import math
import warnings
from typing import Dict, Optional, Sequence

import numpy as np
import pandas as pd
import torch
from scipy.optimize import minimize
from scipy.special import expit, logit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import SplineTransformer

from .config import BINARY, MLP_DEFAULT, NN_MODELS
from .missing import augment_frame
from .nets import Net, predict_logit, train_net
from .prep import Prep

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


def inner_split(y: np.ndarray, seed: int, val_frac: float):
    idx = np.arange(len(y))
    itr, iva = train_test_split(idx, test_size=val_frac, stratify=y, random_state=seed % (2 ** 32 - 1))
    return np.sort(itr), np.sort(iva)


def threshold_for_spec(y: np.ndarray, p: np.ndarray, spec: float) -> float:
    """Lowest threshold with specificity >= spec (v1 rule)."""
    neg = np.sort(p[y == 0])
    if len(neg) == 0:
        return 0.5
    k = min(int(math.ceil(spec * len(neg))), len(neg)) - 1
    return float(np.nextafter(neg[k], 1.0))


def threshold_for_sens(y: np.ndarray, p: np.ndarray, sens: float) -> float:
    """Highest threshold with sensitivity >= sens: at least ceil(sens * n_pos) positives have p >= threshold."""
    pos = np.sort(p[y == 1])[::-1]
    if len(pos) == 0:
        return 0.5
    k = min(int(math.ceil(sens * len(pos))), len(pos)) - 1
    return float(pos[k])


class ConstrainedLR:
    """L2 logistic regression with sign bounds on chosen coefficients; same objective as scikit-learn:
    0.5 * ||beta||^2 + C * sum_i w_i * logloss_i (intercept unpenalised). L-BFGS-B."""

    def __init__(self, C: float, bounds_sign: Sequence[int]):
        self.C, self.bounds_sign = C, list(bounds_sign)

    def fit(self, X: np.ndarray, y: np.ndarray, w: Optional[np.ndarray] = None):
        n, p = X.shape
        w = np.ones(n) if w is None else np.asarray(w, float)
        Xa = np.c_[np.ones(n), X]

        def f(theta):
            z = Xa @ theta
            ll = np.logaddexp(0, z) - y * z
            g = Xa.T @ (w * (expit(z) - y))
            obj = self.C * (w * ll).sum() + 0.5 * (theta[1:] ** 2).sum()
            grad = self.C * g
            grad[1:] += theta[1:]
            return obj, grad

        b = [(None, None)] + [(0, None) if s > 0 else (None, 0) if s < 0 else (None, None) for s in self.bounds_sign]
        x0 = np.zeros(p + 1)
        x0[0] = logit(np.clip((w * y).sum() / w.sum(), 1e-4, 1 - 1e-4))
        res = minimize(f, x0, jac=True, method="L-BFGS-B", bounds=b, options={"maxiter": 2000, "gtol": 1e-8})
        self.intercept_, self.coef_ = res.x[0], res.x[1:]
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        p = expit(self.intercept_ + X @ self.coef_)
        return np.c_[1 - p, p]


class Fitted:
    """name in LR, cLR, GAM, RF, XGB, XGB_mono, LGBM, LGBM_mono, MI_LR, LR_aug, XGB_aug, M1, M2, CKINN, and the post hoc SVM."""

    def __init__(self, name: str, features: Sequence[str], hp: dict, seed: int, cfg, signs: Optional[Dict[str, int]] = None,
                 lam: float = 0.0):
        self.name, self.features, self.hp, self.seed, self.cfg = name, list(features), dict(hp or {}), int(seed), cfg
        self.signs = dict(signs or {})
        self.lam = float(lam)
        self.thr_spec, self.thr_sens, self.hist = 0.5, 0.5, {}

    # ------------------------------------------------------------------ fit
    def fit(self, dtr: pd.DataFrame, ytr: np.ndarray, sw: Optional[np.ndarray] = None) -> "Fitted":
        ytr = np.asarray(ytr).astype(int)
        itr, iva = inner_split(ytr, self.seed, self.cfg.val_frac)
        a, b = dtr.iloc[itr].reset_index(drop=True), dtr.iloc[iva].reset_index(drop=True)
        ya, yb = ytr[itr], ytr[iva]
        wa = None if sw is None else np.asarray(sw, float)[itr]
        wb = None if sw is None else np.asarray(sw, float)[iva]
        if self.name in NN_MODELS:
            self._fit_net(a, ya, wa, b, yb, wb)
        else:
            self._fit_other(a, ya, wa, b, yb)
        pb = self.predict(b)
        self.thr_spec = threshold_for_spec(yb, pb, self.cfg.spec_target)
        self.thr_sens = threshold_for_sens(yb, pb, self.cfg.sens_target)
        return self

    def _fit_net(self, a, ya, wa, b, yb, wb):
        cfg = self.cfg
        self.prep = Prep(self.features, cfg.winsor).fit(a)
        Xa, Ma = self.prep.transform(a)
        Xb, Mb = self.prep.transform(b)
        hp = {**MLP_DEFAULT, **self.hp}
        torch.manual_seed(self.seed)
        self.net = Net(self.features, tuple(hp["hidden"]), hp["enc_dim"], hp["dropout"],
                       self.signs if self.name == "CKINN" else None, use_mask=(self.name != "M1"))
        prev = float(np.average(ya, weights=wa)) if wa is not None else float(ya.mean())
        with torch.no_grad():
            self.net.head.bias.fill_(float(logit(np.clip(prev, 1e-4, 1 - 1e-4))))
        wa_ = np.ones(len(ya), np.float32) if wa is None else wa.astype(np.float32)
        wb_ = None if wb is None else wb
        self.hist = train_net(self.net, (Xa, Ma, ya, wa_), (Xb, Mb, yb, wb_), hp, cfg, self.seed,
                              lam=self.lam if self.name == "CKINN" else 0.0, aug=(self.name != "M1"))

    def _design_lr(self, df: pd.DataFrame) -> np.ndarray:
        return self.prep.transform(df)[0].astype(np.float64)

    def _design_gam(self, df: pd.DataFrame) -> np.ndarray:
        X, M = self.prep.transform(df)
        cols = []
        for j, f in enumerate(self.features):
            if f in BINARY:
                cols.append(X[:, [j]])
            else:
                cols.append(self.splines[j].transform(X[:, [j]].astype(np.float64)))
            if self.miss_ind[j]:
                cols.append(1.0 - M[:, [j]])
        return np.hstack(cols).astype(np.float64)

    def _fit_other(self, a, ya, wa, b, yb):
        name, hp, seed, cfg = self.name, self.hp, self.seed, self.cfg
        mono = tuple(int(self.signs.get(f, 0)) for f in self.features)
        if name in ("LR_aug", "XGB_aug"):
            a, ya = augment_frame(a, ya, self.features, seed, cfg.aug_prob, cfg.p_panel_drop, cfg.p_feat_drop)
            wa = None
        if name in ("LR", "LR_aug", "cLR", "GAM", "MI_LR"):
            self.prep = Prep(self.features, cfg.winsor).fit(a)
        if name in ("LR", "LR_aug"):
            self.model = LogisticRegression(C=hp["C"], max_iter=3000).fit(self._design_lr(a), ya, sample_weight=wa)
        elif name == "cLR":
            self.model = ConstrainedLR(hp["C"], mono).fit(self._design_lr(a), ya, wa)
        elif name == "GAM":
            X, M = self.prep.transform(a)
            self.splines, self.miss_ind = {}, []
            for j, f in enumerate(self.features):
                self.miss_ind.append(bool((M[:, j] == 0).any()))
                if f not in BINARY:
                    obs = M[:, j] == 1
                    self.splines[j] = SplineTransformer(n_knots=cfg.spline_knots, degree=3, extrapolation="constant",
                                                        knots="quantile").fit(X[obs, j:j + 1].astype(np.float64))
            self.model = LogisticRegression(C=hp["C"], max_iter=3000).fit(self._design_gam(a), ya, sample_weight=wa)
        elif name == "MI_LR":
            from sklearn.experimental import enable_iterative_imputer  # noqa: F401
            from sklearn.impute import IterativeImputer
            Xa, Ma = self.prep.transform(a)
            Za = np.where(Ma == 1, Xa, np.nan).astype(np.float64)
            keep = ~np.all(np.isnan(Za), axis=0)
            self.mi_keep = keep
            self.imputers, self.mi_models = [], []
            for m in range(cfg.mi_m):
                imp = IterativeImputer(sample_posterior=True, max_iter=cfg.mi_iter, random_state=(seed + 1009 * m) % (2 ** 32 - 1))
                Zi = imp.fit_transform(Za[:, keep])
                self.imputers.append(imp)
                self.mi_models.append(LogisticRegression(C=hp["C"], max_iter=3000).fit(Zi, ya, sample_weight=wa))
        elif name == "SVM":
            from sklearn.kernel_approximation import Nystroem
            from sklearn.svm import LinearSVC
            self.prep = Prep(self.features, cfg.winsor).fit(a)
            self.kmap = Nystroem(kernel="rbf", n_components=min(300, len(a)), random_state=seed % (2 ** 31 - 1)).fit(self._design_lr(a))
            self.model = LinearSVC(C=hp["C"], dual=False, max_iter=5000).fit(self.kmap.transform(self._design_lr(a)), ya, sample_weight=wa)
            db = self.model.decision_function(self.kmap.transform(self._design_lr(b)))
            self.platt = LogisticRegression(C=1e6, max_iter=1000).fit(db.reshape(-1, 1), yb)
        elif name == "RF":
            R = a[self.features].to_numpy(float)
            self.med = np.nan_to_num(np.nanmedian(R, axis=0), nan=0.0)
            self.model = RandomForestClassifier(n_estimators=cfg.rf_trees, max_depth=hp["max_depth"], min_samples_leaf=hp["min_samples_leaf"],
                                                max_features="sqrt", n_jobs=1, random_state=seed % (2 ** 32 - 1)).fit(
                np.where(np.isnan(R), self.med, R), ya, sample_weight=wa)
        elif name in ("XGB", "XGB_mono", "XGB_aug"):
            import xgboost as xgb
            kw = {"monotone_constraints": "(" + ",".join(map(str, mono)) + ")"} if name == "XGB_mono" else {}
            self.model = xgb.XGBClassifier(n_estimators=hp["n_estimators"], max_depth=hp["max_depth"], learning_rate=hp["learning_rate"],
                                           subsample=0.8, colsample_bytree=0.8, tree_method="hist", n_jobs=1, random_state=seed % (2 ** 31 - 1),
                                           eval_metric="logloss", verbosity=0, **kw)
            self.model.fit(a[self.features].to_numpy(float), ya, sample_weight=wa)
        elif name in ("LGBM", "LGBM_mono"):
            import lightgbm as lgb
            kw = {"monotone_constraints": list(mono)} if name == "LGBM_mono" else {}
            self.model = lgb.LGBMClassifier(n_estimators=hp["n_estimators"], num_leaves=hp["num_leaves"], learning_rate=hp["learning_rate"],
                                            subsample=0.8, subsample_freq=1, colsample_bytree=0.8, n_jobs=1, random_state=seed % (2 ** 31 - 1),
                                            verbose=-1, **kw)
            self.model.fit(a[self.features].to_numpy(float), ya, sample_weight=wa)
        else:
            raise ValueError(name)

    # -------------------------------------------------------------- predict
    def predict_arrays(self, X: np.ndarray, M: np.ndarray) -> np.ndarray:
        return expit(predict_logit(self.net, X, M))

    def predict(self, df: pd.DataFrame) -> np.ndarray:
        name = self.name
        if name in NN_MODELS:
            X, M = self.prep.transform(df)
            return self.predict_arrays(X, M)
        if name in ("LR", "LR_aug", "cLR"):
            return self.model.predict_proba(self._design_lr(df))[:, 1]
        if name == "GAM":
            return self.model.predict_proba(self._design_gam(df))[:, 1]
        if name == "SVM":
            return self.platt.predict_proba(self.model.decision_function(self.kmap.transform(self._design_lr(df))).reshape(-1, 1))[:, 1]
        if name == "MI_LR":
            X, M = self.prep.transform(df)
            Z = np.where(M == 1, X, np.nan).astype(np.float64)[:, self.mi_keep]
            return np.mean([m.predict_proba(imp.transform(Z))[:, 1] for imp, m in zip(self.imputers, self.mi_models)], axis=0)
        R = df[self.features].to_numpy(float)
        if name == "RF":
            return self.model.predict_proba(np.where(np.isnan(R), self.med, R))[:, 1]
        return self.model.predict_proba(R)[:, 1]

    def info(self) -> dict:
        h = self.hist or {}
        return {"epochs": h.get("epochs"), "best_val": h.get("best"), "final_pen": (h.get("pen") or [None])[-1],
                "thr_spec": self.thr_spec, "thr_sens": self.thr_sens, "lam": self.lam}


In [ ]:
%%writefile ckv2_lib/ckv2/metrics.py
"""Metrics, weighted for the bootstrap (protocol_v2.md, sections 6 to 8).

Every metric accepts participant weights w (bootstrap counts; ones for the point estimate). Rank-based metrics take a `Ranked`
object so that the predictions are sorted once and each bootstrap draw costs O(n).
"""
import math
from typing import Dict, Iterator, Optional, Sequence, Tuple

import numpy as np
from scipy.special import expit, logit
from scipy.stats import norm, t as t_dist

EPS = 1e-6


def clip(p):
    return np.clip(np.asarray(p, float), EPS, 1 - EPS)


class Ranked:
    """Predictions sorted once (ascending) with tie groups, for O(n) weighted rank metrics."""

    def __init__(self, p: np.ndarray, y: np.ndarray):
        p = np.asarray(p, float)
        self.order = np.argsort(p, kind="mergesort")
        ps = p[self.order]
        self.starts = np.r_[0, np.flatnonzero(np.diff(ps) > 0) + 1]
        self.ys = np.asarray(y, float)[self.order]
        self.n = len(p)

    def groups(self, w: Optional[np.ndarray]):
        ws = np.ones(self.n) if w is None else np.asarray(w, float)[self.order]
        pos = np.add.reduceat(ws * self.ys, self.starts)
        neg = np.add.reduceat(ws * (1 - self.ys), self.starts)
        return pos, neg


def auroc_w(r: Ranked, w: Optional[np.ndarray] = None) -> float:
    """Weighted AUROC with ties counted one half (equals roc_auc_score for unit weights)."""
    pos, neg = r.groups(w)
    P, N = pos.sum(), neg.sum()
    if P <= 0 or N <= 0:
        return float("nan")
    below = np.cumsum(neg) - neg
    return float((pos * (below + 0.5 * neg)).sum() / (P * N))


def tpr_at_fpr_w(r: Ranked, fpr: float, w: Optional[np.ndarray] = None) -> float:
    """True-positive rate at a given false-positive rate on the (weighted) ROC curve, linear interpolation within tie groups."""
    pos, neg = r.groups(w)
    pos, neg = pos[::-1], neg[::-1]                       # descending score
    P, N = pos.sum(), neg.sum()
    if P <= 0 or N <= 0:
        return float("nan")
    F = np.r_[0.0, np.cumsum(neg) / N]
    T = np.r_[0.0, np.cumsum(pos) / P]
    i = int(np.searchsorted(F, fpr, side="left"))
    if i == 0:
        return 0.0
    if i >= len(F):
        return 1.0
    f0, f1 = F[i - 1], F[i]
    return float(T[i - 1] + (T[i] - T[i - 1]) * ((fpr - f0) / (f1 - f0) if f1 > f0 else 1.0))


def brier_w(y, p, w=None) -> float:
    w = np.ones(len(y)) if w is None else w
    return float((w * (np.asarray(p, float) - y) ** 2).sum() / w.sum())


def calib_w(y, p, w=None, iters: int = 25, start: Tuple[float, float] = (0.0, 1.0)) -> Tuple[float, float]:
    """Logistic recalibration y ~ a + b * logit(p) (weighted Newton). Returns (intercept, slope); ideal (0, 1)."""
    z, y = logit(clip(p)), np.asarray(y, float)
    w = np.ones(len(y)) if w is None else np.asarray(w, float)
    a, b = start
    for _ in range(iters):
        pr = expit(a + b * z)
        v = w * pr * (1 - pr) + 1e-12
        r = w * (y - pr)
        g0, g1 = r.sum(), (r * z).sum()
        h00, h01, h11 = v.sum(), (v * z).sum(), (v * z * z).sum()
        det = h00 * h11 - h01 ** 2
        if det <= 0:
            break
        da, db = (h11 * g0 - h01 * g1) / det, (-h01 * g0 + h00 * g1) / det
        a, b = a + da, b + db
        if max(abs(da), abs(db)) < 1e-10:
            break
    return float(a), float(b)


def citl_w(y, p, w=None, iters: int = 30) -> float:
    """Calibration-in-the-large: intercept a with slope fixed at 1 (weighted Newton on the offset model)."""
    z, y = logit(clip(p)), np.asarray(y, float)
    w = np.ones(len(y)) if w is None else np.asarray(w, float)
    a = 0.0
    for _ in range(iters):
        pr = expit(a + z)
        g, h = (w * (y - pr)).sum(), (w * pr * (1 - pr)).sum() + 1e-12
        a += g / h
        if abs(g / h) < 1e-12:
            break
    return float(a)


def ece_w(y, p, w=None, bins: int = 10, edges: Optional[np.ndarray] = None) -> float:
    """Expected calibration error with equal-frequency bins (edges from the unweighted predictions)."""
    p, y = np.asarray(p, float), np.asarray(y, float)
    w = np.ones(len(y)) if w is None else np.asarray(w, float)
    if edges is None:
        edges = np.unique(np.quantile(p, np.linspace(0, 1, bins + 1)))
    b = np.clip(np.searchsorted(edges, p, side="right") - 1, 0, len(edges) - 2)
    W = np.bincount(b, w, minlength=len(edges) - 1)
    ok = W > 0
    oy = np.bincount(b, w * y, minlength=len(edges) - 1)[ok] / W[ok]
    op = np.bincount(b, w * p, minlength=len(edges) - 1)[ok] / W[ok]
    return float((W[ok] / W.sum() * np.abs(oy - op)).sum())


def threshold_metrics(y, p, thr, w=None) -> Dict[str, float]:
    y, p = np.asarray(y, float), np.asarray(p, float)
    w = np.ones(len(y)) if w is None else np.asarray(w, float)
    hit = p >= thr
    tp, fp = (w * hit * y).sum(), (w * hit * (1 - y)).sum()
    fn, tn = (w * ~hit * y).sum(), (w * ~hit * (1 - y)).sum()
    return {"sens": tp / max(tp + fn, 1e-12), "spec": tn / max(tn + fp, 1e-12), "ppv": tp / max(tp + fp, 1e-12),
            "npv": tn / max(tn + fn, 1e-12), "f1": 2 * tp / max(2 * tp + fp + fn, 1e-12), "fpr": fp / max(fp + tn, 1e-12)}


def net_benefit(y, p, t: float, w=None) -> float:
    y, p = np.asarray(y, float), np.asarray(p, float)
    w = np.ones(len(y)) if w is None else np.asarray(w, float)
    hit = p >= t
    return float(((w * hit * y).sum() - (w * hit * (1 - y)).sum() * t / (1 - t)) / w.sum())


def net_benefit_all(y, t: float, w=None) -> float:
    w = np.ones(len(y)) if w is None else np.asarray(w, float)
    prev = (w * y).sum() / w.sum()
    return float(prev - (1 - prev) * t / (1 - t))


# ---------------------------------------------------------------- DeLong (paired, fast midrank version)
def _midrank(x):
    j = np.argsort(x, kind="mergesort")
    z = x[j]
    n = len(x)
    t = np.zeros(n)
    i = 0
    while i < n:
        k = i
        while k < n and z[k] == z[i]:
            k += 1
        t[i:k] = 0.5 * (i + k - 1) + 1
        i = k
    out = np.empty(n)
    out[j] = t
    return out


def delong_paired(y, p1, p2) -> Dict[str, float]:
    """DeLong test for two correlated AUROCs (Sun and Xu fast algorithm). Returns aucs, difference, SE and two-sided p."""
    y = np.asarray(y).astype(int)
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    m, n = len(pos), len(neg)
    aucs, v10, v01 = [], [], []
    for p in (np.asarray(p1, float), np.asarray(p2, float)):
        tx, ty, tz = _midrank(p[pos]), _midrank(p[neg]), _midrank(np.r_[p[pos], p[neg]])
        aucs.append((tz[:m].sum() - m * (m + 1) / 2) / (m * n))
        v10.append((tz[:m] - tx) / n)
        v01.append(1 - (tz[m:] - ty) / m)
    s10, s01 = np.cov(np.vstack(v10)), np.cov(np.vstack(v01))
    S = s10 / m + s01 / n
    d = aucs[0] - aucs[1]
    se = float(np.sqrt(max(S[0, 0] + S[1, 1] - 2 * S[0, 1], 1e-300)))
    return {"auc1": float(aucs[0]), "auc2": float(aucs[1]), "diff": float(d), "se": se, "p_two_sided": float(2 * norm.sf(abs(d) / se)),
            "se1": float(np.sqrt(S[0, 0])), "se2": float(np.sqrt(S[1, 1]))}


# ---------------------------------------------------------------- censoring-aware metrics
def km_censoring(time, event, w=None):
    """Kaplan-Meier estimate of the censoring survival G(t) (censoring = event 0). Returns (unique times, G right after each time)."""
    time, event = np.asarray(time, float), np.asarray(event, int)
    w = np.ones(len(time)) if w is None else np.asarray(w, float)
    ut = np.unique(time)
    idx = np.searchsorted(ut, time)
    d_cens = np.bincount(idx, w * (event == 0), minlength=len(ut))
    n_at = np.cumsum(np.bincount(idx, w, minlength=len(ut))[::-1])[::-1]
    G = np.cumprod(1 - np.where(n_at > 0, d_cens / np.maximum(n_at, 1e-300), 0))
    return ut, G


def G_at(ut, G, t, left: bool = False):
    """G(t) (right-continuous) or G(t-) when left=True."""
    t = np.atleast_1d(np.asarray(t, float))
    i = np.searchsorted(ut, t, side="left" if left else "right") - 1
    return np.where(i >= 0, G[np.clip(i, 0, len(G) - 1)], 1.0)


def km_risk(time, event, tau: float, w=None) -> float:
    """Kaplan-Meier cumulative incidence 1 - S(tau) of the event."""
    time, event = np.asarray(time, float), np.asarray(event, int)
    w = np.ones(len(time)) if w is None else np.asarray(w, float)
    ut = np.unique(time[time <= tau])
    idx = np.searchsorted(np.unique(time), time)
    allt = np.unique(time)
    d = np.bincount(idx, w * (event == 1), minlength=len(allt))
    n_at = np.cumsum(np.bincount(idx, w, minlength=len(allt))[::-1])[::-1]
    S = np.cumprod(1 - np.where(n_at > 0, d / np.maximum(n_at, 1e-300), 0))
    k = np.searchsorted(allt, tau, side="right") - 1
    return float(1 - S[k]) if k >= 0 else 0.0


def td_auc_weights(time, event, tau: float, ut, G):
    """Cases (event by tau) weighted 1/G(T-), controls (time > tau) weighted 1/G(tau); others weight 0 (cumulative/dynamic AUROC)."""
    time, event = np.asarray(time, float), np.asarray(event, int)
    case = (event == 1) & (time <= tau)
    ctrl = time > tau
    wc = np.where(case, 1.0 / np.maximum(G_at(ut, G, time, left=True), 1e-12), 0.0)
    wk = np.where(ctrl, 1.0 / max(float(G_at(ut, G, tau)[0]), 1e-12), 0.0)
    return case.astype(float), wc + wk


def td_auc(p, time, event, tau: float, ut, G, w=None) -> float:
    lab, iw = td_auc_weights(time, event, tau, ut, G)
    keep = iw > 0
    ww = iw[keep] * (1.0 if w is None else np.asarray(w, float)[keep])
    return auroc_w(Ranked(np.asarray(p)[keep], lab[keep]), ww)


def ipcw_brier(p, time, event, tau: float, ut, G, w=None) -> float:
    """IPCW Brier score at tau for predicted risks p (Graf et al. type estimator)."""
    p, time, event = np.asarray(p, float), np.asarray(time, float), np.asarray(event, int)
    w = np.ones(len(p)) if w is None else np.asarray(w, float)
    case = (event == 1) & (time <= tau)
    ctrl = time > tau
    gc = G_at(ut, G, time, left=True)
    gt = float(G_at(ut, G, tau)[0])
    s = np.where(case, (1 - p) ** 2 / np.maximum(gc, 1e-12), 0.0) + np.where(ctrl, p ** 2 / max(gt, 1e-12), 0.0)
    return float((w * s).sum() / w.sum())


class _BIT:
    def __init__(self, n):
        self.n, self.t = n, [0.0] * (n + 1)

    def add(self, i, v):
        i += 1
        while i <= self.n:
            self.t[i] += v
            i += i & (-i)

    def prefix(self, i):           # sum over ranks < i
        s, j = 0.0, i
        while j > 0:
            s += self.t[j]
            j -= j & (-j)
        return s


def concordance(risk, time, event, event_weight=None, tau: Optional[float] = None) -> float:
    """Harrell's C (event_weight None) or Uno's C (event_weight = 1/G(T)^2, truncated at tau).
    Comparable pair: i has an event and T_j > T_i (Uno: also T_i < tau). Risk ties count one half. O(n log n)."""
    risk, time, event = np.asarray(risk, float), np.asarray(time, float), np.asarray(event, int)
    n = len(risk)
    ranks = np.searchsorted(np.unique(risk), risk)
    nr = int(ranks.max()) + 1
    ew = np.ones(n) if event_weight is None else np.asarray(event_weight, float)
    order = np.argsort(-time, kind="mergesort")            # descending time
    bit_cnt = _BIT(nr)
    num = den = 0.0
    i = 0
    total = 0
    while i < n:
        k = i
        t0 = time[order[i]]
        while k < n and time[order[k]] == t0:
            k += 1
        grp = order[i:k]
        cens = [a for a in grp if event[a] == 0]
        for a in cens:                                       # censored at the same time count as outliving the event (Harrell)
            bit_cnt.add(int(ranks[a]), 1.0)
        total += len(cens)
        for a in grp:                                        # events in this time group vs later times and tied censored
            if event[a] == 1 and (tau is None or time[a] < tau) and total > 0:
                lower = bit_cnt.prefix(int(ranks[a]))
                equal = bit_cnt.prefix(int(ranks[a]) + 1) - lower
                num += ew[a] * (lower + 0.5 * equal)
                den += ew[a] * total
        for a in grp:
            if event[a] == 1:
                bit_cnt.add(int(ranks[a]), 1.0)
                total += 1
        i = k
    return float(num / den) if den > 0 else float("nan")


# ---------------------------------------------------------------- resampling and tests
def boot_weight_iter(n: int, B: int, seed: int) -> Iterator[np.ndarray]:
    """Participant bootstrap weights (counts of each row in a resample of size n)."""
    rng = np.random.default_rng(seed)
    for _ in range(B):
        yield np.bincount(rng.integers(0, n, n), minlength=n).astype(np.float64)


def cluster_weight_iter(cluster: np.ndarray, B: int, seed: int) -> Iterator[np.ndarray]:
    """Cluster bootstrap: resample clusters with replacement; each row gets its cluster's count."""
    rng = np.random.default_rng(seed)
    u, inv = np.unique(cluster, return_inverse=True)
    for _ in range(B):
        cnt = np.bincount(rng.integers(0, len(u), len(u)), minlength=len(u)).astype(np.float64)
        yield cnt[inv]


def strata_psu_weight_iter(strata: np.ndarray, psu: np.ndarray, base_w: np.ndarray, B: int, seed: int) -> Iterator[np.ndarray]:
    """Survey bootstrap: within each stratum resample PSUs (n_h - 1 of n_h, Rao-Wu rescaling); returns replicate weights."""
    rng = np.random.default_rng(seed)
    key = np.char.add(np.asarray(strata).astype(str), "|" + np.asarray(psu).astype(str))
    ukey, inv = np.unique(key, return_inverse=True)
    st_of = np.array([k.split("|")[0] for k in ukey])
    ust = np.unique(st_of)
    members = {s: np.flatnonzero(st_of == s) for s in ust}
    for _ in range(B):
        mult = np.zeros(len(ukey))
        for s, ix in members.items():
            nh = len(ix)
            if nh < 2:
                mult[ix] = 1.0
                continue
            pick = rng.integers(0, nh, nh - 1)
            c = np.bincount(pick, minlength=nh)
            mult[ix] = c * nh / (nh - 1)
        yield base_w * mult[inv]


def p_one_sided(draws: np.ndarray, null: float, direction: int) -> float:
    """Percentile-bootstrap one-sided p for the alternative 'difference > null' (direction +1) or '< null' (-1):
    (1 + number of draws on the null side) / (B + 1)."""
    d = np.asarray(draws, float)
    d = d[np.isfinite(d)]
    k = (d <= null).sum() if direction > 0 else (d >= null).sum()
    return float((1 + k) / (len(d) + 1))


def holm(p: Sequence[float]) -> np.ndarray:
    p = np.asarray(p, float)
    out, ok = np.full(len(p), np.nan), np.flatnonzero(np.isfinite(p))
    order = ok[np.argsort(p[ok], kind="mergesort")]
    adj = np.minimum(1.0, (len(order) - np.arange(len(order))) * p[order])
    out[order] = np.maximum.accumulate(adj)
    return out


def corrected_t(diffs: np.ndarray, n_test: float, n_train: float) -> Dict[str, float]:
    """Nadeau-Bengio corrected resampled t-test over the R x K fold differences."""
    d = np.asarray(diffs, float)
    J = len(d)
    var = d.var(ddof=1) * (1.0 / J + n_test / n_train)
    se = math.sqrt(max(var, 1e-300))
    tt = d.mean() / se
    q = t_dist.ppf(0.975, J - 1)
    return {"mean": float(d.mean()), "se": se, "lo": float(d.mean() - q * se), "hi": float(d.mean() + q * se),
            "p_two_sided": float(2 * t_dist.sf(abs(tt), J - 1)), "df": J - 1}


def spearman_rows(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Row-wise Spearman correlation of two attribution matrices (average ranks for ties)."""
    from scipy.stats import rankdata
    ra, rb = rankdata(np.asarray(A, float), axis=1), rankdata(np.asarray(B, float), axis=1)
    ra -= ra.mean(1, keepdims=True)
    rb -= rb.mean(1, keepdims=True)
    den = np.sqrt((ra ** 2).sum(1) * (rb ** 2).sum(1))
    return np.where(den > 0, (ra * rb).sum(1) / np.where(den > 0, den, 1), np.nan)


In [ ]:
%%writefile ckv2_lib/ckv2/engine.py
"""Task engine: every model fit is one task with a deterministic id, run in a pool of single-threaded worker processes and
checkpointed to disk, so an interrupted run resumes where it stopped (protocol_v2.md, section 6)."""
import os
import pickle
import time
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import get_context
from pathlib import Path
from typing import Dict, List, Optional, Sequence

import numpy as np
import pandas as pd

_FRAMES: Dict[str, pd.DataFrame] = {}
_CACHE_DIR: Optional[Path] = None


def _init_worker(cache_dir: str) -> None:
    global _CACHE_DIR
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    import torch
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True, warn_only=True)
    _CACHE_DIR = Path(cache_dir)


def save_frames(cache_dir: Path, frames: Dict[str, pd.DataFrame]) -> None:
    (cache_dir / "frames").mkdir(parents=True, exist_ok=True)
    for k, f in frames.items():
        p = cache_dir / "frames" / f"{k}.pkl"
        tmp = p.with_suffix(".tmp")
        f.to_pickle(tmp)
        os.replace(tmp, p)
        _FRAMES[k] = f


def frame(key: str) -> pd.DataFrame:
    if key not in _FRAMES:
        _FRAMES[key] = pd.read_pickle(_CACHE_DIR / "frames" / f"{key}.pkl")
    return _FRAMES[key]


def task_path(cache_dir: Path, tid: str) -> Path:
    return cache_dir / "tasks" / f"{tid}.pkl"


def _mahalanobis(X: np.ndarray, mu: np.ndarray, icov: np.ndarray) -> np.ndarray:
    D = X - mu
    return np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", D, icov, D), 0))


def run_task(spec: dict, cfg) -> dict:
    """Fit one model and return predictions and requested extras. Pure function of spec and cfg."""
    from .config import BINARY
    from .missing import corrupt
    from .models import Fitted
    from .nets import integrated_gradients, violation_rows
    t0 = time.perf_counter()
    tr = frame(spec["train_frame"]).iloc[spec["train_idx"]].reset_index(drop=True)
    ytr = np.asarray(spec["train_y"]) if spec.get("train_y") is not None else tr.y.to_numpy()
    sw = spec.get("train_w")
    if sw is not None:                                     # IPCW models: rows with weight 0 do not enter the fit
        keep = np.asarray(sw) > 0
        tr, ytr, sw = tr[keep].reset_index(drop=True), ytr[keep], np.asarray(sw, float)[keep]
    fm = Fitted(spec["model"], spec["feats"], spec["hp"], spec["seed"], cfg, signs=spec.get("signs"), lam=spec.get("lam", 0.0))
    fm.fit(tr, ytr, sw)
    out = {"id": spec["id"], "info": fm.info(), "preds": {}, "fit_seconds": time.perf_counter() - t0}
    for t in spec.get("tests", []):
        df = frame(t["frame"]).iloc[t["idx"]].reset_index(drop=True)
        if t.get("corrupt"):
            mech, rate, cseed = t["corrupt"]
            df = corrupt(df, spec["feats"], mech, rate, cseed)
        for f, v in (t.get("set") or {}).items():                      # post hoc: set a predictor to a fixed value (BMI 24 / 35)
            df[f] = float(v)
        out["preds"][t["tag"]] = fm.predict(df).astype(np.float32)
    nn = spec["model"] in ("M1", "M2", "CKINN")
    if nn and spec.get("explain"):
        e = spec["explain"]
        df = frame(e["frame"]).iloc[e["idx"]].reset_index(drop=True)
        X, M = fm.prep.transform(df)
        A = integrated_gradients(fm.net, X, M, cfg.ig_steps)
        out["attr"] = A.astype(np.float32)
        out["viol"] = violation_rows(fm.net, X, M, e["signs"], spec["feats"], cfg.viol_tol, attributions=A)
        if e.get("sparse"):
            Xtr, _ = fm.prep.transform(tr)
            mu, cov = Xtr.mean(0), np.cov(Xtr, rowvar=False) + 1e-6 * np.eye(Xtr.shape[1])
            icov = np.linalg.pinv(cov)
            thr = float(np.quantile(_mahalanobis(Xtr, mu, icov), cfg.sparse_quantile))
            dte = frame(e["frame"]).iloc[e["sparse_pool"]].reset_index(drop=True)
            Xs, Ms = fm.prep.transform(dte)
            dist = _mahalanobis(Xs, mu, icov)
            sel = np.flatnonzero(dist > thr)[: cfg.n_explain]
            out["sparse"] = {"n_pool": int(len(dte)), "n_sparse": int((dist > thr).sum()), "thr": thr,
                             **violation_rows(fm.net, Xs[sel], Ms[sel], e["signs"], spec["feats"], cfg.viol_tol)}
    if nn and spec.get("extra_viol"):
        out["extra_viol"] = {}
        for tag, fkey, idx, signs in spec["extra_viol"]:
            df = frame(fkey).iloc[idx].reset_index(drop=True)
            X, M = fm.prep.transform(df)
            out["extra_viol"][tag] = violation_rows(fm.net, X, M, signs, spec["feats"], cfg.viol_tol)
    if spec.get("pd"):
        p = spec["pd"]
        df = frame(p["frame"]).iloc[p["idx"]].reset_index(drop=True)
        res = {}
        for f in p["features"]:
            if f not in spec["feats"] or f in BINARY:
                continue
            grid = np.nanpercentile(tr[f].to_numpy(float), np.arange(10, 91, 10))
            curve = []
            for v in grid:
                d2 = df.copy()
                d2[f] = v
                curve.append(float(fm.predict(d2).mean()))
            res[f] = {"grid": grid, "pd": np.array(curve)}
        out["pd"] = res
    if spec.get("perturb"):
        from .posthoc import perturbation_violations
        pz = spec["perturb"]
        df = frame(pz["frame"]).iloc[pz["idx"]].reset_index(drop=True)
        out["perturb"] = perturbation_violations(fm.predict, df, tr, pz["signs"], spec["feats"], pz["delta_sd"], cfg.viol_tol * pz["delta_sd"])
    if spec.get("shap"):
        from .posthoc import permutation_shap
        sh = spec["shap"]
        df = frame(sh["frame"]).iloc[sh["idx"]].reset_index(drop=True)
        rng = np.random.default_rng(sh["seed"])
        bg = tr.iloc[rng.choice(len(tr), min(sh["n_bg"], len(tr)), replace=False)].reset_index(drop=True)
        phi, base, fx = permutation_shap(fm.predict, df, bg, spec["feats"], sh["n_perm"], rng)
        out["shap"] = {"phi": phi.astype(np.float32), "base": base, "fx": fx.astype(np.float32),
                       "x": df[spec["feats"]].to_numpy(float).astype(np.float32), "feats": list(spec["feats"])}
        if spec.get("reason"):
            from .posthoc import reasoning_inputs
            rz = spec["reason"]
            k = min(rz["n"], len(df))
            out["reason"] = reasoning_inputs(fm.predict, df.iloc[:k].reset_index(drop=True), tr, rz["signs"], spec["feats"],
                                             rz["delta_sd"], cfg.viol_tol * rz["delta_sd"], rz["panel"])
            out["reason"]["phi"] = phi[:k].astype(np.float32)
    out["seconds"] = time.perf_counter() - t0
    return out


def _worker(spec: dict, cfg, cache_dir: str) -> str:
    path = task_path(Path(cache_dir), spec["id"])
    if path.exists():
        return spec["id"]
    try:
        res = run_task(spec, cfg)
    except Exception:
        raise RuntimeError(f"task {spec['id']} failed:\n{traceback.format_exc()}")
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as fh:
        pickle.dump(res, fh, protocol=4)
    os.replace(tmp, path)
    return spec["id"]


def run_tasks(specs: Sequence[dict], cfg, cache_dir: Path, n_workers: int, label: str = "") -> None:
    """Run every spec whose checkpoint is missing. Raises if any task fails (after the others finish)."""
    (cache_dir / "tasks").mkdir(parents=True, exist_ok=True)
    ids = [s["id"] for s in specs]
    assert len(ids) == len(set(ids)), "duplicate task ids"
    todo = [s for s in specs if not task_path(cache_dir, s["id"]).exists()]
    t0 = time.perf_counter()
    print(f"[{label}] {len(specs)} tasks, {len(specs) - len(todo)} already done, {len(todo)} to run on {n_workers} workers", flush=True)
    if not todo:
        return
    errors = []
    if n_workers <= 1:
        _init_worker(str(cache_dir))
        for i, s in enumerate(todo):
            try:
                _worker(s, cfg, str(cache_dir))
            except Exception as e:  # noqa: BLE001  (collected and raised below)
                errors.append(str(e))
    else:
        ctx = get_context("spawn")
        with ProcessPoolExecutor(n_workers, mp_context=ctx, initializer=_init_worker, initargs=(str(cache_dir),)) as ex:
            futs = {ex.submit(_worker, s, cfg, str(cache_dir)): s["id"] for s in todo}
            done = 0
            step = max(1, len(todo) // 10)
            for f in as_completed(futs):
                done += 1
                try:
                    f.result()
                except Exception as e:  # noqa: BLE001
                    errors.append(str(e))
                if done % step == 0 or done == len(todo):
                    print(f"[{label}] {done}/{len(todo)} done, {time.perf_counter() - t0:.0f}s", flush=True)
    if errors:
        raise RuntimeError(f"{len(errors)} task(s) failed; first:\n{errors[0]}")


def load(cache_dir: Path, tid: str) -> dict:
    with open(task_path(cache_dir, tid), "rb") as fh:
        return pickle.load(fh)


In [ ]:
%%writefile ckv2_lib/ckv2/plan.py
"""Builds the task specifications of every stage (protocol_v2.md, sections 5, 6 and 8) and assembles out-of-fold predictions."""
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split

from . import metrics as mt
from .config import (C_GRID, MAIN_MODELS, MLP_DEFAULT, MLP_SPACE, NN_MODELS, RF_GRID, RUN0_EXTRA, SIGNS_CORE, SIGNS_EXT, SIGNS_T2,
                     SIGNS_T3, TREE_DEFAULT, TREE_SPACE, seed_for)
from .engine import load
from .missing import scen_tag, scenarios
from .nets import weighted_val_score
from .prep import Prep


def tid(*parts) -> str:
    return "__".join(str(p).replace("/", "-").replace(" ", "") for p in parts)


def signs_for(tier: str, feats: Sequence[str], table: Optional[Dict[str, int]] = None) -> Dict[str, int]:
    t = table if table is not None else (SIGNS_T3 if tier == "T3" else SIGNS_T2)
    return {f: s for f, s in t.items() if f in feats}


# ------------------------------------------------------------------ tuning
def _grid(space: dict) -> List[dict]:
    keys = sorted(space)
    out = [{}]
    for k in keys:
        out = [dict(o, **{k: v}) for o in out for v in space[k]]
    return out


def sample_configs(space: dict, default: dict, n: int, seed: int) -> List[dict]:
    """The default first, then n-1 distinct configurations drawn without replacement from the full grid."""
    grid = [g for g in _grid(space) if g != default]
    rng = np.random.default_rng(seed)
    pick = rng.choice(len(grid), size=min(n - 1, len(grid)), replace=False)
    return [dict(default)] + [grid[i] for i in pick]


def config_lists(cohort: str, cfg) -> Dict[str, List[dict]]:
    mlp = sample_configs(MLP_SPACE, MLP_DEFAULT, cfg.tune_mlp_configs, seed_for("hp", cohort, "MLP"))
    xgb = sample_configs(TREE_SPACE["XGB"], TREE_DEFAULT["XGB"], cfg.tune_tree_configs, seed_for("hp", cohort, "XGB"))
    lgb = sample_configs(TREE_SPACE["LGBM"], TREE_DEFAULT["LGBM"], cfg.tune_tree_configs, seed_for("hp", cohort, "LGBM"))
    cg = [{"C": c} for c in C_GRID]
    rf = RF_GRID if cfg.mode == "full" else RF_GRID[:2]
    if cfg.mode != "full":
        cg = cg[1:3]
    return {"M1": mlp, "M2": mlp, "XGB": xgb, "XGB_mono": xgb, "LGBM": lgb, "LGBM_mono": lgb, "LR": cg, "cLR": cg, "GAM": cg, "RF": rf}


def tune_specs(cohort: str, dev: pd.DataFrame, feats: Sequence[str], cfg) -> List[dict]:
    folds = list(StratifiedKFold(cfg.tune_folds, shuffle=True, random_state=seed_for("devcv", cohort)).split(dev, dev.y))
    sg = signs_for("T2", feats)
    specs = []
    for model, lst in config_lists(cohort, cfg).items():
        for ci, hp in enumerate(lst):
            for k, (i, j) in enumerate(folds):
                specs.append({"id": tid("tune", cohort, model, f"c{ci}", f"k{k}"), "model": model, "feats": list(feats), "hp": hp,
                              "seed": seed_for("devfit", cohort, k), "signs": sg, "train_frame": f"{cohort}_dev", "train_idx": i,
                              "tests": [{"tag": "val", "frame": f"{cohort}_dev", "idx": j}], "_val_idx": j, "_model": model, "_ci": ci})
    return specs


def lam_folds(cohort: str, dev: pd.DataFrame, cfg):
    out = []
    for rep in range(cfg.lam_reps):
        rs = seed_for("devcv", cohort) if rep == 0 else seed_for("devcv", cohort, rep)
        for k, (i, j) in enumerate(StratifiedKFold(cfg.tune_folds, shuffle=True, random_state=rs).split(dev, dev.y)):
            out.append((rep, k, i, j))
    return out


def lam_specs(cohort: str, tier: str, dev: pd.DataFrame, feats: Sequence[str], base_hp: dict, cfg) -> List[dict]:
    sg = signs_for(tier, feats)
    specs = []
    for lam in (0.0,) + tuple(cfg.lam_grid):
        for rep, k, i, j in lam_folds(cohort, dev, cfg):
            specs.append({"id": tid("lam", cohort, tier, f"l{lam:g}", f"r{rep}", f"k{k}"), "model": "CKINN", "feats": list(feats),
                          "hp": base_hp, "seed": seed_for("devlam", cohort, rep, k), "signs": sg, "lam": lam,
                          "train_frame": f"{cohort}_dev", "train_idx": i, "tests": [{"tag": "val", "frame": f"{cohort}_dev", "idx": j}],
                          "_val_idx": j, "_lam": lam})
    return specs


def score_specs(specs: Sequence[dict], dev: pd.DataFrame, cache_dir) -> pd.DataFrame:
    rows = []
    y = dev.y.to_numpy()
    for s in specs:
        r = load(cache_dir, s["id"])
        rows.append({"id": s["id"], "model": s.get("_model", "CKINN"), "ci": s.get("_ci"), "lam": s.get("_lam"),
                     "score": weighted_val_score(y[s["_val_idx"]], r["preds"]["val"].astype(float)), "epochs": r["info"]["epochs"]})
    return pd.DataFrame(rows)


def select_lambda(scores: pd.DataFrame, grid: Sequence[float]) -> Tuple[float, pd.DataFrame]:
    """Protocol 5: largest positive lambda whose mean score is within one SE (over the fits) of the minimum over {0} and the grid."""
    g = scores.groupby("lam").score.agg(["mean", "std", "count"]).reset_index()
    g["se"] = g["std"] / np.sqrt(g["count"])
    best = g.loc[g["mean"].idxmin()]
    ok = g[(g["lam"] > 0) & (g["mean"] <= best["mean"] + best["se"])]
    lam = float(ok["lam"].max()) if len(ok) else float(min(grid))
    g["chosen"] = g["lam"] == lam
    g["within_1se"] = g["mean"] <= best["mean"] + best["se"]
    return lam, g


# ------------------------------------------------------------------ evaluation stages
def explain_rows(cohort: str, test_idx: np.ndarray, k: int, n: int) -> np.ndarray:
    rng = np.random.default_rng(seed_for("expl", cohort, k))
    return np.sort(rng.choice(test_idx, min(n, len(test_idx)), replace=False))


def pd_rows(cohort: str, test_idx: np.ndarray, k: int, n: int) -> np.ndarray:
    rng = np.random.default_rng(seed_for("pd", cohort, k))
    return np.sort(rng.choice(test_idx, min(n, len(test_idx)), replace=False))


def fit_seed(cohort: str, tier: str, r: int, k: int, s: int) -> int:
    """The same seed for every model of a fold and seed index (pairing, A1)."""
    return seed_for("fit", cohort, tier, r, k, s)


def cv_specs(cohort: str, tier: str, folds, feats, HP, LAM, cfg, frame_key: Optional[str] = None, models=None,
             repeats=None, variant: str = "", signs: Optional[Dict[str, int]] = None, scen: bool = True, extras: bool = True) -> List[dict]:
    fk = frame_key or cohort
    sg = signs if signs is not None else signs_for(tier, feats)
    models = models or MAIN_MODELS
    specs = []
    for r in (range(cfg.n_repeats) if repeats is None else repeats):
        for k, (i, j) in enumerate(folds[r]):
            tests = [{"tag": "none", "frame": fk, "idx": j}]
            if scen and tier == "T2":
                tests += [{"tag": scen_tag(m, q), "frame": fk, "idx": j, "corrupt": (m, q, seed_for("corrupt", cohort, r, k, m, q))}
                          for m, q in scenarios(cfg)]
            mlist = list(models) + (list(RUN0_EXTRA) if (r == 0 and extras and models is MAIN_MODELS) else [])
            for m in mlist:
                hp_key = {"MI_LR": "LR", "LR_aug": "LR", "XGB_aug": "XGB"}.get(m, m)
                spec = {"id": tid("cv", cohort, tier, variant or "main", f"r{r}", f"k{k}", m, f"s{r}"), "model": m, "feats": list(feats),
                        "hp": HP[(cohort, hp_key)], "seed": fit_seed(cohort, tier, r, k, r), "signs": sg,
                        "lam": LAM[(cohort, tier)] if m == "CKINN" else 0.0, "train_frame": fk, "train_idx": i, "tests": tests,
                        "_r": r, "_k": k, "_s": r, "_model": m}
                if r == 0 and extras:
                    spec["pd"] = {"frame": fk, "idx": pd_rows(cohort, j, k, cfg.n_pd_rows), "features": list(sg)}
                    if m in NN_MODELS:
                        spec["explain"] = {"frame": fk, "idx": explain_rows(cohort, j, k, cfg.n_explain), "signs": signs_for(tier, feats),
                                           "sparse": True, "sparse_pool": j}
                specs.append(spec)
    return specs


def seed_specs(cohort: str, tier: str, folds, feats, HP, LAM, cfg) -> List[dict]:
    sg = signs_for(tier, feats)
    specs = []
    for s in range(1, cfg.n_seeds):
        for k, (i, j) in enumerate(folds[0]):
            for m in NN_MODELS:
                specs.append({"id": tid("cv", cohort, tier, "seed", "r0", f"k{k}", m, f"s{s}"), "model": m, "feats": list(feats),
                              "hp": HP[(cohort, m)], "seed": fit_seed(cohort, tier, 0, k, s), "signs": sg,
                              "lam": LAM[(cohort, tier)] if m == "CKINN" else 0.0, "train_frame": cohort, "train_idx": i,
                              "tests": [{"tag": "none", "frame": cohort, "idx": j}],
                              "explain": {"frame": cohort, "idx": explain_rows(cohort, j, k, cfg.n_explain), "signs": sg},
                              "_r": 0, "_k": k, "_s": s, "_model": m})
    return specs


def variant_tables(feats: Sequence[str]) -> Dict[str, Dict[str, int]]:
    base = signs_for("T2", feats)
    out = {f"loco_{f}": {g: s for g, s in base.items() if g != f} for f in base}
    flip = lambda keys: {g: (-s if g in keys else s) for g, s in base.items()}
    out["wrong_age"], out["wrong_bmi"], out["wrong_all"] = flip({"age_years"}), flip({"bmi_kgm2"}), flip(set(base))
    out["sign_core"] = signs_for("T2", feats, SIGNS_CORE)
    out["sign_ext"] = signs_for("T2", feats, SIGNS_EXT)
    return out


def variant_specs(cohort: str, folds, feats, HP, LAM, cfg) -> List[dict]:
    """Protocol 8: non-circular constraint analyses (repeat 0, seed index 0, same seed as the main CKINN of repeat 0)."""
    base = signs_for("T2", feats)
    specs = []
    tables = variant_tables(feats)
    lam_path = {f"lam_{l:g}": l for l in cfg.lam_grid}
    for k, (i, j) in enumerate(folds[0]):
        ex = explain_rows(cohort, j, k, cfg.n_explain)
        for v, table in list(tables.items()) + [(n, base) for n in lam_path]:
            lam = lam_path.get(v, LAM[(cohort, "T2")])
            specs.append({"id": tid("var", cohort, "T2", v, "r0", f"k{k}", "CKINN", "s0"), "model": "CKINN", "feats": list(feats),
                          "hp": HP[(cohort, "CKINN")], "seed": fit_seed(cohort, "T2", 0, k, 0), "signs": table, "lam": lam,
                          "train_frame": cohort, "train_idx": i, "tests": [{"tag": "none", "frame": cohort, "idx": j}],
                          "explain": {"frame": cohort, "idx": ex, "signs": base},
                          "extra_viol": [("own_table", cohort, ex, table)] + ([("dropped", cohort, ex, {v[5:]: base[v[5:]]})] if v.startswith("loco_") else []),
                          "_variant": v, "_k": k})
    return specs


def ipcw_weights(time: np.ndarray, event: np.ndarray, tau: float):
    """3-year endpoint and inverse-probability-of-censoring weights from the training rows (protocol 8)."""
    ut, G = mt.km_censoring(time, event)
    y3 = ((event == 1) & (time <= tau)).astype(int)
    w = np.where(y3 == 1, 1 / np.maximum(mt.G_at(ut, G, time, left=True), 1e-6), np.where(time > tau, 1 / max(float(mt.G_at(ut, G, tau)[0]), 1e-6), 0.0))
    return y3, w


def ipcw_specs(cohort: str, folds, feats, HP, LAM, cfg, fr: pd.DataFrame) -> List[dict]:
    specs = []
    sg = signs_for("T2", feats)
    for k, (i, j) in enumerate(folds[0]):
        y3, w = ipcw_weights(fr.follow_years.to_numpy()[i], fr.y.to_numpy()[i], cfg.horizon)
        for m in ("LR", "XGB_mono", "M2", "CKINN"):
            specs.append({"id": tid("ipcw", cohort, "T2", "r0", f"k{k}", m, "s0"), "model": m, "feats": list(feats), "hp": HP[(cohort, m)],
                          "seed": fit_seed(cohort, "T2", 0, k, 0), "signs": sg, "lam": LAM[(cohort, "T2")] if m == "CKINN" else 0.0,
                          "train_frame": cohort, "train_idx": i, "train_y": y3, "train_w": w,
                          "tests": [{"tag": "none", "frame": cohort, "idx": j}], "_k": k, "_model": m})
    return specs


def shared_features(fr_a: pd.DataFrame, fr_b: pd.DataFrame, candidates: Sequence[str]) -> List[str]:
    """v1 rule: observed in more than 90% of NAGALA and more than 30% of RICH."""
    return [c for c in candidates if fr_a[c].notna().mean() > 0.9 and fr_b[c].notna().mean() > 0.3]


def domain_weights(src: pd.DataFrame, tgt: pd.DataFrame, feats: Sequence[str], clip: Tuple[float, float]) -> np.ndarray:
    """Label-free importance weights for source rows: odds of target membership from an L2 LR on the shared predictors."""
    both = pd.concat([src[feats], tgt[feats]], ignore_index=True)
    z = np.r_[np.zeros(len(src)), np.ones(len(tgt))]
    pp = Prep(feats).fit(both)
    X = pp.transform(both)[0]
    lr = LogisticRegression(C=1.0, max_iter=2000).fit(X, z)
    p = lr.predict_proba(X[: len(src)])[:, 1]
    w = p / (1 - p) * (len(src) / len(tgt))
    w = np.clip(w, *clip)
    return w / w.mean()


def transport_specs(FR, FOLDS, shared, HP, LAM, cfg) -> List[dict]:
    specs = []
    for src, tgt in (("NAGALA", "RICH"), ("RICH", "NAGALA")):
        sg = signs_for("T2", shared)
        n_src, n_tgt = len(FR[src]), len(FR[tgt])
        ex_t = np.sort(np.random.default_rng(seed_for("trexpl", tgt)).choice(n_tgt, min(cfg.n_explain, n_tgt), replace=False))
        ex_s = np.sort(np.random.default_rng(seed_for("trexpl", src)).choice(n_src, min(cfg.n_explain, n_src), replace=False))
        w_rw = domain_weights(FR[src], FR[tgt], shared, cfg.weight_clip)
        for m in ("LR", "XGB", "XGB_mono") + NN_MODELS:
            seeds = range(cfg.n_seeds) if m in NN_MODELS else [0]
            for s in seeds:
                base = {"model": m, "feats": list(shared), "hp": HP[(src, m)], "signs": sg, "lam": LAM[(src, "T2")] if m == "CKINN" else 0.0}
                specs.append({**base, "id": tid("trext", src, tgt, m, f"s{s}"), "seed": seed_for("tr_ext", src, s), "train_frame": src,
                              "train_idx": np.arange(n_src), "tests": [{"tag": "tgt", "frame": tgt, "idx": np.arange(n_tgt)}],
                              "extra_viol": [("tgt", tgt, ex_t, sg), ("src", src, ex_s, sg)], "_src": src, "_tgt": tgt, "_model": m, "_s": s})
                for k, (i, j) in enumerate(FOLDS[tgt][0]):
                    bt = {"model": m, "feats": list(shared), "hp": HP[(tgt, m)], "signs": sg, "lam": LAM[(tgt, "T2")] if m == "CKINN" else 0.0}
                    specs.append({**bt, "id": tid("trref", tgt, m, f"k{k}", f"s{s}"), "seed": seed_for("tr_ref", tgt, k, s), "train_frame": tgt,
                                  "train_idx": i, "tests": [{"tag": "none", "frame": tgt, "idx": j}], "_tgt": tgt, "_model": m, "_s": s, "_k": k})
                if s == 0 and m in ("LR", "M2", "CKINN"):
                    specs.append({**base, "id": tid("trrw", src, tgt, m, "s0"), "seed": seed_for("tr_ext", src, 0), "train_frame": src,
                                  "train_idx": np.arange(n_src), "train_w": w_rw, "tests": [{"tag": "tgt", "frame": tgt, "idx": np.arange(n_tgt)}],
                                  "_src": src, "_tgt": tgt, "_model": m})
    return specs


def pooled_specs(FR, FOLDS, pooled_feats, HP, LAM, cfg, cap: int = 20000) -> List[dict]:
    """A17: pooled NAGALA + RICH training (each cohort's training rows capped at 20,000 as in v1); training-only standardisation."""
    off = {"NAGALA": 0, "RICH": len(FR["NAGALA"])}
    specs = []
    for k in range(cfg.n_splits):
        tr, tests = [], []
        for c in ("NAGALA", "RICH"):
            i, j = FOLDS[c][0][k]
            if len(i) > cap:
                i = np.sort(train_test_split(i, train_size=cap, stratify=FR[c].y.to_numpy()[i], random_state=seed_for("poolcap", c, k))[0])
            tr.append(i + off[c])
            tests.append({"tag": c, "frame": "POOL", "idx": j + off[c]})
        for m in ("M2", "CKINN"):
            specs.append({"id": tid("pool", m, f"k{k}"), "model": m, "feats": list(pooled_feats), "hp": HP[("RICH", m)],
                          "seed": seed_for("pool", k), "signs": signs_for("T2", pooled_feats), "lam": LAM[("RICH", "T2")] if m == "CKINN" else 0.0,
                          "train_frame": "POOL", "train_idx": np.concatenate(tr), "tests": tests, "_k": k, "_model": m})
    return specs


def site_specs(fr: pd.DataFrame, feats, HP, LAM, cfg) -> Tuple[List[dict], List[str]]:
    ev = fr.groupby("site").y.sum()
    sites = sorted([s for s in ev.index if ev[s] >= cfg.site_min_events], key=lambda s: int(s))
    specs = []
    site = fr.site.to_numpy()
    for s in sites:
        te, tr = np.flatnonzero(site == s), np.flatnonzero(site != s)
        for m in ("LR", "XGB", "XGB_mono") + NN_MODELS:
            specs.append({"id": tid("site", s, m), "model": m, "feats": list(feats), "hp": HP[("RICH", m)], "seed": seed_for("site", s, 0),
                          "signs": signs_for("T2", feats), "lam": LAM[("RICH", "T2")] if m == "CKINN" else 0.0, "train_frame": "RICH",
                          "train_idx": tr, "tests": [{"tag": "none", "frame": "RICH", "idx": te}], "_site": s, "_model": m})
    return specs, sites


# ------------------------------------------------------------------ assembly
def assemble(specs: Sequence[dict], n: int, cache_dir, tag: str = "none", key=lambda s: (s["_model"], s["_r"], s["_s"])) -> Dict[tuple, np.ndarray]:
    """Out-of-fold vectors: for each key, place each fold's predictions at its test rows. Every row must be filled."""
    out: Dict[tuple, np.ndarray] = {}
    for s in specs:
        k = key(s)
        if k not in out:
            out[k] = np.full(n, np.nan, np.float32)
        t = [x for x in s["tests"] if x["tag"] == tag]
        if not t:
            continue
        out[k][t[0]["idx"]] = load(cache_dir, s["id"])["preds"][tag]
    for k, v in out.items():
        assert np.isfinite(v).all(), f"incomplete out-of-fold predictions for {k} ({tag})"
    return out


In [ ]:
%%writefile ckv2_lib/ckv2/analysis.py
"""Analysis of stored predictions (protocol_v2.md, sections 6 to 8). Pure functions of predictions, labels and configuration.

Bootstrap convention: in each draw the same participant weights are applied to every run and every model, the metric is computed
for each run and averaged over runs, and differences are taken between run averages (paired). Draws are split into chunks run in
worker processes; chunk c of a unit uses seed seed_for(<unit>, c), so results do not depend on the number of workers.
"""
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import get_context
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.metrics import average_precision_score

from . import metrics as mt
from .config import seed_for
from .engine import load

N_CHUNKS = 8


# ================================================================== assembly of stored predictions
def collect(specs: Sequence[dict], n: int, cache_dir, key=lambda s: (s["_model"], s["_r"], s["_s"])):
    """P[key][tag] = out-of-fold vector; THR[key] = (row threshold for specificity target, for sensitivity target); FOLD[key] = fold id."""
    P, THR, FOLD = {}, {}, {}
    for s in specs:
        k = key(s)
        r = load(cache_dir, s["id"])
        for t in s["tests"]:
            P.setdefault(k, {}).setdefault(t["tag"], np.full(n, np.nan, np.float32))[t["idx"]] = r["preds"][t["tag"]]
        none = [t for t in s["tests"] if t["tag"] in ("none", "tgt")]
        if none:
            ts, tn = THR.setdefault(k, (np.full(n, np.nan), np.full(n, np.nan)))
            ts[none[0]["idx"]], tn[none[0]["idx"]] = r["info"]["thr_spec"], r["info"]["thr_sens"]
            FOLD.setdefault(k, np.full(n, -1))[none[0]["idx"]] = s.get("_k", 0)
    for k, d in P.items():
        for tag, v in d.items():
            assert np.isfinite(v).all(), f"incomplete predictions {k} {tag}"
    return P, THR, FOLD


# ================================================================== bootstrap engine
def _weights(payload: dict, B: int, seed: int):
    n = len(payload["y"])
    base = payload.get("base_w")
    if payload.get("mode") == "cluster":
        it = mt.cluster_weight_iter(payload["cluster"], B, seed)
    elif payload.get("mode") == "survey":
        it = mt.strata_psu_weight_iter(payload["strata"], payload["psu"], base, B, seed)
        base = None
    else:
        it = mt.boot_weight_iter(n, B, seed)
    for w in it:
        yield w if base is None else w * base


class _Vec:
    """One prediction vector with its precomputed rank structures and point-estimate calibration (warm start)."""

    def __init__(self, y, p, groups, base_w, thr=None):
        self.p = np.asarray(p, float)
        self.hit = None if thr is None else (self.p >= np.asarray(thr, float))
        self.r = mt.Ranked(self.p, y)
        self.a, self.b = mt.calib_w(y, self.p, base_w)
        self.citl = mt.citl_w(y, self.p, base_w)
        self.edges = np.unique(np.quantile(self.p, np.linspace(0, 1, 11)))
        self.g = {gn: [(ix, mt.Ranked(self.p[ix], y[ix])) for ix in idxs] for gn, idxs in groups.items()}


def _stat(name: str, v: _Vec, y, w, payload) -> float:
    if name == "auroc":
        return mt.auroc_w(v.r, w)
    if name == "brier":
        return mt.brier_w(y, v.p, w)
    if name == "slope_err":
        return abs(mt.calib_w(y, v.p, w, iters=4, start=(v.a, v.b))[1] - 1)
    if name == "slope":
        return mt.calib_w(y, v.p, w, iters=4, start=(v.a, v.b))[1]
    if name == "citl":
        return _citl_warm(y, v.p, w, v.citl)
    if name == "citl_abs":
        return abs(_citl_warm(y, v.p, w, v.citl))
    if name == "ece":
        return mt.ece_w(y, v.p, w, edges=v.edges)
    if name in ("tpr_sex_gap", "tpr_age_range"):
        gn = "sex" if name == "tpr_sex_gap" else "age"
        vals = [mt.tpr_at_fpr_w(r, payload["fpr"], None if w is None else w[ix]) for ix, r in v.g[gn]]
        if len(vals) < 2:
            return float("nan")
        return float(abs(vals[0] - vals[1])) if gn == "sex" else float(max(vals) - min(vals))
    if name.startswith("gap_"):
        vals = []
        for ix, r in v.g["fair"]:
            wg = None if w is None else w[ix]
            yg = y[ix]
            if name == "gap_auroc":
                vals.append(mt.auroc_w(r, wg))
            elif name == "gap_tpr":
                vals.append(mt.tpr_at_fpr_w(r, payload["fpr"], wg))
            elif name in ("gap_tprthr", "gap_fprthr"):
                ww = np.ones(len(ix)) if wg is None else wg
                h = v.hit[ix]
                pos = yg == 1 if name == "gap_tprthr" else yg == 0
                vals.append(float((ww * h * pos).sum() / max((ww * pos).sum(), 1e-12)))
            elif name == "gap_citl":
                vals.append(_citl_warm(yg, v.p[ix], wg, 0.0, iters=8))
        vals = [x for x in vals if np.isfinite(x)]
        return float(max(vals) - min(vals)) if len(vals) > 1 else float("nan")
    if name.startswith("nb@"):
        return mt.net_benefit(y, v.p, float(name[3:]), w)
    raise ValueError(name)


def _citl_warm(y, p, w, start, iters=5):
    z = mt.logit(mt.clip(p))
    w = np.ones(len(y)) if w is None else w
    a = start
    for _ in range(iters):
        pr = 1 / (1 + np.exp(-(a + z)))
        a += (w * (y - pr)).sum() / ((w * pr * (1 - pr)).sum() + 1e-12)
    return float(a)


def boot_chunk(payload: dict, B: int, seed: int) -> Dict[Tuple[str, str], np.ndarray]:
    """Run-averaged statistics for B draws (B = 0: the point estimate, as an array of length 1)."""
    y = np.asarray(payload["y"], float)
    base = payload.get("base_w") if payload.get("mode") != "survey" else payload.get("base_w")
    groups = payload.get("groups", {})
    thrs = payload.get("thrs", {})
    vecs = {m: [_Vec(y, p, groups, base, (thrs.get(m) or [None] * len(ps))[i]) for i, p in enumerate(ps)] for m, ps in payload["vecs"].items()}
    stats = payload["stats"]
    if B == 0:
        draws = [base]
    else:
        draws = _weights(payload, B, seed)
    out = {(m, s): [] for m in vecs for s in stats}
    for w in draws:
        for m, vs in vecs.items():
            for s in stats:
                out[(m, s)].append(float(np.mean([_stat(s, v, y, w, payload) for v in vs])))
    return {k: np.asarray(v) for k, v in out.items()}


def bootstrap(payloads: Dict[str, dict], B: int, n_workers: int) -> Dict[str, Dict[str, Dict[Tuple[str, str], np.ndarray]]]:
    """For each unit: {'point': {...}, 'draws': {...}}. Draws are split into N_CHUNKS chunks with fixed seeds."""
    jobs = []
    for u, pl in payloads.items():
        jobs.append((u, "point", pl, 0, 0))
        sizes = [B // N_CHUNKS + (1 if c < B % N_CHUNKS else 0) for c in range(N_CHUNKS)]
        for c, b in enumerate(sizes):
            if b:
                jobs.append((u, c, pl, b, seed_for("boot", u, c)))
    res: Dict[str, Dict] = {u: {"point": None, "draws": []} for u in payloads}
    if n_workers > 1 and len(jobs) > 1:
        with ProcessPoolExecutor(n_workers, mp_context=get_context("spawn")) as ex:
            outs = list(ex.map(_job, jobs))
    else:
        outs = [_job(j) for j in jobs]
    for (u, kind, _, _, _), o in zip(jobs, outs):
        if kind == "point":
            res[u]["point"] = {k: float(v[0]) for k, v in o.items()}
        else:
            res[u]["draws"].append(o)
    for u in res:
        ch = res[u]["draws"]
        res[u]["draws"] = {k: np.concatenate([c[k] for c in ch]) for k in ch[0]} if ch else {}
    return res


def _job(j):
    u, kind, pl, b, seed = j
    return boot_chunk(pl, b, seed)


def ci(draws: np.ndarray) -> Tuple[float, float]:
    d = np.asarray(draws, float)
    d = d[np.isfinite(d)]
    return (float(np.quantile(d, 0.025)), float(np.quantile(d, 0.975))) if len(d) > 20 else (np.nan, np.nan)


# ================================================================== hypothesis families
def groups_for(fr: pd.DataFrame, y: np.ndarray, min_events: int):
    sex = [np.flatnonzero(fr.sex_female.to_numpy() == 1), np.flatnonzero(fr.sex_female.to_numpy() == 0)]
    if any(y[ix].sum() < min_events or (y[ix] == 0).sum() < min_events for ix in sex):
        sex = []                                             # protocol 7: groups need at least min_events events
    bands = []
    for b in ("20-39", "40-59", "60+"):
        ix = np.flatnonzero(fr.age_band.to_numpy() == b)
        if y[ix].sum() >= min_events and (y[ix] == 0).sum() >= min_events:
            bands.append(ix)
    return {"sex": sex, "age": bands}, len(bands)


def endpoint_row(fam, hyp, unit, endpoint, predicted, point_diff, draws_diff, margin=0.0, primary=True, note=""):
    if hyp == "H1":
        p_pred, p_opp = mt.p_one_sided(draws_diff, -margin, +1), mt.p_one_sided(draws_diff, -margin, -1)
    elif predicted == "higher":
        p_pred, p_opp = mt.p_one_sided(draws_diff, 0.0, +1), mt.p_one_sided(draws_diff, 0.0, -1)
    else:
        p_pred, p_opp = mt.p_one_sided(draws_diff, 0.0, -1), mt.p_one_sided(draws_diff, 0.0, +1)
    lo, hi = ci(draws_diff)
    return {"family": fam, "hypothesis": hyp, "unit": unit, "endpoint": endpoint, "predicted": predicted, "primary": primary,
            "difference": point_diff, "ci_lo": lo, "ci_hi": hi, "boot_sd": float(np.nanstd(draws_diff, ddof=1)), "p_pred": p_pred,
            "p_opp": p_opp, "margin": margin, "n_draws": int(np.isfinite(draws_diff).sum()), "note": note}


def verdicts(ep: pd.DataFrame, alpha: float) -> Tuple[pd.DataFrame, pd.DataFrame]:
    out = []
    ep = ep.copy()
    ep["p_pred_holm"], ep["p_opp_holm"] = np.nan, np.nan
    for fam, g in ep[ep.primary].groupby("family"):
        ep.loc[g.index, "p_pred_holm"] = mt.holm(g.p_pred.to_numpy())
        ep.loc[g.index, "p_opp_holm"] = mt.holm(g.p_opp.to_numpy())
    half = alpha / 2
    ep["status"] = np.where(~ep.primary, "descriptive", np.where(ep.p_pred_holm < half, "predicted",
                                                                  np.where(ep.p_opp_holm < half, "opposite", "null")))
    for (fam, h), g in ep[ep.primary].groupby(["family", "hypothesis"]):
        st = []
        for _, gu in g.groupby("unit", sort=False):
            s = list(gu.status)
            st.append("predicted" if all(x == "predicted" for x in s) else "opposite" if any(x == "opposite" for x in s) else "null")
        n_u, n_p, n_o = len(st), st.count("predicted"), st.count("opposite")
        v = "PASS" if (n_o == 0 and (n_p == n_u if h == "H1" else n_p > n_u / 2)) else "FAIL" if n_o > n_u / 2 else "NULL"
        out.append({"family": fam, "hypothesis": h, "units": n_u, "units_predicted": n_p, "units_opposite": n_o, "verdict": v})
    return ep, pd.DataFrame(out)


# ================================================================== attribution stability (H4)
def stability_rows(attr_by_seed: List[np.ndarray]) -> np.ndarray:
    """Per row: mean Spearman correlation of attribution vectors over all seed pairs."""
    S = len(attr_by_seed)
    vals = [mt.spearman_rows(attr_by_seed[a], attr_by_seed[b]) for a in range(S) for b in range(a + 1, S)]
    return np.nanmean(np.vstack(vals), axis=0)


def mean_boot(x: Dict[str, np.ndarray], B: int, seed: int) -> Dict[str, np.ndarray]:
    """Bootstrap of row means for several aligned per-row arrays (same rows resampled for all)."""
    n = len(next(iter(x.values())))
    out = {k: [] for k in x}
    for w in mt.boot_weight_iter(n, B, seed):
        for k, v in x.items():
            ok = np.isfinite(v)
            out[k].append(float((w[ok] * v[ok]).sum() / w[ok].sum()))
    return {k: np.asarray(v) for k, v in out.items()}


# ================================================================== simple helpers used by several sections
def run_mean(P: dict, model: str, runs: Sequence[Tuple[int, int]], fn, tag: str = "none") -> float:
    return float(np.mean([fn(P[(model, r, s)][tag]) for r, s in runs]))


def auprc(y, p) -> float:
    return float(average_precision_score(y, p))


def fold_aurocs(y, p, fold) -> np.ndarray:
    return np.array([mt.auroc_w(mt.Ranked(p[fold == k], y[fold == k])) for k in np.unique(fold)])


def classify_interval(sign: int, lo: float, hi: float) -> str:
    """'supported' if the whole interval lies on the prior's side of zero, 'contradicted' if wholly on the other side."""
    lo_s, hi_s = sorted((sign * lo, sign * hi))
    return "supported" if lo_s > 0 else "contradicted" if hi_s < 0 else "uncertain"


def empirical_signs(fr: pd.DataFrame, feats: Sequence[str], signs: Dict[str, int]) -> pd.DataFrame:
    """Adjusted logistic regression on the evaluation rows (descriptive, protocol 8c): Wald 95% intervals on the standardised scale;
    and the Spearman correlation between quintile index and observed event rate."""
    from scipy.stats import spearmanr
    from sklearn.linear_model import LogisticRegression
    from .prep import Prep
    pp = Prep(feats).fit(fr)
    X = pp.transform(fr)[0].astype(float)
    y = fr.y.to_numpy()
    lr = LogisticRegression(C=1e6, max_iter=5000).fit(X, y)
    p = lr.predict_proba(X)[:, 1]
    Xa = np.c_[np.ones(len(X)), X]
    cov = np.linalg.pinv(Xa.T @ (Xa * (p * (1 - p))[:, None]))
    se = np.sqrt(np.diag(cov))[1:]
    rows = []
    for f, s in signs.items():
        j = feats.index(f)
        b, lo, hi = lr.coef_[0, j], lr.coef_[0, j] - 1.96 * se[j], lr.coef_[0, j] + 1.96 * se[j]
        obs = fr[f].notna().to_numpy()
        if fr[f].nunique() > 5:
            q = pd.qcut(fr.loc[obs, f], 5, labels=False, duplicates="drop")
            rate = fr.loc[obs].groupby(q).y.mean()
            rho = float(spearmanr(rate.index, rate.values)[0]) if len(rate) > 2 else np.nan
        else:
            rate = fr.loc[obs].groupby(f).y.mean()
            rho = float(np.sign(rate.iloc[-1] - rate.iloc[0])) if len(rate) > 1 else np.nan
        cat = classify_interval(s, lo, hi)
        rows.append({"feature": f, "prior_sign": s, "adjusted_coef": b, "coef_lo": lo, "coef_hi": hi, "adjusted_evidence": cat,
                     "quintile_spearman": rho, "quintile_agrees": bool(np.sign(rho) == s) if np.isfinite(rho) else None})
    return pd.DataFrame(rows)


def conc_job(job):
    """Harrell's or Uno's C for one prediction vector (run in worker processes)."""
    c, m, r, kind, risk, t_, e_, w, tau = job
    return (c, m, r, kind, mt.concordance(risk, t_, e_, event_weight=w, tau=tau))


def parallel_map(fn, items, n_workers):
    if n_workers > 1 and len(items) > 1:
        with ProcessPoolExecutor(n_workers, mp_context=get_context("spawn")) as ex:
            return list(ex.map(fn, items, chunksize=max(1, len(items) // (4 * n_workers))))
    return [fn(x) for x in items]


In [ ]:
%%writefile ckv2_lib/ckv2/posthoc.py
"""Post hoc additions for the 28-step protocol (protocol_v2.md, section 10; label PH).

Decided after the v2 results had been seen, to cover protocol steps the v2 analysis left partly or not covered. None of these
analyses enters a hypothesis family or changes a v2 verdict; the frozen configuration and its hash are untouched.
Model fits that repeat a main-run fit use the same seed and rows, so their unmodified predictions must equal the stored ones
(checked in the notebook as a reproducibility test).
"""
from typing import Dict, List, Optional, Sequence

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

from .config import BINARY, C_GRID, MAIN_MODELS, PANEL, RUN0_EXTRA, seed_for
from .plan import fit_seed, signs_for, tid

SHAP_MODELS = ("LR", "XGB", "XGB_mono", "M1", "M2", "CKINN")
REFIT_MODELS = tuple(MAIN_MODELS) + tuple(RUN0_EXTRA) + ("SVM",)
TRANSPORT_EXTRA = ("RF", "LGBM", "LGBM_mono", "cLR", "GAM", "SVM", "MI_LR")
PROGRESSION_MODELS = ("LR", "XGB_mono", "M1", "M2", "CKINN")
TASK_LABEL = {"A": "undiagnosed diabetes", "B": "incident type 2 diabetes during follow-up"}

# One PubMed-verified reference per sign-table row (references.md); found after the table was frozen.
SIGN_CITATIONS = {
    "age_years": (17533210, "Wilson et al. 2007, Arch Intern Med (Framingham Offspring)"),
    "bmi_kgm2": (20493574, "Abdullah et al. 2010, Diabetes Res Clin Pract (meta-analysis)"),
    "waist_cm": (17494056, "Vazquez et al. 2007, Epidemiol Rev (meta-analysis)"),
    "sbp_mmHg": (26429079, "Emdin et al. 2015, J Am Coll Cardiol (4.1 million adults and meta-analysis)"),
    "dbp_mmHg": (26429079, "Emdin et al. 2015, J Am Coll Cardiol (4.1 million adults and meta-analysis)"),
    "family_history_dm": (23052052, "InterAct Consortium 2013, Diabetologia"),
    "tg_mmol": (17533210, "Wilson et al. 2007, Arch Intern Med (Framingham Offspring)"),
    "hdl_mmol": (17533210, "Wilson et al. 2007, Arch Intern Med (Framingham Offspring)"),
    "exercise_regular": (26092138, "Aune et al. 2015, Eur J Epidemiol (meta-analysis)"),
    "fpg_mmol": (17533210, "Wilson et al. 2007, Arch Intern Med (Framingham Offspring)"),
    "hba1c_pct": (20587727, "Zhang et al. 2010, Diabetes Care (systematic review)"),
}

LABELS = {"age_years": "age", "sex_female": "female sex", "bmi_kgm2": "BMI", "waist_cm": "waist circumference", "sbp_mmHg": "systolic BP",
          "dbp_mmHg": "diastolic BP", "smoker": "current smoking", "alcohol_level": "alcohol intake", "exercise_regular": "regular exercise",
          "family_history_dm": "family history of diabetes", "tg_mmol": "triglycerides", "hdl_mmol": "HDL cholesterol",
          "ldl_mmol": "LDL cholesterol", "tc_mmol": "total cholesterol", "alt_u_l": "ALT", "ast_u_l": "AST", "ggt_u_l": "GGT",
          "fpg_mmol": "fasting glucose", "hba1c_pct": "HbA1c", "tg_hdl_ratio": "TG/HDL ratio", "prediabetes_told": "told of prediabetes",
          "hypertension_told": "told of hypertension"}


def settings(cfg) -> dict:
    if cfg.mode == "full":
        return {"pair_rows": 1000, "shap_rows": 200, "n_bg": 50, "n_perm": 8, "reason_n": 50, "delta_sd": 0.25}
    return {"pair_rows": 150, "shap_rows": 30, "n_bg": 15, "n_perm": 4, "reason_n": 10, "delta_sd": 0.25}


def _logit(p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


# ------------------------------------------------------------------ probes run inside the worker
def perturbation_violations(predict, df: pd.DataFrame, tr: pd.DataFrame, signs: Dict[str, int], feats: Sequence[str], delta_sd: float,
                            tol: float) -> Dict[str, dict]:
    """Model-agnostic violation rate (step 2): continuous predictors raised by delta_sd training SDs, binary predictors moved from 0 to
    1, observed values only; a violation is a sign-adjusted change of the log-odds below -tol. Works for every model."""
    z0 = _logit(predict(df))
    res = {}
    for f, s in signs.items():
        if f not in feats:
            continue
        v = df[f].to_numpy(float)
        obs = ~np.isnan(v)
        d2 = df.copy()
        if f in BINARY:
            rows = obs & (v == 0)
            d2.loc[rows, f] = 1.0
        else:
            rows = obs
            d2.loc[rows, f] = v[rows] + delta_sd * float(np.nanstd(tr[f].to_numpy(float)))
        if not rows.any():
            res[f] = {"n": 0, "viol": 0, "mean_change": float("nan")}
            continue
        ch = s * (_logit(predict(d2)) - z0)[rows]
        res[f] = {"n": int(rows.sum()), "viol": int((ch < -tol).sum()), "mean_change": float(ch.mean())}
    return res


def permutation_shap(predict, rows: pd.DataFrame, bg: pd.DataFrame, feats: Sequence[str], n_perm: int, rng):
    """Interventional SHAP values on the log-odds scale by permutation sampling with antithetic permutations and a background sample
    of training rows. Additive by construction: phi.sum(1) == fx - base for every row."""
    feats = list(feats)
    Xr, B = rows[feats].to_numpy(float), bg[feats].to_numpy(float)
    nb, p = B.shape
    half = max(1, n_perm // 2)
    base = float(_logit(predict(pd.DataFrame(B, columns=feats))).mean())
    phi, fx = np.zeros((len(Xr), p)), np.zeros(len(Xr))
    for i in range(len(Xr)):
        perms = [rng.permutation(p) for _ in range(half)]
        perms += [q[::-1] for q in perms]
        blocks = []
        for q in perms:
            cur = B.copy()
            blocks.append(cur.copy())
            for j in q:
                cur[:, j] = Xr[i, j]
                blocks.append(cur.copy())
        v = _logit(predict(pd.DataFrame(np.vstack(blocks), columns=feats))).reshape(len(perms), p + 1, nb).mean(axis=2)
        for k, q in enumerate(perms):
            phi[i, q] += np.diff(v[k])
        phi[i] /= len(perms)
        fx[i] = v[0, -1]
    return phi, base, fx


def reasoning_inputs(predict, rows: pd.DataFrame, tr: pd.DataFrame, signs: Dict[str, int], feats: Sequence[str], delta_sd: float, tol: float,
                     panel: Sequence[str]) -> dict:
    """Inputs for the patient-level reasoning text (steps 15 and 27): predicted probability, point-wise check of every constrained
    predictor, and the model's response when each blood-panel value is set to the training median (a model response, not advice)."""
    p0 = predict(rows)
    z0 = _logit(p0)
    cfeats = [f for f in signs if f in feats]
    viol = np.zeros((len(rows), len(cfeats)), bool)
    tested = np.zeros((len(rows), len(cfeats)), bool)
    for j, f in enumerate(cfeats):
        v = rows[f].to_numpy(float)
        obs = ~np.isnan(v)
        d2 = rows.copy()
        if f in BINARY:
            d2.loc[obs, f] = 1.0 - v[obs]
            direction = np.where(v == 0, 1.0, -1.0)
        else:
            d2.loc[obs, f] = v[obs] + delta_sd * float(np.nanstd(tr[f].to_numpy(float)))
            direction = np.ones(len(v))
        ch = signs[f] * direction * (_logit(predict(d2)) - z0)
        viol[:, j], tested[:, j] = obs & (ch < -tol), obs
    cf = [f for f in panel if f in feats]
    med = {f: float(np.nanmedian(tr[f].to_numpy(float))) for f in cf}
    p_cf = np.full((len(rows), len(cf)), np.nan)
    for j, f in enumerate(cf):
        obs = rows[f].notna().to_numpy()
        d2 = rows.copy()
        d2.loc[obs, f] = med[f]
        p_cf[obs, j] = predict(d2)[obs]
    return {"p": np.asarray(p0, float), "constrained": cfeats, "viol_point": viol, "tested": tested, "cf_feats": cf, "medians": med,
            "p_cf": p_cf, "x": rows[list(feats)].to_numpy(float), "feats": list(feats)}


def reasoning_text(rec: dict, i: int, task: str) -> str:
    """Fixed template (steps 15 and 27): top three contributions with direction, constraint consistency, one blood-panel counterfactual.
    Worded as the model's behaviour; never as a diagnosis or advice."""
    feats, phi, x, p = rec["feats"], rec["phi"][i], rec["x"][i], rec["p"][i]
    order = [j for j in np.argsort(-np.abs(phi)) if abs(phi[j]) > 0][:3]
    parts = []
    for j in order:
        val = "not recorded" if np.isnan(x[j]) else f"{x[j]:.3g}"
        parts.append(f"{LABELS.get(feats[j], feats[j])} ({val}) {'raises' if phi[j] > 0 else 'lowers'} the model's estimate")
    text = [f"Predicted probability of {TASK_LABEL[task]}: {p:.1%}.",
            "Largest contributions on the log-odds scale: " + ("; ".join(parts) if parts else "none") + "."]
    bad = [LABELS.get(f, f) for f, b in zip(rec["constrained"], rec["viol_point"][i]) if b]
    text.append("At these values the model's response agrees with every clinical prior it was given." if not bad
                else "At these values the model's response runs against the clinical prior for: " + ", ".join(bad) + ".")
    best, best_drop = None, 0.0
    for j, f in enumerate(rec["cf_feats"]):
        if np.isfinite(rec["p_cf"][i, j]) and p - rec["p_cf"][i, j] > best_drop:
            best, best_drop = j, p - rec["p_cf"][i, j]
    if best is None:
        text.append("Setting any one blood-panel value to the training median would not lower the model's estimate.")
    else:
        f = rec["cf_feats"][best]
        text.append(f"With {LABELS.get(f, f)} at the training median ({rec['medians'][f]:.3g}) the model's predicted probability would be "
                    f"{rec['p_cf'][i, best]:.1%} instead of {p:.1%}.")
    text.append("This describes the model's behaviour; it is not a diagnosis or a recommendation.")
    return " ".join(text)


# ------------------------------------------------------------------ weights
def balanced_weights(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y).astype(int)
    n, n1 = len(y), max(int(y.sum()), 1)
    w = np.where(y == 1, n / (2 * n1), n / (2 * max(n - n1, 1)))
    return w / w.mean()


def group_weights(fr: pd.DataFrame) -> np.ndarray:
    """Inverse-frequency weights over sex x age band, mean 1 (ablation M5, the v1 fairness weighting)."""
    g = fr["sex"].astype(str) + "|" + fr["age_band"].astype(str)
    w = 1.0 / g.map(g.value_counts()).to_numpy(float)
    return w / w.mean()


def wilson(k: int, n: int, z: float = 1.959964) -> tuple:
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return float(c - h), float(c + h)


# ------------------------------------------------------------------ task specifications
def svm_tune_specs(cohort: str, dev: pd.DataFrame, feats: Sequence[str], cfg) -> List[dict]:
    folds = list(StratifiedKFold(cfg.tune_folds, shuffle=True, random_state=seed_for("devcv", cohort)).split(dev, dev.y))
    grid = C_GRID if cfg.mode == "full" else C_GRID[1:3]
    return [{"id": tid("phtune", cohort, "SVM", f"c{ci}", f"k{k}"), "model": "SVM", "feats": list(feats), "hp": {"C": c},
             "seed": seed_for("devfit", cohort, k), "train_frame": f"{cohort}_dev", "train_idx": i,
             "tests": [{"tag": "val", "frame": f"{cohort}_dev", "idx": j}], "_val_idx": j, "_model": "SVM", "_ci": ci}
            for ci, c in enumerate(grid) for k, (i, j) in enumerate(folds)]


def refit_specs(cohort: str, folds, feats: Sequence[str], HP, LAM, cfg) -> List[dict]:
    """Repeat 0, seed index 0, every model: identical to the main fit, plus BMI 24/35 pairs, perturbation violations, SHAP and the
    reasoning inputs."""
    st = settings(cfg)
    sg = signs_for("T2", feats)
    specs = []
    for k, (i, j) in enumerate(folds[0]):
        pr = np.sort(np.random.default_rng(seed_for("phpairs", cohort, k)).choice(j, min(st["pair_rows"], len(j)), replace=False))
        tests = [{"tag": "none", "frame": cohort, "idx": j}]
        if "bmi_kgm2" in feats:
            tests += [{"tag": "bmi24", "frame": cohort, "idx": pr, "set": {"bmi_kgm2": 24.0}},
                      {"tag": "bmi35", "frame": cohort, "idx": pr, "set": {"bmi_kgm2": 35.0}}]
        sr = np.sort(np.random.default_rng(seed_for("phshap", cohort)).choice(j, min(st["shap_rows"], len(j)), replace=False))
        for m in REFIT_MODELS:
            hp_key = {"MI_LR": "LR", "LR_aug": "LR", "XGB_aug": "XGB"}.get(m, m)
            spec = {"id": tid("ph", cohort, "T2", "refit", "r0", f"k{k}", m, "s0"), "model": m, "feats": list(feats), "hp": HP[(cohort, hp_key)],
                    "seed": fit_seed(cohort, "T2", 0, k, 0), "signs": sg, "lam": LAM[(cohort, "T2")] if m == "CKINN" else 0.0,
                    "train_frame": cohort, "train_idx": i, "tests": tests,
                    "perturb": {"frame": cohort, "idx": pr, "signs": sg, "delta_sd": st["delta_sd"]}, "_k": k, "_model": m, "_pairs": pr}
            if k == 0 and m in SHAP_MODELS:
                spec["shap"] = {"frame": cohort, "idx": sr, "n_bg": st["n_bg"], "n_perm": st["n_perm"], "seed": seed_for("phshapbg", cohort)}
                if m == "CKINN":
                    spec["reason"] = {"n": st["reason_n"], "signs": sg, "delta_sd": st["delta_sd"], "panel": [f for f in PANEL if f in feats]}
            specs.append(spec)
    return specs


def variant_list(cohort: str, feats: Sequence[str]) -> List[tuple]:
    """(variant, model, features, sign table, weights) for the repeat-0 post hoc variants."""
    sg = signs_for("T2", feats)
    out = []
    if "dbp_mmHg" in feats:
        out.append(("dbp_constrained", "CKINN", list(feats), dict(sg, dbp_mmHg=1), None))
    if "hdl_mmol" in sg:
        out.append(("wrong_hdl", "CKINN", list(feats), dict(sg, hdl_mmol=-sg["hdl_mmol"]), None))
    out.append(("m5_group_weights", "CKINN", list(feats), sg, "group"))
    for m in ("LR", "M2", "CKINN"):
        out.append(("class_weights", m, list(feats), sg, "balanced"))
    if "tg_mmol" in feats and "hdl_mmol" in feats:
        for m in ("LR", "M2", "CKINN"):
            out.append(("tg_hdl_ratio", m, list(feats) + ["tg_hdl_ratio"], dict(sg, tg_hdl_ratio=1), None))
    if cohort == "NHANES":
        for m in ("LR", "M2", "CKINN"):
            out.append(("history_questions", m, list(feats) + ["prediabetes_told", "hypertension_told"],
                        dict(sg, prediabetes_told=1, hypertension_told=1), None))
    return out


def variant_specs(cohort: str, folds, feats: Sequence[str], HP, LAM, cfg, fr: pd.DataFrame) -> List[dict]:
    specs = []
    for k, (i, j) in enumerate(folds[0]):
        for v, m, fts, sg, wkind in variant_list(cohort, feats):
            w = None
            if wkind == "group":
                w = group_weights(fr.iloc[i])
            elif wkind == "balanced":
                w = balanced_weights(fr.y.to_numpy()[i])
            specs.append({"id": tid("ph", cohort, "T2", v, "r0", f"k{k}", m, "s0"), "model": m, "feats": fts, "hp": HP[(cohort, m)],
                          "seed": fit_seed(cohort, "T2", 0, k, 0), "signs": sg, "lam": LAM[(cohort, "T2")] if m == "CKINN" else 0.0,
                          "train_frame": cohort, "train_idx": i, "train_w": w, "tests": [{"tag": "none", "frame": cohort, "idx": j}],
                          "_variant": v, "_model": m, "_k": k})
    return specs


def pool_indicator_specs(EVAL, FOLDS, pooled_feats: Sequence[str], HP, LAM, cfg, cap: int = 20000) -> List[dict]:
    """M4 of the 28-step ablation: pooled NAGALA + RICH training with a cohort indicator. Same rows, cap and seeds as the v2 pooled
    ablation, so the only difference is the indicator. Evaluated within the two training cohorts only (it has no value elsewhere)."""
    off = {"NAGALA": 0, "RICH": len(EVAL["NAGALA"])}
    feats = list(pooled_feats) + ["cohort_rich"]
    specs = []
    for k in range(cfg.n_splits):
        tr, tests = [], []
        for c in ("NAGALA", "RICH"):
            i, j = FOLDS[c][0][k]
            if len(i) > cap:
                i = np.sort(train_test_split(i, train_size=cap, stratify=EVAL[c].y.to_numpy()[i], random_state=seed_for("poolcap", c, k))[0])
            tr.append(i + off[c])
            tests.append({"tag": c, "frame": "POOL_IND", "idx": j + off[c]})
        for m in ("M2", "CKINN"):
            specs.append({"id": tid("phpool", m, f"k{k}"), "model": m, "feats": feats, "hp": HP[("RICH", m)], "seed": seed_for("pool", k),
                          "signs": signs_for("T2", feats), "lam": LAM[("RICH", "T2")] if m == "CKINN" else 0.0, "train_frame": "POOL_IND",
                          "train_idx": np.concatenate(tr), "tests": tests, "_k": k, "_model": m})
    return specs


def transport_specs(EVAL, FOLDS, shared: Sequence[str], HP, cfg, models=TRANSPORT_EXTRA) -> List[dict]:
    """Transport for the models the v2 transport analysis left out (seed 0), same seeds and rows as v2 (step 12 matrix)."""
    specs = []
    for src, tgt in (("NAGALA", "RICH"), ("RICH", "NAGALA")):
        sg = signs_for("T2", shared)
        for m in models:
            hp_key = {"MI_LR": "LR"}.get(m, m)
            specs.append({"id": tid("phtrext", src, tgt, m, "s0"), "model": m, "feats": list(shared), "hp": HP[(src, hp_key)], "signs": sg,
                          "seed": seed_for("tr_ext", src, 0), "train_frame": src, "train_idx": np.arange(len(EVAL[src])),
                          "tests": [{"tag": "tgt", "frame": tgt, "idx": np.arange(len(EVAL[tgt]))}], "_src": src, "_tgt": tgt, "_model": m})
            for k, (i, j) in enumerate(FOLDS[tgt][0]):
                specs.append({"id": tid("phtrref", tgt, m, f"k{k}", "s0"), "model": m, "feats": list(shared), "hp": HP[(tgt, hp_key)],
                              "signs": sg, "seed": seed_for("tr_ref", tgt, k, 0), "train_frame": tgt, "train_idx": i,
                              "tests": [{"tag": "none", "frame": tgt, "idx": j}], "_tgt": tgt, "_model": m, "_k": k, "_r": 0, "_s": 0})
    return specs


def progression_frame(ev_rich: pd.DataFrame) -> pd.DataFrame:
    """Proposal section 3C, one step: among RICH evaluation participants with baseline FPG below 5.6 mmol/L, the outcome is FPG of at
    least 5.6 mmol/L at the final visit or diabetes during follow-up. The transition time is not observed."""
    rp = ev_rich[ev_rich.fpg_mmol < 5.6].copy()
    rp["y"] = ((rp.fpg_final >= 5.6) | (rp.label == 1)).astype(int)
    return rp.reset_index(drop=True)


def progression_specs(folds, feats: Sequence[str], HP, LAM, cfg) -> List[dict]:
    sg = signs_for("T2", feats)
    return [{"id": tid("phprog", m, f"k{k}"), "model": m, "feats": list(feats), "hp": HP[("RICH", m)], "seed": seed_for("phprog", k),
             "signs": sg, "lam": LAM[("RICH", "T2")] if m == "CKINN" else 0.0, "train_frame": "RICH_prog", "train_idx": i,
             "tests": [{"tag": "none", "frame": "RICH_prog", "idx": j}], "_k": k, "_model": m, "_r": 0, "_s": 0}
            for k, (i, j) in enumerate(folds[0]) for m in PROGRESSION_MODELS]


# ------------------------------------------------------------------ audit table (step 5)
UNAVAILABLE = [("ethnicity / ancestry", "NHANES only (descriptive fairness groups); absent in NAGALA and RICH"),
               ("previous prediabetes", "NHANES only (DIQ160, cycles 2005-06 onwards); post hoc exploratory input"),
               ("cardiovascular disease", "not in any of the three files used"),
               ("hypertension diagnosis", "NHANES only (BPQ020); post hoc exploratory input"),
               ("glucose-lowering medication", "NHANES only; used to define the exclusions of the detection label, not as an input"),
               ("diet", "not in any of the three files"), ("sleep", "not in any of the three files"),
               ("earlier glucose readings", "not in any of the three files"),
               ("weight and height", "NAGALA weight and RICH height and weight exist; BMI is used instead (derived from them)"),
               ("socio-economic status", "not in any of the three files")]


def audit_table(TASK: Dict[str, pd.DataFrame], FEATS: Dict[tuple, list], features: Sequence[str]) -> pd.DataFrame:
    rows = []
    for f in features:
        r = {"predictor": f}
        for c in ("NAGALA", "RICH", "NHANES"):
            r[f"missing_pct_{c}"] = 100 * float(TASK[c][f].isna().mean()) if f in TASK[c] else 100.0
            r[f"used_{c}"] = ("T2" if f in FEATS.get((c, "T2"), []) else "") + (" T3" if f in FEATS.get((c, "T3"), []) and f not in FEATS.get((c, "T2"), []) else "")
        r["available_at_prediction_time"] = "yes (baseline measurement or questionnaire)"
        r["leakage_risk"] = ("high: defines the detection label, banned for NHANES; incident task only in T3" if f in ("fpg_mmol", "hba1c_pct")
                             else "moderate: implies an earlier glucose test; exploratory only" if f == "prediabetes_told" else "low")
        r["fairness_relevance"] = "group variable for subgroup analysis (also a predictor)" if f in ("sex_female", "age_years") else "none"
        r["prior_sign"] = {1: "+", -1: "-"}.get(signs_for("T3", [f]).get(f), "unconstrained")
        rows.append(r)
    return pd.DataFrame(rows)


In [ ]:
%%writefile ckv2_lib/tests/test_ckv2.py
"""Unit tests for the v2 library. Toy arrays only (CLAUDE.md); no cohort data are read here.
Run: python -m pytest -q realdata_v2/tests  (the notebook runs the same checks in its test cells)."""
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pytest
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from ckv2 import metrics as mt  # noqa: E402
from ckv2.config import make_config, SIGNS_T2  # noqa: E402
from ckv2.models import ConstrainedLR, Fitted, threshold_for_sens, threshold_for_spec  # noqa: E402
from ckv2.nets import Net, clinical_penalty  # noqa: E402
from ckv2.data import repeated_folds  # noqa: E402

CFG = make_config("smoke")


def toy_binary(n=4000, seed=0, ties=False):
    rng = np.random.default_rng(seed)
    p = rng.uniform(0.01, 0.6, n)
    if ties:
        p = np.round(p, 2)
    y = (rng.uniform(size=n) < p).astype(int)
    return y, p


def toy_frame(n=3000, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({"age_years": rng.uniform(20, 80, n), "sex_female": rng.integers(0, 2, n).astype(float),
                      "bmi_kgm2": rng.normal(24, 3, n), "sbp_mmHg": rng.normal(125, 15, n), "tg_mmol": np.exp(rng.normal(0.2, 0.5, n)),
                      "hdl_mmol": rng.normal(1.4, 0.3, n)})
    z = -5 + 0.05 * (d.age_years - 50) + 0.15 * (d.bmi_kgm2 - 24) + 0.8 * np.log(d.tg_mmol) - 1.0 * (d.hdl_mmol - 1.4)
    d["y"] = (rng.uniform(size=n) < 1 / (1 + np.exp(-z - 2))).astype(int)
    d.loc[rng.random(n) < 0.2, "hdl_mmol"] = np.nan
    return d


# ------------------------------------------------------------------ metrics
def test_auroc_matches_sklearn_with_ties_and_weights():
    from sklearn.metrics import roc_auc_score
    y, p = toy_binary(ties=True)
    r = mt.Ranked(p, y)
    assert abs(mt.auroc_w(r) - roc_auc_score(y, p)) < 1e-12
    w = np.random.default_rng(2).integers(0, 3, len(y)).astype(float)
    assert abs(mt.auroc_w(r, w) - roc_auc_score(y, p, sample_weight=w)) < 1e-12


def test_bootstrap_weights_equal_explicit_resample():
    from sklearn.metrics import roc_auc_score
    y, p = toy_binary(2000, 3)
    w = next(mt.boot_weight_iter(len(y), 1, 7))
    idx = np.repeat(np.arange(len(y)), w.astype(int))
    assert abs(mt.auroc_w(mt.Ranked(p, y), w) - roc_auc_score(y[idx], p[idx])) < 1e-12
    assert abs(mt.brier_w(y, p, w) - np.mean((p[idx] - y[idx]) ** 2)) < 1e-12


def test_calibration_known_answers():
    y, p = toy_binary(20000, 4)
    a, b = mt.calib_w(y, p)
    assert abs(b - 1) < 0.08 and abs(mt.citl_w(y, p)) < 0.08
    assert mt.citl_w(y, np.clip(p * 1.8, 0, 0.99)) < -0.3
    from sklearn.linear_model import LogisticRegression
    z = mt.logit(mt.clip(p))[:, None]
    lr = LogisticRegression(C=1e10, max_iter=1000).fit(z, y)
    assert abs(lr.coef_[0, 0] - b) < 1e-4 and abs(lr.intercept_[0] - a) < 1e-4
    assert mt.ece_w(y, p) < 0.02 and mt.ece_w(y, np.clip(p * 1.5, 0, 1)) > 0.05


def test_tpr_at_fpr():
    from sklearn.metrics import roc_curve
    y, p = toy_binary(5000, 5)
    f, t, _ = roc_curve(y, p)
    assert abs(mt.tpr_at_fpr_w(mt.Ranked(p, y), 0.1) - np.interp(0.1, f, t)) < 2e-3
    yy = np.r_[np.zeros(50), np.ones(50)]
    pp = np.r_[np.linspace(0, .4, 50), np.linspace(.6, 1, 50)]
    assert mt.tpr_at_fpr_w(mt.Ranked(pp, yy), 0.1) == 1.0


def test_delong_against_bootstrap_and_identity():
    y, p = toy_binary(3000, 6)
    rng = np.random.default_rng(1)
    q = np.clip(p + rng.normal(0, 0.1, len(p)), 0, 1)
    d = mt.delong_paired(y, p, q)
    from sklearn.metrics import roc_auc_score
    assert abs(d["auc1"] - roc_auc_score(y, p)) < 1e-12 and abs(d["auc2"] - roc_auc_score(y, q)) < 1e-12
    rp, rq = mt.Ranked(p, y), mt.Ranked(q, y)
    bs = [mt.auroc_w(rp, w) - mt.auroc_w(rq, w) for w in mt.boot_weight_iter(len(y), 400, 3)]
    assert 0.7 < d["se"] / np.std(bs) < 1.3
    assert mt.delong_paired(y, p, p)["diff"] == 0.0


def test_net_benefit():
    y = np.array([1, 0, 1, 0, 0, 0, 0, 0, 0, 1])
    p = np.array([.9, .8, .7, .1, .1, .1, .1, .1, .1, .05])
    # t = 0.5: TP 2, FP 1 -> (2 - 1 * 1) / 10
    assert abs(mt.net_benefit(y, p, 0.5) - 0.1) < 1e-12
    assert abs(mt.net_benefit_all(y, 0.5) - (0.3 - 0.7)) < 1e-12


def test_censoring_metrics_against_reference():
    rng = np.random.default_rng(8)
    n = 600
    risk = rng.normal(size=n)
    tt = rng.exponential(1 / np.exp(0.7 * risk))
    cc = rng.exponential(1.5, n)
    time, event = np.minimum(tt, cc), (tt <= cc).astype(int)
    lifelines = pytest.importorskip("lifelines")
    from lifelines.utils import concordance_index
    assert abs(mt.concordance(risk, time, event) - concordance_index(time, -risk, event)) < 1e-10
    sksurv = pytest.importorskip("sksurv")
    from sksurv.metrics import concordance_index_ipcw, cumulative_dynamic_auc, brier_score
    from sksurv.util import Surv
    s = Surv.from_arrays(event.astype(bool), time)
    tau = float(np.quantile(time, 0.6))
    ut, G = mt.km_censoring(time, event)
    auc_ref = cumulative_dynamic_auc(s, s, risk, [tau])[0][0]
    assert abs(mt.td_auc(risk, time, event, tau, ut, G) - auc_ref) < 1e-6
    w = np.where(event == 1, 1 / np.maximum(mt.G_at(ut, G, time), 1e-12) ** 2, 0)
    c_uno = mt.concordance(risk, time, event, event_weight=w, tau=tau)
    assert abs(c_uno - concordance_index_ipcw(s, s, risk, tau=tau)[0]) < 5e-3
    prob = 1 / (1 + np.exp(-risk))
    bs_ref = brier_score(s, s, 1 - prob[:, None], [tau])[1][0]
    assert abs(mt.ipcw_brier(prob, time, event, tau, ut, G) - bs_ref) < 5e-3


def test_holm_tost_corrected_t():
    assert abs(mt.holm([0.01, 0.04, 0.03])[0] - 0.03) < 1e-12 and mt.holm([0.01, 0.04, 0.03])[1] == 0.06
    d = np.r_[np.full(10, 0.0), np.full(10, 1.0)]
    assert mt.p_one_sided(d, 0.5, +1) == (1 + 10) / 21
    r = mt.corrected_t(np.array([0.01, 0.02, 0.0, 0.015, 0.012]), 100, 400)
    assert r["lo"] < r["mean"] < r["hi"] and r["df"] == 4


def test_cluster_bootstrap_weights():
    cl = np.repeat(np.arange(5), [3, 4, 5, 6, 7])
    w = next(mt.cluster_weight_iter(cl, 1, 0))
    for c in range(5):
        assert len(set(w[cl == c])) == 1
    assert w.sum() > 0


def test_spearman_rows():
    A = np.array([[1, 2, 3, 4.0], [4, 3, 2, 1.0]])
    B = np.array([[10, 20, 30, 40.0], [1, 2, 3, 4.0]])
    assert np.allclose(mt.spearman_rows(A, B), [1.0, -1.0])


# ------------------------------------------------------------------ thresholds, folds
def test_thresholds():
    y, p = toy_binary(20000, 9)
    t = threshold_for_spec(y, p, 0.9)
    assert abs((p[y == 0] < t).mean() - 0.9) < 0.01
    t2 = threshold_for_sens(y, p, 0.8)
    assert (p[y == 1] >= t2).mean() >= 0.8 and (p[y == 1] >= t2).mean() < 0.81


def test_repeated_folds_partition_and_stratify():
    y = (np.random.default_rng(0).random(1000) < 0.1).astype(int)
    rf = repeated_folds(y, "toy", 5, 3)
    for rep in rf:
        test = np.concatenate([j for _, j in rep])
        assert np.array_equal(np.sort(test), np.arange(1000))
        for i, j in rep:
            assert len(np.intersect1d(i, j)) == 0 and abs(y[j].mean() - y.mean()) < 0.02
    assert not np.array_equal(rf[0][0][1], rf[1][0][1])


# ------------------------------------------------------------------ constrained models
def test_constrained_lr_matches_sklearn_when_unconstrained_and_respects_bounds():
    from sklearn.linear_model import LogisticRegression
    rng = np.random.default_rng(0)
    X = rng.normal(size=(2000, 3))
    y = (rng.random(2000) < 1 / (1 + np.exp(-(X @ [1.0, -0.5, 0.3]) + 1))).astype(int)
    a = ConstrainedLR(1.0, [0, 0, 0]).fit(X, y)
    b = LogisticRegression(C=1.0, max_iter=2000).fit(X, y)
    assert np.allclose(a.coef_, b.coef_[0], atol=1e-4) and abs(a.intercept_ - b.intercept_[0]) < 1e-4
    c = ConstrainedLR(1.0, [-1, 1, 0]).fit(X, y)            # both signs contradict the data -> held at 0
    assert c.coef_[0] <= 1e-12 and c.coef_[1] >= -1e-12


def _is_monotone(model, df, feat, sign):
    grid = np.quantile(df[feat].dropna(), np.linspace(0.05, 0.95, 9))
    base = df.iloc[:200].copy()
    prev = None
    for v in grid:
        base[feat] = v
        cur = model.predict(base)
        if prev is not None and np.any(sign * (cur - prev) < -1e-9):
            return False
        prev = cur
    return True


@pytest.mark.parametrize("name", ["XGB_mono", "LGBM_mono", "cLR"])
def test_monotone_baselines_are_monotone(name):
    d = toy_frame()
    feats = ["age_years", "sex_female", "bmi_kgm2", "sbp_mmHg", "tg_mmol", "hdl_mmol"]
    hp = {"C": 1.0, "max_depth": 3, "learning_rate": 0.1, "n_estimators": 60, "num_leaves": 7}
    m = Fitted(name, feats, hp, 3, CFG, signs=SIGNS_T2).fit(d, d.y.to_numpy())
    for f in ("age_years", "bmi_kgm2", "sbp_mmHg", "tg_mmol", "hdl_mmol"):
        assert _is_monotone(m, d, f, SIGNS_T2[f]), (name, f)


@pytest.mark.parametrize("name", ["LR", "GAM", "RF", "XGB", "LGBM", "MI_LR", "LR_aug", "XGB_aug", "M1", "M2", "SVM"])
def test_every_model_returns_valid_probabilities(name):
    d = toy_frame()
    feats = ["age_years", "sex_female", "bmi_kgm2", "sbp_mmHg", "tg_mmol", "hdl_mmol"]
    hp = {"C": 1.0, "max_depth": 4, "min_samples_leaf": 20, "learning_rate": 0.1, "n_estimators": 60, "num_leaves": 7,
          "hidden": (32, 16), "dropout": 0.1, "lr": 3e-3, "weight_decay": 1e-4, "enc_dim": 8}
    m = Fitted(name, feats, hp, 3, CFG).fit(d.iloc[:2400], d.y.to_numpy()[:2400])
    p = m.predict(d.iloc[2400:])
    assert p.shape == (600,) and np.isfinite(p).all() and p.min() >= 0 and p.max() <= 1


# ------------------------------------------------------------------ CKINN
def test_penalty_analytic_and_zero_for_monotone():
    class Toy(torch.nn.Module):
        def __init__(self, coef):
            super().__init__()
            self.coef = torch.nn.Parameter(torch.tensor(coef, dtype=torch.float32))

        def forward(self, X, M):
            return (X * M) @ self.coef
    g = torch.Generator().manual_seed(1)
    X, M = torch.randn(200, 4, generator=g), torch.ones(200, 4)
    M[:50, 0] = 0.0
    X[:, 3] = (torch.rand(200, generator=g) < 0.5).float()
    signs, isb = torch.tensor([1.0, -1.0, 0.0, 1.0]), torch.tensor([False, False, False, True])
    pen, rate = clinical_penalty(Toy([0.8, -0.3, -5.0, 0.4]), X, M, signs, isb)
    assert pen.item() == 0.0 and rate == 0.0
    pen, rate = clinical_penalty(Toy([-0.5, 0.5, 2.0, -0.7]), X, M, signs, isb)
    assert abs(pen.item() - (150 * 0.25 + 200 * 0.25 + 200 * 0.49) / 550) < 1e-5 and rate == 1.0


def test_ckinn_lambda0_equals_m2_and_penalty_changes_fit():
    d = toy_frame(2000)
    feats = ["age_years", "sex_female", "bmi_kgm2", "sbp_mmHg", "tg_mmol", "hdl_mmol"]
    hp = {"hidden": (32, 16), "dropout": 0.1, "lr": 3e-3, "weight_decay": 1e-4, "enc_dim": 8}
    y = d.y.to_numpy()
    m2 = Fitted("M2", feats, hp, 5, CFG).fit(d, y)
    c0 = Fitted("CKINN", feats, hp, 5, CFG, signs=SIGNS_T2, lam=0.0).fit(d, y)
    c1 = Fitted("CKINN", feats, hp, 5, CFG, signs=SIGNS_T2, lam=10.0).fit(d, y)
    assert np.array_equal(m2.predict(d), c0.predict(d))
    assert not np.array_equal(m2.predict(d), c1.predict(d)) and max(c1.hist["pen"]) > 0


def test_classify_interval_both_signs():
    from ckv2.analysis import classify_interval
    assert classify_interval(1, 0.1, 0.3) == "supported" and classify_interval(1, -0.3, -0.1) == "contradicted"
    assert classify_interval(-1, -0.3, -0.1) == "supported" and classify_interval(-1, 0.1, 0.3) == "contradicted"
    assert classify_interval(-1, -0.07, 0.03) == "uncertain" and classify_interval(1, -0.07, 0.03) == "uncertain"


def test_select_lambda_one_se_rule():
    from ckv2.plan import select_lambda
    rows = []
    for lam, base in ((0.0, 0.100), (0.1, 0.1001), (1.0, 0.1004), (10.0, 0.120)):
        rows += [{"lam": lam, "score": base + e} for e in (-0.001, 0.0, 0.001)]
    lam, tab = select_lambda(pd.DataFrame(rows), (0.1, 1.0, 10.0))
    assert lam == 1.0 and tab.chosen.sum() == 1          # 1.0 is within one SE of the best (lambda 0); 10 is not
    rows = [{"lam": l, "score": 0.1 + (0.05 if l > 0 else 0) + e} for l in (0.0, 0.1, 1.0) for e in (-1e-4, 0, 1e-4)]
    assert select_lambda(pd.DataFrame(rows), (0.1, 1.0))[0] == 0.1   # none qualifies: the smallest grid value


def test_verdict_rules():
    from ckv2.analysis import verdicts
    def row(h, unit, p_pred, p_opp):
        return {"family": "A", "hypothesis": h, "unit": unit, "endpoint": "e", "primary": True, "p_pred": p_pred, "p_opp": p_opp}
    ep = pd.DataFrame([row("H1", "u1", 1e-4, 1.0), row("H1", "u2", 1e-4, 1.0), row("H1", "u3", 0.5, 0.5),
                       row("H2", "u1", 1e-4, 1.0), row("H2", "u2", 1e-4, 1.0), row("H2", "u3", 0.5, 0.5),
                       row("H3", "u1", 1.0, 1e-4), row("H3", "u2", 1.0, 1e-4)])
    _, v = verdicts(ep, 0.05)
    v = v.set_index("hypothesis").verdict
    assert v["H1"] == "NULL" and v["H2"] == "PASS" and v["H3"] == "FAIL"


def test_groups_for_minimum_events():
    from ckv2.analysis import groups_for
    n = 600
    fr = pd.DataFrame({"sex_female": np.r_[np.ones(300), np.zeros(300)],
                       "age_band": np.tile(["20-39", "40-59", "60+"], 200)})
    y = np.zeros(n)
    y[:25] = 1                                            # 25 events in women, 5 in men
    y[300:305] = 1
    g, nb = groups_for(fr, y, 20)
    assert g["sex"] == [] and nb == 0
    y[300:330] = 1
    y[np.arange(0, 600, 3)[:30]] = 1
    g, nb = groups_for(fr, y, 20)
    assert len(g["sex"]) == 2 and nb >= 1


def test_survey_replicate_weights():
    st = np.repeat([1, 2], 40)
    psu = np.tile(np.repeat([1, 2], 20), 2)
    base = np.ones(80) * 2.0
    reps = list(mt.strata_psu_weight_iter(st, psu, base, 200, 3))
    assert all((r >= 0).all() for r in reps)
    assert abs(np.mean([r.sum() for r in reps]) / base.sum() - 1) < 0.05
    r0 = reps[0]
    for s in (1, 2):
        for p in (1, 2):
            assert len(set(r0[(st == s) & (psu == p)])) == 1


# ------------------------------------------------------------------ post hoc additions (PH)
def _linear_predict(beta, feats):
    def predict(df):
        z = sum(b * np.nan_to_num(df[f].to_numpy(float)) for b, f in zip(beta, feats))
        return 1 / (1 + np.exp(-z))
    return predict


def test_permutation_shap_is_exact_for_a_linear_logit_and_additive():
    from ckv2.posthoc import permutation_shap
    rng = np.random.default_rng(0)
    feats = ["a", "b", "c"]
    beta = [0.8, -0.5, 0.0]
    rows = pd.DataFrame(rng.normal(size=(5, 3)), columns=feats)
    bg = pd.DataFrame(rng.normal(size=(40, 3)), columns=feats)
    phi, base, fx = permutation_shap(_linear_predict(beta, feats), rows, bg, feats, 4, np.random.default_rng(1))
    expected = (rows.to_numpy() - bg.to_numpy().mean(0)) * np.array(beta)
    assert np.allclose(phi, expected, atol=1e-8)
    assert np.allclose(phi.sum(1), fx - base, atol=1e-8)


def test_perturbation_violations_monotone_and_anti_monotone():
    from ckv2.posthoc import perturbation_violations
    rng = np.random.default_rng(2)
    df = pd.DataFrame({"bmi_kgm2": rng.normal(25, 3, 300), "hdl_mmol": rng.normal(1.3, 0.3, 300), "exercise_regular": rng.integers(0, 2, 300).astype(float)})
    df.loc[:9, "bmi_kgm2"] = np.nan
    signs = {"bmi_kgm2": 1, "hdl_mmol": -1, "exercise_regular": -1}
    good = perturbation_violations(_linear_predict([0.2, -1.0, -0.5], list(df)), df, df, signs, list(df), 0.25, 1e-6)
    bad = perturbation_violations(_linear_predict([-0.2, 1.0, 0.5], list(df)), df, df, signs, list(df), 0.25, 1e-6)
    assert all(v["viol"] == 0 for v in good.values())
    assert bad["bmi_kgm2"]["viol"] == bad["bmi_kgm2"]["n"] == 290
    assert bad["exercise_regular"]["n"] == int((df.exercise_regular == 0).sum()) == bad["exercise_regular"]["viol"]


def test_reasoning_text_wording_and_flags():
    from ckv2.posthoc import reasoning_inputs, reasoning_text
    rng = np.random.default_rng(3)
    df = pd.DataFrame({"bmi_kgm2": rng.normal(25, 3, 50), "tg_mmol": np.exp(rng.normal(0.2, 0.4, 50)), "hdl_mmol": rng.normal(1.3, 0.3, 50)})
    feats = list(df)
    for beta, consistent in (([0.2, 0.8, -1.0], True), ([0.2, -0.8, -1.0], False)):
        pred = _linear_predict(beta, feats)
        rec = reasoning_inputs(pred, df.iloc[:5], df, {"bmi_kgm2": 1, "tg_mmol": 1, "hdl_mmol": -1}, feats, 0.25, 1e-6, ["tg_mmol", "hdl_mmol"])
        rec["phi"] = np.tile(np.array(beta), (5, 1))
        txt = reasoning_text(rec, 0, "B")
        low = txt.lower()
        for word in ("should", "lose", "prevent", "you will", "advise"):
            assert word not in low, word
        assert ("agrees with every clinical prior" in txt) == consistent
        assert "not a diagnosis" in txt


def test_weights_and_wilson_and_progression_label():
    from ckv2.posthoc import balanced_weights, group_weights, progression_frame, wilson
    y = np.r_[np.ones(10), np.zeros(90)]
    w = balanced_weights(y)
    assert abs(w.mean() - 1) < 1e-12 and abs(w[y == 1].sum() - w[y == 0].sum()) < 1e-9
    fr = pd.DataFrame({"sex": ["f"] * 30 + ["m"] * 70, "age_band": ["20-39"] * 50 + ["60+"] * 50})
    g = group_weights(fr)
    assert abs(g.mean() - 1) < 1e-12
    lo, hi = wilson(30, 100)
    assert lo < 0.3 < hi
    ev = pd.DataFrame({"fpg_mmol": [5.0, 5.5, 5.7, 5.2], "fpg_final": [5.7, 5.0, 7.5, 5.0], "label": [0.0, 1.0, 1.0, 0.0]})
    pf = progression_frame(ev)
    assert len(pf) == 3 and pf.y.tolist() == [1, 1, 0]


In [ ]:
# CELL 15 | unit tests of the library (toy arrays only): metrics against scikit-learn, lifelines and scikit-survival where installed,
# monotone baselines, repeated folds, penalty, CKINN(lambda = 0) = M2
import importlib
import pytest
for _m in [m for m in list(sys.modules) if m == "ckv2" or m.startswith("ckv2.")]:
    del sys.modules[_m]
_t0 = time.perf_counter()
_rc = pytest.main(["-q", "-p", "no:cacheprovider", "-W", "ignore", str(LIB / "tests" / "test_ckv2.py")])
assert _rc == 0, "unit tests failed"
from ckv2 import config as C, data as D, metrics as MT, engine as EN, plan as PL, analysis as AN, posthoc as PH
from ckv2.missing import scen_tag, scenarios
print(f"CELL 15 OK | unit tests passed | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 16 | frozen configuration and protocol hash (protocol section 9); full mode stops if the configuration differs
CFG = C.make_config(MODE)
PROTOCOL_HASH = C.config_hash(CFG)
EXPECTED_FULL_HASH = "7cdc5bdde12b17df19a15650f00f1fe693168b33f39b193219ded6d70ea0468b"
(OUT / "report" / "frozen_protocol_v2.json").write_text(json.dumps({"config": json.loads(CFG.to_json()), "sha256": PROTOCOL_HASH}, indent=1))
print("configuration sha256:", PROTOCOL_HASH)
if MODE == "full":
    assert PROTOCOL_HASH == EXPECTED_FULL_HASH, "the full-mode configuration differs from the frozen protocol: record a deviation first"
print("CELL 16 OK | configuration frozen")

In [ ]:
# CELL 17 | data: locate and hash the three files, harmonise, audit, task frames, development / evaluation split, feature sets
_t0 = time.perf_counter()
_roots = [DATA_DIR, Path("."), Path("/content"), Path("/mnt/user-data/uploads")]
FILES = {"NAGALA": D.find_file(_roots, ["*DatasetIntJObesity*.xlsx"]), "RICH": D.find_file(_roots, ["*RCHealthCareData*.xlsx"]),
         "NHANES": D.find_file(_roots, ["*nhanes_t2d_extract*.csv"])}
FILE_HASH = {k: D.sha256_file(v) for k, v in FILES.items()}
EXPECTED_SHA = {"NAGALA": "671359aea2019c848f969395f88e57f888eaba8463ea0351bcda8f1c947f5523",
                "RICH": "ffd341a1a91ac65585b9fba70847a2681f5671a6035d9cfbf92e50795d74fbf6",
                "NHANES": "d39795db4a687cd5c37e7eb1e85ac8850f78192f031d0dcf3887e413f976899f"}
for k in FILES:
    assert FILE_HASH[k] == EXPECTED_SHA[k], f"{k}: file differs from the one audited (sha256 {FILE_HASH[k][:16]})"
    print(f"{k:7s} {FILES[k].name} sha256 {FILE_HASH[k][:16]}... verified")
RAW = {"NAGALA": D.load_nagala(D.read_cached(FILES["NAGALA"], CACHE)), "RICH": D.load_rich(D.read_cached(FILES["RICH"], CACHE)),
       "NHANES": D.load_nhanes(D.read_cached(FILES["NHANES"], CACHE))}
_audit = []
for k, d in RAW.items():
    oor = {c: float(((d[c] < lo) | (d[c] > hi)).mean()) for c, (lo, hi) in C.RANGES.items() if d[c].notna().any()}
    _audit.append({"cohort": k, "rows": len(d), "labelled": int(d.label.notna().sum()), "positives": int(np.nansum(d.label)),
                   "prevalence_pct": 100 * float(np.nanmean(d.label)), "predictors_present": int(sum(d[c].notna().any() for c in C.ALL_FEATURES)),
                   "max_pct_out_of_range": 100 * max(oor.values()), "worst_variable": max(oor, key=oor.get),
                   "follow_up_min_years": float(d.follow_years.min()), "follow_up_median_years": float(d.follow_years.median()),
                   "follow_up_max_years": float(d.follow_years.max())})
DATA_CARD = pd.DataFrame(_audit)
MISS = pd.DataFrame({k: d[C.ALL_FEATURES].isna().mean() for k, d in RAW.items()}).reset_index().rename(columns={"index": "variable"})
_nh = RAW["NHANES"]
NH_LABEL = pd.DataFrame([{"definition": "v1 (no reported diagnosis)", "labelled": int(_nh.label_v1.notna().sum()), "positives": int(np.nansum(_nh.label_v1))},
                         {"definition": "v2 (A11: also no medication, answered, fasting >= 8 h for FPG)", "labelled": int(_nh.label.notna().sum()),
                          "positives": int(np.nansum(_nh.label))}])
TASK = {k: D.task_frame(d) for k, d in RAW.items()}
TASK["NHANESv1"] = D.task_frame(RAW["NHANES"], "label_v1")
DEV, EVAL, DEV_ROWS = {}, {}, {}
for k, t in TASK.items():
    key = "NHANES" if k == "NHANESv1" else k                      # v1 split seed
    DEV[k], EVAL[k] = D.dev_eval_split(t, key, CFG.dev_frac)
    DEV_ROWS[k] = set(DEV[k].raw_row)
    if k == "NHANESv1":                                           # sensitivity frame: drop rows used to tune the NHANES models,
        EVAL[k] = t[~t.raw_row.isin(DEV_ROWS["NHANES"])].reset_index(drop=True)
        DEV[k], DEV_ROWS[k] = DEV["NHANES"], DEV_ROWS["NHANES"]   # whose tuned configurations it reuses
    DEV[k] = D.subsample(DEV[k], CFG.max_rows.get(key, 0) // 4, "dev" + k)
    EVAL[k] = D.subsample(EVAL[k], CFG.max_rows.get(key, 0), "eval" + k)
    DEV[k] = D.cap_rows(DEV[k], CFG.dev_max_rows, key)
SPLIT = pd.DataFrame([{"cohort": k, "dev_rows": len(DEV[k]), "dev_events": int(DEV[k].y.sum()), "eval_rows": len(EVAL[k]),
                       "eval_events": int(EVAL[k].y.sum())} for k in TASK])
assert not (set(EVAL["NHANESv1"].raw_row) & DEV_ROWS["NHANES"]), "NHANES tuning rows must not be evaluated in the v1-label analysis"
for k in ("NAGALA", "RICH", "NHANES"):
    assert not (set(EVAL[k].raw_row) & DEV_ROWS[k]), f"{k}: development rows in the evaluation subset"
FEATS = {(k, "T2"): D.features_for(TASK[k], "T2") for k in ("NAGALA", "RICH", "NHANES")}
FEATS.update({(k, "T3"): D.features_for(TASK[k], "T3") for k in ("NAGALA", "RICH")})
FEATS[("NHANESv1", "T2")] = FEATS[("NHANES", "T2")]
for (k, tier), f in FEATS.items():
    D.assert_no_leakage(f, TASK[k].task.iloc[0], tier)
for _bad, _task, _tier in ((["fpg_mmol"], "A", "T2"), (["hba1c_pct"], "B", "T2"), (["age_years", "age_years"], "B", "T2")):
    try:
        D.assert_no_leakage(_bad, _task, _tier)
        raise RuntimeError("leakage guard did not fire")
    except AssertionError:
        pass
SHARED = PL.shared_features(TASK["NAGALA"], TASK["RICH"], C.LOW_COST + C.PANEL)
POOLED_FEATS = list(dict.fromkeys(FEATS[("NAGALA", "T2")] + FEATS[("RICH", "T2")]))
FEAT_TBL = pd.DataFrame([{"cohort": k, "tier": t, "n_features": len(f), "features": ", ".join(f)} for (k, t), f in FEATS.items()]
                        + [{"cohort": "NAGALA+RICH", "tier": "shared (transport)", "n_features": len(SHARED), "features": ", ".join(SHARED)}])
FR = {k: EVAL[k] for k in EVAL}
FR.update({f"{k}_dev": DEV[k] for k in DEV})
FR["POOL"] = pd.concat([EVAL["NAGALA"], EVAL["RICH"]], ignore_index=True)
EN.save_frames(CACHE, FR)
FOLDS = {k: D.repeated_folds(EVAL[k].y.to_numpy(), "NHANES" if k == "NHANESv1" else k, CFG.n_splits, CFG.n_repeats) for k in EVAL}
for name, df in (("table01_data_card", DATA_CARD), ("table02_missingness", MISS), ("table03_nhanes_label", NH_LABEL),
                 ("table04_split", SPLIT), ("table05_feature_sets", FEAT_TBL)):
    df.to_csv(OUT / "tables" / f"{name}.csv", index=False)
with pd.option_context("display.width", 200, "display.max_columns", 30):
    print(DATA_CARD.round(3).to_string(index=False)); print(NH_LABEL.to_string(index=False)); print(SPLIT.to_string(index=False))
    print(FEAT_TBL.to_string(index=False))
print(f"CELL 17 OK | data loaded, audited and split | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 18 | tuning on development rows only (protocol section 5): 32 configurations for M1 and M2; CKINN = M2's best of its first 24
# plus 8 lambda values scored with lambda 0 on repeated 3-fold CV; competitors on their own grids
_t0 = time.perf_counter()
HP, LAM, TUNE_ROWS, LAM_ROWS = {}, {}, [], []
_specs = {c: PL.tune_specs(c, DEV[c], FEATS[(c, "T2")], CFG) for c in ("NAGALA", "RICH", "NHANES")}
EN.run_tasks([s for v in _specs.values() for s in v], CFG, CACHE, N_WORKERS, "tuning")
for c, sp in _specs.items():
    sc = PL.score_specs(sp, DEV[c], CACHE)
    g = sc.groupby(["model", "ci"]).score.mean().reset_index()
    for m, gm in g.groupby("model"):
        best = gm.loc[gm.score.idxmin()]
        HP[(c, m)] = PL.config_lists(c, CFG)[m][int(best.ci)]
        TUNE_ROWS += [{"cohort": c, "model": m, "config": int(r.ci), "dev_score": r.score, "chosen": int(r.ci) == int(best.ci),
                       "hp": json.dumps(PL.config_lists(c, CFG)[m][int(r.ci)])} for r in gm.itertuples()]
    gm2 = g[(g.model == "M2") & (g.ci < CFG.tune_ck_base)]
    HP[(c, "CKINN")] = PL.config_lists(c, CFG)["M2"][int(gm2.loc[gm2.score.idxmin()].ci)]
_lspecs = {(c, t): PL.lam_specs(c, t, DEV[c], FEATS[(c, t)], HP[(c, "CKINN")], CFG)
           for c, t in (("NAGALA", "T2"), ("RICH", "T2"), ("NHANES", "T2"), ("NAGALA", "T3"), ("RICH", "T3"))}
EN.run_tasks([s for v in _lspecs.values() for s in v], CFG, CACHE, N_WORKERS, "lambda")
for (c, t), sp in _lspecs.items():
    LAM[(c, t)], tab = PL.select_lambda(PL.score_specs(sp, DEV[c], CACHE), CFG.lam_grid)
    LAM_ROWS += [dict(cohort=c, tier=t, **r) for r in tab.to_dict("records")]
for m in list(C.MAIN_MODELS):
    HP[("NHANESv1", m)] = HP[("NHANES", m)]
LAM[("NHANESv1", "T2")] = LAM[("NHANES", "T2")]
TUNING, LAMBDAS = pd.DataFrame(TUNE_ROWS), pd.DataFrame(LAM_ROWS)
TUNING.to_csv(OUT / "tables" / "table06_tuning.csv", index=False)
LAMBDAS.to_csv(OUT / "tables" / "table07_lambda.csv", index=False)
print(TUNING[TUNING.chosen].drop(columns=["chosen"]).round(4).to_string(index=False))
print(LAMBDAS.round(5).to_string(index=False))
assert set(LAMBDAS.groupby(["cohort", "tier"]).chosen.sum()) == {1}
HP_SHARED = {c: HP[(c, "CKINN")] == HP[(c, "M2")] for c in ("NAGALA", "RICH", "NHANES")}
pd.DataFrame([{"cohort": c, "ckinn_uses_m2_configuration": v} for c, v in HP_SHARED.items()]).to_csv(OUT / "tables" / "table07b_ckinn_m2_configuration.csv", index=False)
print("CKINN uses the same network configuration as M2 (so family A isolates the penalty):", HP_SHARED)
print(f"CELL 18 OK | tuning done on development rows only | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 19 | main runs: 5 repeats x 5 folds, every model, network seed r in repeat r; test-time missingness scenarios;
# repeat 0 also stores attributions, violation statistics, sparse-region probes and partial dependence
_t0 = time.perf_counter()
CV_SPECS = {c: PL.cv_specs(c, "T2", FOLDS[c], FEATS[(c, "T2")], HP, LAM, CFG) for c in ("NAGALA", "RICH", "NHANES")}
SEED_SPECS = {c: PL.seed_specs(c, "T2", FOLDS[c], FEATS[(c, "T2")], HP, LAM, CFG) for c in ("NAGALA", "RICH", "NHANES")}
EN.run_tasks([s for c in CV_SPECS for s in CV_SPECS[c] + SEED_SPECS[c]], CFG, CACHE, N_WORKERS, "main runs and seed study")
print(f"CELL 19 OK | main runs and seed study fitted | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 20 | repeat 0 analyses: constraint variants (leave one out, wrong signs, alternative tables, lambda path), tier T3,
# RICH without family history, NHANES with the v1 label, 3-year IPCW models
_t0 = time.perf_counter()
VAR_SPECS = {c: PL.variant_specs(c, FOLDS[c], FEATS[(c, "T2")], HP, LAM, CFG) for c in ("NAGALA", "RICH", "NHANES")}
_t3m = ("LR", "XGB", "XGB_mono", "M1", "M2", "CKINN")
T3_SPECS = {c: PL.cv_specs(c, "T3", FOLDS[c], FEATS[(c, "T3")], HP, LAM, CFG, models=_t3m, repeats=[0], scen=False, extras=False)
            for c in ("NAGALA", "RICH")}
_nofh = [f for f in FEATS[("RICH", "T2")] if f != "family_history_dm"]
NOFH_SPECS = PL.cv_specs("RICH", "T2", FOLDS["RICH"], _nofh, HP, LAM, CFG, models=("LR", "M2", "CKINN"), repeats=[0], variant="noFH",
                         scen=False, extras=False)
NHV1_SPECS = PL.cv_specs("NHANESv1", "T2", FOLDS["NHANESv1"], FEATS[("NHANESv1", "T2")], HP, LAM, CFG, models=("LR", "M2", "CKINN"),
                         repeats=[0], variant="nodevoverlap", scen=False, extras=False)
IPCW_SPECS = {c: PL.ipcw_specs(c, FOLDS[c], FEATS[(c, "T2")], HP, LAM, CFG, EVAL[c]) for c in ("NAGALA", "RICH")}
_all = [s for v in VAR_SPECS.values() for s in v] + [s for v in T3_SPECS.values() for s in v] + NOFH_SPECS + NHV1_SPECS + \
       [s for v in IPCW_SPECS.values() for s in v]
EN.run_tasks(_all, CFG, CACHE, N_WORKERS, "repeat-0 analyses")
print(f"CELL 20 OK | repeat-0 analyses fitted | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 21 | transport (H3), reweighting, pooled training and RICH leave-one-site-out
_t0 = time.perf_counter()
TR_SPECS = PL.transport_specs(EVAL, FOLDS, SHARED, HP, LAM, CFG)
POOL_SPECS = PL.pooled_specs(EVAL, FOLDS, POOLED_FEATS, HP, LAM, CFG)
SITE_SPECS, SITES_HELD = PL.site_specs(EVAL["RICH"], FEATS[("RICH", "T2")], HP, LAM, CFG)
EN.run_tasks(TR_SPECS + POOL_SPECS + SITE_SPECS, CFG, CACHE, N_WORKERS, "transport, pooled, sites")
print(f"held-out RICH sites: {SITES_HELD}")
print(f"CELL 21 OK | transport, pooled and site analyses fitted | {time.perf_counter() - _t0:.0f}s")

## Analysis of the stored predictions
Everything below reads the checkpointed predictions; no model is fitted. Each table states its label (P1, P2, E).

In [ ]:
# CELL 23 | assemble out-of-fold predictions of every stage and save them locally (row-level: never archived)
_t0 = time.perf_counter()
COHORTS = ("NAGALA", "RICH", "NHANES")
NE = {c: len(EVAL[c]) for c in EVAL}
Y = {c: EVAL[c].y.to_numpy().astype(float) for c in EVAL}
MAINRUNS = [(r, r) for r in range(CFG.n_repeats)]
SEEDRUNS = [(0, s) for s in range(CFG.n_seeds)]
NN = list(C.NN_MODELS)
P, THR, FOLDID = {}, {}, {}
for c in COHORTS:
    P[c], THR[c], FOLDID[c] = AN.collect(CV_SPECS[c] + SEED_SPECS[c], NE[c], CACHE)
PV = {c: AN.collect(VAR_SPECS[c], NE[c], CACHE, key=lambda s: s["_variant"])[0] for c in VAR_SPECS}
PT3 = {c: AN.collect(T3_SPECS[c], NE[c], CACHE)[0] for c in T3_SPECS}
PNOFH = AN.collect(NOFH_SPECS, NE["RICH"], CACHE)[0]
PNHV1 = AN.collect(NHV1_SPECS, NE["NHANESv1"], CACHE)[0]
PIPCW = {c: AN.collect(IPCW_SPECS[c], NE[c], CACHE, key=lambda s: s["_model"])[0] for c in IPCW_SPECS}
_ext = [s for s in TR_SPECS if s["id"].startswith("trext")]
_ref = [s for s in TR_SPECS if s["id"].startswith("trref")]
_rw = [s for s in TR_SPECS if s["id"].startswith("trrw")]
TRX = {(s["_src"], s["_tgt"], s["_model"], s["_s"]): EN.load(CACHE, s["id"])["preds"]["tgt"].astype(float) for s in _ext}
TRW = {(s["_src"], s["_tgt"], s["_model"]): EN.load(CACHE, s["id"])["preds"]["tgt"].astype(float) for s in _rw}
TRREF = {}
for _tgt in ("RICH", "NAGALA"):
    _p = AN.collect([s for s in _ref if s["_tgt"] == _tgt], NE[_tgt], CACHE, key=lambda s: (s["_model"], s["_s"]))[0]
    TRREF.update({(_tgt, m, s): v["none"].astype(float) for (m, s), v in _p.items()})
POOLP = {}
for s in POOL_SPECS:
    r_ = EN.load(CACHE, s["id"])
    for t in s["tests"]:
        off = 0 if t["tag"] == "NAGALA" else NE["NAGALA"]
        POOLP.setdefault((s["_model"], t["tag"]), np.full(NE[t["tag"]], np.nan))[t["idx"] - off] = r_["preds"][t["tag"]]
assert all(np.isfinite(v).all() for v in POOLP.values())
SITEP = {}
for s in SITE_SPECS:
    SITEP.setdefault(s["_model"], np.full(NE["RICH"], np.nan))[s["tests"][0]["idx"]] = EN.load(CACHE, s["id"])["preds"]["none"]
SITE_ROWS = np.isfinite(SITEP["CKINN"])
for c in COHORTS:
    np.savez_compressed(CACHE / "preds" / f"{c}_runs.npz", y=Y[c],
                        **{f"{m}|r{r}|s{s}|{tag}": v for (m, r, s), d in P[c].items() for tag, v in d.items()},
                        **{f"THRSPEC|{m}|r{r}|s{s}": THR[c][(m, r, s)][0] for (m, r, s) in THR[c]},
                        **{f"FOLD|{m}|r{r}|s{s}": FOLDID[c][(m, r, s)] for (m, r, s) in FOLDID[c]})
np.savez_compressed(CACHE / "preds" / "transport.npz", **{f"ext|{a}|{b}|{m}|s{s}": v for (a, b, m, s), v in TRX.items()},
                    **{f"ref|{a}|{m}|s{s}": v for (a, m, s), v in TRREF.items()}, y_NAGALA=Y["NAGALA"], y_RICH=Y["RICH"])
print(f"CELL 23 OK | predictions assembled for {sum(len(P[c]) for c in COHORTS)} cohort-model-run keys | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 24 | performance of every model (P2): run-averaged AUROC, AUPRC, Brier, scaled Brier, calibration slope,
# calibration-in-the-large and ECE, with 95% intervals from the paired bootstrap (descriptive draws)
_t0 = time.perf_counter()
_pl = {}
for c in COHORTS:
    vecs = {m: [P[c][(m, r, s)]["none"] for r, s in MAINRUNS] for m in C.MAIN_MODELS}
    vecs.update({m: [P[c][(m, 0, 0)]["none"]] for m in C.RUN0_EXTRA})
    vecs.update({f"{m}_r0": [P[c][(m, 0, 0)]["none"]] for m in NN})
    _pl[c] = {"y": Y[c], "vecs": vecs, "stats": ["auroc", "brier", "slope", "citl", "ece"]}
BOOT_DESC = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
rows = []
for c in COHORTS:
    b, y = BOOT_DESC[c], Y[c]
    prev = y.mean()
    for m in list(C.MAIN_MODELS) + list(C.RUN0_EXTRA):
        runs = MAINRUNS if m in C.MAIN_MODELS else [(0, 0)]
        row = {"cohort": c, "model": m, "runs": len(runs), "n": len(y), "events": int(y.sum()),
               "auprc": float(np.mean([AN.auprc(y, P[c][(m, r, s)]["none"]) for r, s in runs]))}
        for st in ("auroc", "brier", "slope", "citl", "ece"):
            row[st] = b["point"][(m, st)]
            row[st + "_lo"], row[st + "_hi"] = AN.ci(b["draws"][(m, st)])
        row["scaled_brier"] = 1 - row["brier"] / (prev * (1 - prev))
        rows.append(row)
PERF = pd.DataFrame(rows)
PERF.to_csv(OUT / "tables" / "table08_performance.csv", index=False)
with pd.option_context("display.width", 250, "display.max_columns", 30):
    print(PERF[["cohort", "model", "runs", "events", "auroc", "auroc_lo", "auroc_hi", "auprc", "brier", "scaled_brier", "slope", "citl", "ece"]].round(4).to_string(index=False))
print(f"CELL 24 OK | performance table | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 25 | hypothesis families A (CKINN minus M2, primary) and B (CKINN minus M1): H1 to H5 with Holm correction (P2)
_t0 = time.perf_counter()
GROUPS, N_BANDS = {}, {}
_pl = {}
for c in COHORTS:
    GROUPS[c], N_BANDS[c] = AN.groups_for(EVAL[c], Y[c], CFG.min_group_events)
    _pl[c] = {"y": Y[c], "vecs": {m: [P[c][(m, r, s)]["none"] for r, s in MAINRUNS] for m in NN},
              "stats": ["auroc", "brier", "slope_err", "tpr_sex_gap", "tpr_age_range"], "groups": GROUPS[c], "fpr": CFG.fpr_matched}
for src, tgt in (("NAGALA", "RICH"), ("RICH", "NAGALA")):
    v = {}
    for m in NN:
        v[f"ext_{m}"] = [TRX[(src, tgt, m, s)] for s in range(CFG.n_seeds)]
        v[f"ref_{m}"] = [TRREF[(tgt, m, s)] for s in range(CFG.n_seeds)]
    for m in ("LR", "XGB", "XGB_mono"):
        v[f"ext_{m}"], v[f"ref_{m}"] = [TRX[(src, tgt, m, 0)]], [TRREF[(tgt, m, 0)]]
    _pl[f"{src}->{tgt}"] = {"y": Y[tgt], "vecs": v, "stats": ["auroc", "citl_abs", "brier", "slope"]}
BOOT_FAM = AN.bootstrap(_pl, CFG.n_boot, N_WORKERS)
# H4: across-seed attribution stability on the explained rows of repeat 0
STAB, STAB_DRAWS = {}, {}
for c in COHORTS:
    rows_m = {m: [] for m in NN}
    for k in range(CFG.n_splits):
        for m in NN:
            A = [EN.load(CACHE, PL.tid("cv", c, "T2", "main" if s == 0 else "seed", "r0", f"k{k}", m, f"s{s}"))["attr"] for s in range(CFG.n_seeds)]
            rows_m[m].append(AN.stability_rows(A))
    STAB[c] = {m: np.concatenate(v) for m, v in rows_m.items()}
    STAB_DRAWS[c] = AN.mean_boot(STAB[c], CFG.n_boot, C.seed_for("h4", c))
EP = []
for fam, F in (("A: CKINN-M2", "M2"), ("B: CKINN-M1", "M1")):
    for c in COHORTS:
        u, pt, dr = f"{c} (T2)", BOOT_FAM[c]["point"], BOOT_FAM[c]["draws"]
        d = lambda st: (pt[("CKINN", st)] - pt[(F, st)], dr[("CKINN", st)] - dr[(F, st)])
        EP.append(AN.endpoint_row(fam, "H1", u, "AUROC (non-inferiority, margin 0.01)", "higher", *d("auroc"), margin=CFG.ni_margin))
        EP.append(AN.endpoint_row(fam, "H1", u, "AUROC (superiority, descriptive)", "higher", *d("auroc"), primary=False))
        EP.append(AN.endpoint_row(fam, "H2", u, "Brier score", "lower", *d("brier")))
        EP.append(AN.endpoint_row(fam, "H2", u, "|calibration slope - 1|", "lower", *d("slope_err")))
        s_ck, s_f = STAB[c]["CKINN"], STAB[c][F]
        EP.append(AN.endpoint_row(fam, "H4", u, "across-seed attribution agreement (Spearman)", "higher",
                                  float(np.nanmean(s_ck) - np.nanmean(s_f)), STAB_DRAWS[c]["CKINN"] - STAB_DRAWS[c][F],
                                  note=f"{len(s_ck)} explained rows, {CFG.n_seeds} seeds"))
        if len(GROUPS[c]["sex"]) == 2:
            EP.append(AN.endpoint_row(fam, "H5", u, "sex gap in sensitivity at FPR 0.10", "lower", *d("tpr_sex_gap")))
        if N_BANDS[c] >= 2:
            EP.append(AN.endpoint_row(fam, "H5", u, "age-band range in sensitivity at FPR 0.10", "lower", *d("tpr_age_range"),
                                      note=f"{N_BANDS[c]} age bands with >= {CFG.min_group_events} events"))
    for src, tgt in (("NAGALA", "RICH"), ("RICH", "NAGALA")):
        u = f"{src} -> {tgt}"
        pt, dr = BOOT_FAM[u.replace(" ", "")]["point"], BOOT_FAM[u.replace(" ", "")]["draws"]
        gap = lambda m, D_: D_[(f"ref_{m}", "auroc")] - D_[(f"ext_{m}", "auroc")]
        EP.append(AN.endpoint_row(fam, "H3", u, "transport gap (target CV AUROC - external AUROC)", "lower",
                                  gap("CKINN", pt) - gap(F, pt), gap("CKINN", dr) - gap(F, dr)))
        EP.append(AN.endpoint_row(fam, "H3", u, "|calibration-in-the-large| in the target", "lower",
                                  pt[("ext_CKINN", "citl_abs")] - pt[(f"ext_{F}", "citl_abs")], dr[("ext_CKINN", "citl_abs")] - dr[(f"ext_{F}", "citl_abs")]))
EP, VERDICTS = AN.verdicts(pd.DataFrame(EP), CFG.alpha)
EP.to_csv(OUT / "tables" / "table09_hypothesis_endpoints.csv", index=False)
VERDICTS.to_csv(OUT / "tables" / "table10_verdicts.csv", index=False)
with pd.option_context("display.width", 250, "display.max_columns", 30, "display.max_rows", 200):
    print(EP[["family", "hypothesis", "unit", "endpoint", "difference", "ci_lo", "ci_hi", "p_pred_holm", "p_opp_holm", "status"]].round(5).to_string(index=False))
    print(VERDICTS.to_string(index=False))
assert EP[EP.primary].groupby("family").size().nunique() == 1, "families must have the same endpoints"
print(f"CELL 25 OK | verdicts: " + "; ".join(f"{r.family[:1]} {r.hypothesis} {r.verdict}" for r in VERDICTS.itertuples()) + f" | {time.perf_counter() - _t0:.0f}s")

In [ ]:
from scipy.stats import norm
# CELL 26 | cross-checks of H1 (P2): DeLong on run 0, corrected repeated-CV t-test over 25 folds, TOST equivalence, RICH cluster
# bootstrap by site, minimum detectable effect for NAGALA; variance components (sampling, seed, split)
_t0 = time.perf_counter()
rows = []
_clu = {"RICH": {**{k: v for k, v in _pl["RICH"].items() if k != "groups"}, "stats": ["auroc", "brier", "slope_err"], "mode": "cluster",
                 "cluster": EVAL["RICH"].site.to_numpy()}}
BOOT_CLU = AN.bootstrap(_clu, CFG.n_boot, N_WORKERS)
for fam, F in (("A: CKINN-M2", "M2"), ("B: CKINN-M1", "M1")):
    for c in COHORTS:
        y = Y[c]
        dl = MT.delong_paired(y, P[c][("CKINN", 0, 0)]["none"], P[c][(F, 0, 0)]["none"])
        diffs = []
        for r, s in MAINRUNS:
            fd = FOLDID[c][("CKINN", r, s)]
            diffs.append(AN.fold_aurocs(y, P[c][("CKINN", r, s)]["none"], fd) - AN.fold_aurocs(y, P[c][(F, r, s)]["none"], fd))
        nb = MT.corrected_t(np.concatenate(diffs), 1.0, CFG.n_splits - 1.0)
        dr = BOOT_FAM[c]["draws"][("CKINN", "auroc")] - BOOT_FAM[c]["draws"][(F, "auroc")]
        p_tost = max(MT.p_one_sided(dr, -CFG.ni_margin, +1), MT.p_one_sided(dr, CFG.ni_margin, -1))
        row = {"family": fam, "cohort": c, "auroc_diff_runavg": BOOT_FAM[c]["point"][("CKINN", "auroc")] - BOOT_FAM[c]["point"][(F, "auroc")],
               "boot_lo": AN.ci(dr)[0], "boot_hi": AN.ci(dr)[1], "boot_sd": float(np.std(dr, ddof=1)),
               "p_superiority_two_sided": min(1.0, 2 * min(MT.p_one_sided(dr, 0, +1), MT.p_one_sided(dr, 0, -1))),
               "p_tost_margin_0.01": p_tost, "equivalent_within_0.01": p_tost < CFG.alpha,
               "delong_diff_run0": dl["diff"], "delong_se": dl["se"], "delong_p": dl["p_two_sided"],
               "nb_mean_fold_diff": nb["mean"], "nb_lo": nb["lo"], "nb_hi": nb["hi"], "nb_p": nb["p_two_sided"],
               "mde_80pct_power": (norm.ppf(0.975) + norm.ppf(0.80)) * float(np.std(dr, ddof=1)),
               "power_noninferiority_if_true_diff_0": float(norm.cdf(CFG.ni_margin / float(np.std(dr, ddof=1)) - norm.ppf(0.975)))}
        if c == "RICH":
            cd = BOOT_CLU["RICH"]["draws"]
            dcl = cd[("CKINN", "auroc")] - cd[(F, "auroc")]
            row.update({"cluster_lo": AN.ci(dcl)[0], "cluster_hi": AN.ci(dcl)[1],
                        "cluster_brier_lo": AN.ci(cd[("CKINN", "brier")] - cd[(F, "brier")])[0],
                        "cluster_brier_hi": AN.ci(cd[("CKINN", "brier")] - cd[(F, "brier")])[1]})
        rows.append(row)
XCHECK = pd.DataFrame(rows)
XCHECK.to_csv(OUT / "tables" / "table11_h1_crosschecks.csv", index=False)
# variance components: sampling SE (bootstrap of run 0 alone), seed SD (repeat 0, 5 seeds), between-run SD (5 main runs)
rows = []
for c in COHORTS:
    y = Y[c]
    for m in NN + ["LR", "XGB_mono"]:
        for st, fn in (("auroc", lambda p: MT.auroc_w(MT.Ranked(p, y))), ("brier", lambda p: MT.brier_w(y, p))):
            row = {"cohort": c, "model": m, "metric": st,
                   "sampling_se_run0": float(np.std(BOOT_DESC[c]["draws"][(f"{m}_r0" if m in NN else m, st)], ddof=1)) if m in NN else np.nan,
                   "sd_between_main_runs": float(np.std([fn(P[c][(m, r, s)]["none"]) for r, s in MAINRUNS], ddof=1))}
            row["sd_across_seeds_repeat0"] = float(np.std([fn(P[c][(m, 0, s)]["none"]) for _, s in SEEDRUNS], ddof=1)) if m in NN else np.nan
            rows.append(row)
    for F in ("M2", "M1"):
        for st, fn in (("auroc", lambda p: MT.auroc_w(MT.Ranked(p, y))), ("brier", lambda p: MT.brier_w(y, p))):
            rows.append({"cohort": c, "model": f"CKINN-{F}", "metric": st,
                         "sd_across_seeds_repeat0": float(np.std([fn(P[c][("CKINN", 0, s)]["none"]) - fn(P[c][(F, 0, s)]["none"]) for _, s in SEEDRUNS], ddof=1)),
                         "sd_between_main_runs": float(np.std([fn(P[c][("CKINN", r, s)]["none"]) - fn(P[c][(F, r, s)]["none"]) for r, s in MAINRUNS], ddof=1)),
                         "sampling_se_run0": float(np.std(BOOT_DESC[c]["draws"][("CKINN_r0", st)] - BOOT_DESC[c]["draws"][(f"{F}_r0", st)], ddof=1))})
VARCOMP = pd.DataFrame(rows)
VARCOMP.to_csv(OUT / "tables" / "table12_variance_components.csv", index=False)
with pd.option_context("display.width", 250, "display.max_columns", 30):
    print(XCHECK.round(5).to_string(index=False)); print(VARCOMP.round(5).to_string(index=False))
print(f"CELL 26 OK | cross-checks and variance components | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 27 | CKINN against every competitor (P2, descriptive, no multiplicity correction): AUROC and Brier differences
_t0 = time.perf_counter()
rows = []
for c in COHORTS:
    pt, dr = BOOT_DESC[c]["point"], BOOT_DESC[c]["draws"]
    for m in list(C.MAIN_MODELS) + list(C.RUN0_EXTRA):
        if m == "CKINN":
            continue
        ck = "CKINN" if m in C.MAIN_MODELS else "CKINN_r0"
        for st in ("auroc", "brier"):
            d = dr[(ck, st)] - dr[(m, st)]
            rows.append({"cohort": c, "comparator": m, "metric": st, "ckinn_minus_comparator": pt[(ck, st)] - pt[(m, st)],
                         "lo": AN.ci(d)[0], "hi": AN.ci(d)[1], "runs": CFG.n_repeats if m in C.MAIN_MODELS else 1})
SEC = pd.DataFrame(rows)
SEC.to_csv(OUT / "tables" / "table13_ckinn_vs_competitors.csv", index=False)
print(SEC[SEC.metric == "auroc"].round(4).to_string(index=False))
print(f"CELL 27 OK | competitor comparisons | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 28 | calibration (P2): reliability curves with 95% bootstrap bands (run 0, seed 0; equal-frequency tenths) for M1, M2, CKINN,
# LR and monotone XGBoost; slope, calibration-in-the-large and ECE intervals are in table 8
_t0 = time.perf_counter()
CAL_MODELS = ("M1", "M2", "CKINN", "LR", "XGB_mono")
rows = []
for c in COHORTS:
    y = Y[c]
    for m in CAL_MODELS:
        p = P[c][(m, 0, 0)]["none"].astype(float)
        edges = np.unique(np.quantile(p, np.linspace(0, 1, 11)))
        b = np.clip(np.searchsorted(edges, p, side="right") - 1, 0, len(edges) - 2)
        nb_ = len(edges) - 1
        obs = np.bincount(b, y, nb_) / np.bincount(b, minlength=nb_)
        pred = np.bincount(b, p, nb_) / np.bincount(b, minlength=nb_)
        bands = []
        for w in MT.boot_weight_iter(len(y), CFG.n_boot_band, C.seed_for("band", c, m)):
            bands.append(np.bincount(b, w * y, nb_) / np.maximum(np.bincount(b, w, nb_), 1e-12))
        lo, hi = np.quantile(np.vstack(bands), [0.025, 0.975], axis=0)
        rows += [{"cohort": c, "model": m, "bin": i, "mean_predicted": pred[i], "observed": obs[i], "obs_lo": lo[i], "obs_hi": hi[i],
                  "n": int((b == i).sum())} for i in range(nb_)]
CALCURVE = pd.DataFrame(rows)
CALCURVE.to_csv(OUT / "tables" / "table14_reliability_curves.csv", index=False)
print(f"CELL 28 OK | reliability curves | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 29 | clinical utility (P2): decision curves (net benefit, run-averaged) with intervals at the stated thresholds, and threshold
# metrics at fixed sensitivity 0.80 and fixed specificity 0.90 (thresholds from the inner validation split of each training fold)
_t0 = time.perf_counter()
UTIL_MODELS = ("M1", "M2", "CKINN", "LR", "cLR", "GAM", "XGB", "XGB_mono", "LGBM_mono")
rows, rows_pts, rows_thr = [], [], []
_pl = {}
for c in COHORTS:
    a, b_, st = CFG.dca_a if c == "NHANES" else CFG.dca_b
    pts = CFG.nb_points_a if c == "NHANES" else CFG.nb_points_b
    grid = np.round(np.arange(a, b_ + 1e-9, st), 4)
    y = Y[c]
    for t in grid:
        rows.append({"cohort": c, "model": "treat all", "threshold": t, "net_benefit": MT.net_benefit_all(y, t)})
        rows.append({"cohort": c, "model": "treat none", "threshold": t, "net_benefit": 0.0})
        for m in UTIL_MODELS:
            rows.append({"cohort": c, "model": m, "threshold": t,
                         "net_benefit": float(np.mean([MT.net_benefit(y, P[c][(m, r, s)]["none"], t) for r, s in MAINRUNS]))})
    _pl[c] = {"y": y, "vecs": {m: [P[c][(m, r, s)]["none"] for r, s in MAINRUNS] for m in UTIL_MODELS}, "stats": [f"nb@{t}" for t in pts]}
    for m in UTIL_MODELS:
        for which, tgt in (("fixed specificity 0.90", 0), ("fixed sensitivity 0.80", 1)):
            vals = [MT.threshold_metrics(y, P[c][(m, r, s)]["none"], THR[c][(m, r, s)][tgt]) for r, s in MAINRUNS]
            rows_thr.append({"cohort": c, "model": m, "operating_point": which, **{k: float(np.mean([v[k] for v in vals])) for k in vals[0]}})
BOOT_NB = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
for c in COHORTS:
    pts = CFG.nb_points_a if c == "NHANES" else CFG.nb_points_b
    for m in UTIL_MODELS:
        for t in pts:
            k = (m, f"nb@{t}")
            d = BOOT_NB[c]["draws"][k] - BOOT_NB[c]["draws"][("M2", f"nb@{t}")]
            rows_pts.append({"cohort": c, "model": m, "threshold": t, "net_benefit": BOOT_NB[c]["point"][k], "lo": AN.ci(BOOT_NB[c]["draws"][k])[0],
                             "hi": AN.ci(BOOT_NB[c]["draws"][k])[1], "treat_all": MT.net_benefit_all(Y[c], t),
                             "minus_M2": BOOT_NB[c]["point"][k] - BOOT_NB[c]["point"][("M2", f"nb@{t}")], "minus_M2_lo": AN.ci(d)[0], "minus_M2_hi": AN.ci(d)[1]})
DCA, NBPTS, THRMET = pd.DataFrame(rows), pd.DataFrame(rows_pts), pd.DataFrame(rows_thr)
DCA.to_csv(OUT / "tables" / "table15_decision_curves.csv", index=False)
NBPTS.to_csv(OUT / "tables" / "table16_net_benefit_points.csv", index=False)
THRMET.to_csv(OUT / "tables" / "table17_threshold_metrics.csv", index=False)
print(NBPTS.round(5).to_string(index=False)); print(THRMET.round(3).to_string(index=False))
print(f"CELL 29 OK | decision curves and threshold metrics | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 30 | censoring-aware evaluation in NAGALA and RICH (P2): Harrell's C, Uno's C at 3 years, IPCW cumulative/dynamic AUROC at
# 3 years (NAGALA also 5 years); and the 3-year IPCW-trained models (E): IPCW Brier, scaled Brier, observed and predicted 3-year risk
_t0 = time.perf_counter()
TTE_MODELS = list(C.MAIN_MODELS)
rows, rows_ip, _jobs, _pl = [], [], [], {}
for c in ("NAGALA", "RICH"):
    t_, e_ = EVAL[c].follow_years.to_numpy(), Y[c].astype(int)
    ut, G = MT.km_censoring(t_, e_)
    uw = np.where(e_ == 1, 1 / np.maximum(MT.G_at(ut, G, t_), 1e-12) ** 2, 0.0)
    for m in TTE_MODELS:
        for r, s in MAINRUNS:
            _jobs.append((c, m, r, "harrell", P[c][(m, r, s)]["none"].astype(float), t_, e_, None, None))
            _jobs.append((c, m, r, "uno3", P[c][(m, r, s)]["none"].astype(float), t_, e_, uw, CFG.horizon))
    for tau in ((CFG.horizon, CFG.nagala_horizon2) if c == "NAGALA" else (CFG.horizon,)):
        lab, iw = MT.td_auc_weights(t_, e_, tau, ut, G)
        keep = iw > 0
        _pl[f"{c}|{tau:g}"] = {"y": lab[keep], "base_w": iw[keep], "stats": ["auroc"],
                               "vecs": {m: [P[c][(m, r, s)]["none"][keep] for r, s in MAINRUNS] for m in TTE_MODELS}}
    for m, d in PIPCW[c].items():
        p3 = d["none"].astype(float)
        bs = MT.ipcw_brier(p3, t_, e_, CFG.horizon, ut, G)
        km3 = MT.km_risk(t_, e_, CFG.horizon)
        bs0 = MT.ipcw_brier(np.full(len(p3), km3), t_, e_, CFG.horizon, ut, G)
        rows_ip.append({"cohort": c, "model": m, "ipcw_brier_3y": bs, "scaled_brier_3y": 1 - bs / bs0, "td_auroc_3y": MT.td_auc(p3, t_, e_, CFG.horizon, ut, G),
                        "km_observed_risk_3y": km3, "mean_predicted_risk_3y": float(p3.mean())})

_conc_out = AN.parallel_map(AN.conc_job, _jobs, N_WORKERS)
BOOT_TTE = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
_cd = pd.DataFrame(_conc_out, columns=["cohort", "model", "run", "kind", "value"]).groupby(["cohort", "model", "kind"]).value.mean().unstack()
for c in ("NAGALA", "RICH"):
    for m in TTE_MODELS:
        row = {"cohort": c, "model": m, "harrell_c": _cd.loc[(c, m), "harrell"], "uno_c_3y": _cd.loc[(c, m), "uno3"]}
        for tau in ((CFG.horizon, CFG.nagala_horizon2) if c == "NAGALA" else (CFG.horizon,)):
            k = f"{c}|{tau:g}"
            row[f"td_auroc_{tau:g}y"] = BOOT_TTE[k]["point"][(m, "auroc")]
            row[f"td_auroc_{tau:g}y_lo"], row[f"td_auroc_{tau:g}y_hi"] = AN.ci(BOOT_TTE[k]["draws"][(m, "auroc")])
        rows.append(row)
TTE, IPCWM = pd.DataFrame(rows), pd.DataFrame(rows_ip)
TTE.to_csv(OUT / "tables" / "table18_censoring_aware.csv", index=False)
IPCWM.to_csv(OUT / "tables" / "table19_ipcw_3y_models.csv", index=False)
print(TTE.round(4).to_string(index=False)); print(IPCWM.round(4).to_string(index=False))
print(f"CELL 30 OK | censoring-aware metrics | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 31 | fairness (E, with H5 in cell 25): by sex and age band (and RICH site, NHANES ethnic group, descriptive): AUROC,
# sensitivity at FPR 0.10, true- and false-positive rates at the common threshold, calibration-in-the-large; gap intervals
_t0 = time.perf_counter()
FAIR_MODELS = ("M1", "M2", "CKINN", "LR", "XGB_mono")
rows, gap_rows, _pl = [], [], {}
for c in COHORTS:
    fr, y = EVAL[c], Y[c]
    gtypes = {"sex": fr.sex.to_numpy(), "age_band": fr.age_band.to_numpy()}
    if c == "RICH":
        gtypes["site (descriptive)"] = fr.site.to_numpy()
    if c == "NHANES":
        gtypes["ethnic group (descriptive)"] = fr.group_extra.to_numpy()
    for gt, gv in gtypes.items():
        est = [g for g in pd.unique(gv) if y[gv == g].sum() >= CFG.min_group_events and (y[gv == g] == 0).sum() >= CFG.min_group_events]
        for g in sorted(pd.unique(gv), key=str):
            ix = np.flatnonzero(gv == g)
            for m in FAIR_MODELS:
                ok = g in est
                vals = []
                for r, s in MAINRUNS:
                    p, thr = P[c][(m, r, s)]["none"][ix].astype(float), THR[c][(m, r, s)][0][ix]
                    h = p >= thr
                    vals.append({"auroc": MT.auroc_w(MT.Ranked(p, y[ix])) if ok else np.nan,
                                 "tpr_at_fpr_0.10": MT.tpr_at_fpr_w(MT.Ranked(p, y[ix]), CFG.fpr_matched) if ok else np.nan,
                                 "tpr_common_thr": (h & (y[ix] == 1)).sum() / max((y[ix] == 1).sum(), 1) if ok else np.nan,
                                 "fpr_common_thr": (h & (y[ix] == 0)).sum() / max((y[ix] == 0).sum(), 1),
                                 "citl": MT.citl_w(y[ix], p) if ok else np.nan, "slope": MT.calib_w(y[ix], p)[1] if ok else np.nan,
                                 "mean_pred": p.mean()})
                rows.append({"cohort": c, "group_type": gt, "group": str(g), "model": m, "n": len(ix), "events": int(y[ix].sum()),
                             "estimable": ok, **{k: float(np.mean([v[k] for v in vals])) for k in vals[0]}, "observed_rate": float(y[ix].mean())})
        if "descriptive" not in gt:
            key = f"{c}|{gt}"
            _pl[key] = {"y": y, "vecs": {m: [P[c][(m, r, s)]["none"] for r, s in MAINRUNS] for m in FAIR_MODELS},
                        "thrs": {m: [THR[c][(m, r, s)][0] for r, s in MAINRUNS] for m in FAIR_MODELS},
                        "groups": {"fair": [np.flatnonzero(gv == g) for g in sorted(est, key=str)]}, "fpr": CFG.fpr_matched,
                        "stats": ["gap_auroc", "gap_tpr", "gap_tprthr", "gap_fprthr", "gap_citl"]}
BOOT_FAIR = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
for key, b in BOOT_FAIR.items():
    c, gt = key.split("|")
    for m in FAIR_MODELS:
        for st in ("gap_auroc", "gap_tpr", "gap_tprthr", "gap_fprthr", "gap_citl"):
            d = b["draws"][(m, st)] - b["draws"][("M2", st)]
            gap_rows.append({"cohort": c, "group_type": gt, "model": m, "gap": st[4:], "value": b["point"][(m, st)], "lo": AN.ci(b["draws"][(m, st)])[0],
                             "hi": AN.ci(b["draws"][(m, st)])[1], "minus_M2": b["point"][(m, st)] - b["point"][("M2", st)],
                             "minus_M2_lo": AN.ci(d)[0], "minus_M2_hi": AN.ci(d)[1]})
FAIR, FAIRGAP = pd.DataFrame(rows), pd.DataFrame(gap_rows)
FAIR.to_csv(OUT / "tables" / "table20_subgroups.csv", index=False)
FAIRGAP.to_csv(OUT / "tables" / "table21_subgroup_gaps.csv", index=False)
print(FAIRGAP[FAIRGAP.model.isin(["M2", "CKINN"])].round(4).to_string(index=False))
print(f"CELL 31 OK | subgroup tables | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 32 | missingness (P2): test-time removal of the laboratory panel (MCAR, MAR, MNAR at 10, 30, 50%), AUROC loss at MCAR 30% for
# CKINN against each comparator; RICH structural missingness patterns; RICH leave-one-site-out
_t0 = time.perf_counter()
SCEN = [scen_tag(m, q) for m, q in scenarios(CFG)]
PRIM = scen_tag(*CFG.primary_miss)
rows, _pl = [], {}
for c in COHORTS:
    y = Y[c]
    for m in list(C.MAIN_MODELS) + list(C.RUN0_EXTRA):
        runs = MAINRUNS if m in C.MAIN_MODELS else [(0, 0)]
        row = {"cohort": c, "model": m, "runs": len(runs)}
        for tag in ["none"] + SCEN:
            row[tag] = float(np.mean([MT.auroc_w(MT.Ranked(P[c][(m, r, s)][tag], y)) for r, s in runs]))
        rows.append(row)
    v = {m: [P[c][(m, r, s)]["none"] for r, s in MAINRUNS] for m in C.MAIN_MODELS}
    v.update({f"{m}@{PRIM}": [P[c][(m, r, s)][PRIM] for r, s in MAINRUNS] for m in C.MAIN_MODELS})
    for m in list(C.RUN0_EXTRA) + ["CKINN"]:
        v[f"{m}_r0"], v[f"{m}_r0@{PRIM}"] = [P[c][(m, 0, 0)]["none"]], [P[c][(m, 0, 0)][PRIM]]
    _pl[c] = {"y": y, "vecs": v, "stats": ["auroc"]}
MISS_AUC = pd.DataFrame(rows)
BOOT_MISS = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
rows = []
for c in COHORTS:
    pt, dr = BOOT_MISS[c]["point"], BOOT_MISS[c]["draws"]
    for m in [x for x in C.MAIN_MODELS if x != "CKINN"] + [f"{x}_r0" for x in C.RUN0_EXTRA]:
        ck = "CKINN_r0" if m.endswith("_r0") else "CKINN"
        lk = lambda name, D_: D_[(name, "auroc")] - D_[(f"{name}@{PRIM}", "auroc")]
        d = lk(ck, dr) - lk(m, dr)
        rows.append({"cohort": c, "scenario": PRIM, "comparator": m.replace("_r0", ""), "runs": 1 if m.endswith("_r0") else CFG.n_repeats,
                     "loss_ckinn": lk(ck, pt), "loss_comparator": lk(m, pt), "ckinn_minus_comparator": lk(ck, pt) - lk(m, pt),
                     "lo": AN.ci(d)[0], "hi": AN.ci(d)[1], "comparator_trained_with_matching_augmentation": m.replace("_r0", "") in ("M2", "LR_aug", "XGB_aug")})
MISS_TEST = pd.DataFrame(rows)
# structural missingness in RICH: performance by observed pattern of (HDL, LDL, AST, smoking) missingness
fr = EVAL["RICH"]
pat = (fr.hdl_mmol.isna().astype(int).astype(str) + fr.ldl_mmol.isna().astype(int).astype(str) + fr.ast_u_l.isna().astype(int).astype(str)
       + fr.smoker.isna().astype(int).astype(str)).to_numpy()
rows = []
for pv in sorted(pd.unique(pat)):
    ix = np.flatnonzero(pat == pv)
    if Y["RICH"][ix].sum() < CFG.min_group_events:
        continue
    for m in ("LR", "XGB", "XGB_mono", "MI_LR", "M1", "M2", "CKINN"):
        runs = MAINRUNS if m in C.MAIN_MODELS else [(0, 0)]
        rows.append({"pattern_missing_hdl_ldl_ast_smoking": pv, "n": len(ix), "events": int(Y["RICH"][ix].sum()), "model": m,
                     "auroc": float(np.mean([MT.auroc_w(MT.Ranked(P["RICH"][(m, r, s)]["none"][ix], Y["RICH"][ix])) for r, s in runs]))})
STRUCT = pd.DataFrame(rows)
# leave-one-site-out
rows = []
for s_ in SITES_HELD:
    ix = np.flatnonzero(EVAL["RICH"].site.to_numpy() == s_)
    for m in SITEP:
        p = SITEP[m][ix]
        rows.append({"held_out_site": s_, "model": m, "n": len(ix), "events": int(Y["RICH"][ix].sum()), "auroc": MT.auroc_w(MT.Ranked(p, Y["RICH"][ix])),
                     "brier": MT.brier_w(Y["RICH"][ix], p), "citl": MT.citl_w(Y["RICH"][ix], p)})
SITE_TBL = pd.DataFrame(rows)
_ys = Y["RICH"][SITE_ROWS]
_bs = AN.bootstrap({"loso": {"y": _ys, "vecs": {m: [SITEP[m][SITE_ROWS]] for m in SITEP}, "stats": ["auroc", "brier"]}}, CFG.n_boot_desc, N_WORKERS)["loso"]
SITE_POOL = pd.DataFrame([{"model": m, "auroc_pooled_held_out": _bs["point"][(m, "auroc")],
                           "ckinn_minus_model": _bs["point"][("CKINN", "auroc")] - _bs["point"][(m, "auroc")],
                           "lo": AN.ci(_bs["draws"][("CKINN", "auroc")] - _bs["draws"][(m, "auroc")])[0],
                           "hi": AN.ci(_bs["draws"][("CKINN", "auroc")] - _bs["draws"][(m, "auroc")])[1]} for m in SITEP])
for name, df in (("table22_missingness_auroc", MISS_AUC), ("table23_missingness_loss_tests", MISS_TEST), ("table24_rich_structural_missingness", STRUCT),
                 ("table25_rich_site_held_out", SITE_TBL), ("table26_rich_site_pooled", SITE_POOL)):
    df.to_csv(OUT / "tables" / f"{name}.csv", index=False)
print(MISS_TEST.round(4).to_string(index=False)); print(SITE_POOL.round(4).to_string(index=False))
print(f"CELL 32 OK | missingness and site tables | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 33 | constraint evidence (E; manipulation check P1): in-distribution violation rates of M1, M2, CKINN by seed; leave-one-out,
# wrong-sign, alternative tables and lambda path; empirical signs against held-out partial dependence; sparse regions; other cohort
_t0 = time.perf_counter()
rows = []
for c in COHORTS:
    for m in NN:
        for s in range(CFG.n_seeds):
            vv, vo, ic, inn = 0.0, 0.0, 0.0, 0.0
            for k in range(CFG.n_splits):
                r_ = EN.load(CACHE, PL.tid("cv", c, "T2", "main" if s == 0 else "seed", "r0", f"k{k}", m, f"s{s}"))
                vv, vo = vv + r_["viol"]["viol"].sum(), vo + r_["viol"]["vobs"].sum()
                ic, inn = ic + r_["viol"]["ig_cons"].sum(), inn + r_["viol"]["ig_n"].sum()
            rows.append({"cohort": c, "model": m, "seed": s, "violation_rate": vv / max(vo, 1), "ig_direction_consistency": ic / max(inn, 1)})
MANIP = pd.DataFrame(rows).groupby(["cohort", "model"]).agg(violation_rate=("violation_rate", "mean"), violation_min=("violation_rate", "min"),
                                                             violation_max=("violation_rate", "max"),
                                                             ig_direction_consistency=("ig_direction_consistency", "mean")).reset_index()
# variants
rows, _pl = [], {}
for c in COHORTS:
    base = P[c][("CKINN", 0, 0)]["none"]
    _pl[c] = {"y": Y[c], "vecs": {"CKINN": [base], **{v: [PV[c][v]["none"]] for v in PV[c]}}, "stats": ["auroc", "brier"]}
BOOT_VAR = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
for c in COHORTS:
    pt, dr = BOOT_VAR[c]["point"], BOOT_VAR[c]["draws"]
    for v in PV[c]:
        vb = vo = vt = vto = vd = vdo = 0.0
        for k in range(CFG.n_splits):
            r_ = EN.load(CACHE, PL.tid("var", c, "T2", v, "r0", f"k{k}", "CKINN", "s0"))
            vb, vo = vb + r_["viol"]["viol"].sum(), vo + r_["viol"]["vobs"].sum()
            vt, vto = vt + r_["extra_viol"]["own_table"]["viol"].sum(), vto + r_["extra_viol"]["own_table"]["vobs"].sum()
            if "dropped" in r_["extra_viol"]:
                vd, vdo = vd + r_["extra_viol"]["dropped"]["viol"].sum(), vdo + r_["extra_viol"]["dropped"]["vobs"].sum()
        da, db = dr[(v, "auroc")] - dr[("CKINN", "auroc")], dr[(v, "brier")] - dr[("CKINN", "brier")]
        rows.append({"cohort": c, "variant": v, "auroc": pt[(v, "auroc")], "auroc_minus_ckinn": pt[(v, "auroc")] - pt[("CKINN", "auroc")],
                     "lo": AN.ci(da)[0], "hi": AN.ci(da)[1], "brier_minus_ckinn": pt[(v, "brier")] - pt[("CKINN", "brier")],
                     "brier_lo": AN.ci(db)[0], "brier_hi": AN.ci(db)[1], "violation_rate_v1_table": vb / max(vo, 1),
                     "violation_rate_own_table": vt / max(vto, 1), "violation_rate_dropped_feature": (vd / vdo) if vdo else np.nan})
VARIANTS = pd.DataFrame(rows)
# empirical signs (descriptive) against partial dependence of each model on held-out rows of repeat 0
rows = []
for c in COHORTS:
    feats = FEATS[(c, "T2")]
    sg = PL.signs_for("T2", feats)
    emp = AN.empirical_signs(EVAL[c], feats, sg)
    for m in ("LR", "GAM", "XGB", "XGB_mono", "M1", "M2", "CKINN"):
        slopes = {}
        for k in range(CFG.n_splits):
            r_ = EN.load(CACHE, PL.tid("cv", c, "T2", "main", "r0", f"k{k}", m, "s0"))
            for f, d in r_.get("pd", {}).items():
                slopes.setdefault(f, []).append(float(d["pd"][-1] - d["pd"][0]))
        for f, sl in slopes.items():
            e = emp[emp.feature == f].iloc[0]
            rows.append({"cohort": c, "model": m, "feature": f, "prior_sign": sg[f], "pd_change_p10_to_p90": float(np.mean(sl)),
                         "pd_sign_agrees_with_prior": bool(np.sign(np.mean(sl)) == sg[f]), "folds_agreeing": int(sum(np.sign(x) == sg[f] for x in sl)),
                         "adjusted_evidence": e.adjusted_evidence, "adjusted_coef": e.adjusted_coef, "coef_lo": e.coef_lo, "coef_hi": e.coef_hi,
                         "quintile_agrees": e.quintile_agrees})
EMPSIGN = pd.DataFrame(rows)
# sparse regions (held-out rows beyond the training 95th percentile of Mahalanobis distance) and the other cohort
rows = []
for c in COHORTS:
    for m in NN:
        sv = so = dv = do = npool = nsp = 0.0
        for k in range(CFG.n_splits):
            r_ = EN.load(CACHE, PL.tid("cv", c, "T2", "main", "r0", f"k{k}", m, "s0"))
            sv, so = sv + r_["sparse"]["viol"].sum(), so + r_["sparse"]["vobs"].sum()
            dv, do = dv + r_["viol"]["viol"].sum(), do + r_["viol"]["vobs"].sum()
            npool, nsp = npool + r_["sparse"]["n_pool"], nsp + r_["sparse"]["n_sparse"]
        rows.append({"cohort": c, "model": m, "region": "held-out sparse rows", "violation_rate": sv / max(so, 1), "share_sparse": nsp / max(npool, 1)})
        rows.append({"cohort": c, "model": m, "region": "held-out random rows", "violation_rate": dv / max(do, 1), "share_sparse": np.nan})
for s_ in [s for s in TR_SPECS if s["id"].startswith("trext") and s["_model"] in NN]:
    r_ = EN.load(CACHE, s_["id"])
    for tag in ("src", "tgt"):
        ev_ = r_["extra_viol"][tag]
        rows.append({"cohort": f"{s_['_src']}-trained", "model": s_["_model"], "region": f"{'source' if tag == 'src' else 'other cohort'} ({s_['_src'] if tag == 'src' else s_['_tgt']})",
                     "violation_rate": ev_["viol"].sum() / max(ev_["vobs"].sum(), 1), "share_sparse": np.nan, "seed": s_["_s"]})
SPARSE = pd.DataFrame(rows).groupby(["cohort", "model", "region"], dropna=False).agg(violation_rate=("violation_rate", "mean"),
                                                                                       share_sparse=("share_sparse", "mean")).reset_index()
for name, df in (("table27_manipulation_check", MANIP), ("table28_constraint_variants", VARIANTS), ("table29_empirical_signs_vs_pd", EMPSIGN),
                 ("table30_violation_sparse_and_other_cohort", SPARSE)):
    df.to_csv(OUT / "tables" / f"{name}.csv", index=False)
print(MANIP.round(4).to_string(index=False)); print(VARIANTS.round(4).to_string(index=False)); print(SPARSE.round(4).to_string(index=False))
print(f"CELL 33 OK | constraint evidence tables | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 34 | transport detail (P2) and secondary transport analyses (E): seed-averaged internal reference and external metrics;
# recalibration in the small with 1,000 target participants (target labels used); label-free reweighting; pooled training
_t0 = time.perf_counter()
rows = []
for src, tgt in (("NAGALA", "RICH"), ("RICH", "NAGALA")):
    b = BOOT_FAM[f"{src}->{tgt}"]
    y = Y[tgt]
    for m in NN + ["LR", "XGB", "XGB_mono"]:
        seeds = range(CFG.n_seeds) if m in NN else [0]
        ext = [TRX[(src, tgt, m, s)] for s in seeds]
        rows.append({"source": src, "target": tgt, "model": m, "seeds": len(ext), "auroc_target_cv": b["point"][(f"ref_{m}", "auroc")],
                     "auroc_external": b["point"][(f"ext_{m}", "auroc")], "transport_gap": b["point"][(f"ref_{m}", "auroc")] - b["point"][(f"ext_{m}", "auroc")],
                     "gap_lo": AN.ci(b["draws"][(f"ref_{m}", "auroc")] - b["draws"][(f"ext_{m}", "auroc")])[0],
                     "gap_hi": AN.ci(b["draws"][(f"ref_{m}", "auroc")] - b["draws"][(f"ext_{m}", "auroc")])[1],
                     "abs_citl_external": b["point"][(f"ext_{m}", "citl_abs")], "slope_external": b["point"][(f"ext_{m}", "slope")],
                     "brier_external": b["point"][(f"ext_{m}", "brier")]})
TRANS = pd.DataFrame(rows)
rows = []
for src, tgt in (("NAGALA", "RICH"), ("RICH", "NAGALA")):
    y = Y[tgt]
    rng = np.random.default_rng(C.seed_for("recal", tgt))
    cal_ix = np.sort(rng.choice(len(y), CFG.recal_n, replace=False))
    rest = np.setdiff1d(np.arange(len(y)), cal_ix)
    for m in NN + ["LR", "XGB_mono"]:
        p = TRX[(src, tgt, m, 0)]
        a, b_ = MT.calib_w(y[cal_ix], p[cal_ix])
        prc = 1 / (1 + np.exp(-(a + b_ * MT.logit(MT.clip(p)))))
        row = {"source": src, "target": tgt, "model": m, "events_in_recalibration_sample": int(y[cal_ix].sum()),
               "brier_before": MT.brier_w(y[rest], p[rest]), "brier_after": MT.brier_w(y[rest], prc[rest]),
               "citl_before": MT.citl_w(y[rest], p[rest]), "citl_after": MT.citl_w(y[rest], prc[rest]),
               "slope_before": MT.calib_w(y[rest], p[rest])[1], "slope_after": MT.calib_w(y[rest], prc[rest])[1]}
        if (src, tgt, m) in TRW:
            pw = TRW[(src, tgt, m)]
            row.update({"auroc_unweighted_s0": MT.auroc_w(MT.Ranked(p, y)), "auroc_reweighted": MT.auroc_w(MT.Ranked(pw, y)),
                        "abs_citl_unweighted_s0": abs(MT.citl_w(y, p)), "abs_citl_reweighted": abs(MT.citl_w(y, pw))})
        rows.append(row)
TRANS2 = pd.DataFrame(rows)
_pl = {c: {"y": Y[c], "vecs": {"M2": [P[c][("M2", 0, 0)]["none"]], "CKINN": [P[c][("CKINN", 0, 0)]["none"]],
                               "M2_pooled": [POOLP[("M2", c)]], "CKINN_pooled": [POOLP[("CKINN", c)]]}, "stats": ["auroc", "brier"]}
       for c in ("NAGALA", "RICH")}
BOOT_POOL = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
rows = []
for c in ("NAGALA", "RICH"):
    pt, dr = BOOT_POOL[c]["point"], BOOT_POOL[c]["draws"]
    for m in ("M2", "CKINN", "M2_pooled", "CKINN_pooled"):
        d = dr[(m, "auroc")] - dr[("M2", "auroc")]
        rows.append({"cohort": c, "variant": m, "auroc": pt[(m, "auroc")], "auroc_minus_M2": pt[(m, "auroc")] - pt[("M2", "auroc")],
                     "lo": AN.ci(d)[0], "hi": AN.ci(d)[1], "brier": pt[(m, "brier")]})
POOLTBL = pd.DataFrame(rows)
for name, df in (("table31_transport", TRANS), ("table32_transport_recalibration_reweighting", TRANS2), ("table33_pooled_training", POOLTBL)):
    df.to_csv(OUT / "tables" / f"{name}.csv", index=False)
print(TRANS.round(4).to_string(index=False)); print(TRANS2.round(4).to_string(index=False)); print(POOLTBL.round(4).to_string(index=False))
print(f"CELL 34 OK | transport tables | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 35 | sensitivity analyses (E): tier T3, RICH without family history, NHANES with the v1 label, NHANES survey-weighted metrics
_t0 = time.perf_counter()
rows, _pl = [], {}
for c in ("NAGALA", "RICH"):
    _pl[f"T3|{c}"] = {"y": Y[c], "vecs": {m: [PT3[c][(m, 0, 0)]["none"]] for m in _t3m}, "stats": ["auroc", "brier"]}
_pl["noFH|RICH"] = {"y": Y["RICH"], "vecs": {m: [PNOFH[(m, 0, 0)]["none"]] for m in ("LR", "M2", "CKINN")}, "stats": ["auroc", "brier"]}
_pl["v1label|NHANES"] = {"y": Y["NHANESv1"], "vecs": {m: [PNHV1[(m, 0, 0)]["none"]] for m in ("LR", "M2", "CKINN")}, "stats": ["auroc", "brier"]}
_sv = EVAL["NHANES"]
_wk = _sv.svy_w.to_numpy() > 0
_pl["survey|NHANES"] = {"y": Y["NHANES"][_wk], "vecs": {m: [P["NHANES"][(m, r, s)]["none"][_wk] for r, s in MAINRUNS] for m in FAIR_MODELS},
                        "stats": ["auroc", "brier", "citl"], "mode": "survey", "base_w": _sv.svy_w.to_numpy()[_wk],
                        "strata": _sv.svy_strata.to_numpy()[_wk], "psu": _sv.svy_psu.to_numpy()[_wk]}
BOOT_SENS = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
for key, b in BOOT_SENS.items():
    an, c = key.split("|")
    ms = sorted({m for m, _ in b["point"]})
    for m in ms:
        row = {"analysis": an, "cohort": c, "model": m, "auroc": b["point"][(m, "auroc")], "brier": b["point"][(m, "brier")]}
        for F in ("M2", "M1"):
            if F in ms and m == "CKINN":
                d = b["draws"][("CKINN", "auroc")] - b["draws"][(F, "auroc")]
                row[f"ckinn_minus_{F}"] = b["point"][("CKINN", "auroc")] - b["point"][(F, "auroc")]
                row[f"ckinn_minus_{F}_lo"], row[f"ckinn_minus_{F}_hi"] = AN.ci(d)
        if (m, "citl") in b["point"]:
            row["citl"] = b["point"][(m, "citl")]
        rows.append(row)
SENS = pd.DataFrame(rows)
SENS.to_csv(OUT / "tables" / "table34_sensitivity.csv", index=False)
print(SENS.round(4).to_string(index=False))
print(f"CELL 35 OK | sensitivity analyses | {time.perf_counter() - _t0:.0f}s")

## Post hoc additions for the 28-step protocol (label PH)
These cells close the steps of the 28-step protocol that the v2 analysis covered only partly or not at all, where the data allow it (`COVERAGE_28_STEPS.md`). They were decided **after** the v2 results had been seen, so every table they produce is labelled post hoc (PH) and none enters a hypothesis family or changes a v2 verdict. The frozen configuration and its hash are unchanged. The list, with reasons, is in `protocol_v2.md`, section 10.

Where a post hoc fit repeats a main-run fit (same seed, rows and configuration), its unmodified predictions must equal the stored ones; the reproducibility table reports the largest difference.

In [ ]:
# CELL PH1 | post hoc fits: SVM tuning and main runs (step 19); repeat-0 refits of every model with BMI 24/35 pairs and perturbation
# violations (step 2), SHAP (step 14) and reasoning inputs (steps 15, 27); variants (steps 6, 8, 20, 21, section 5); pooled training
# with a cohort indicator (step 20); transport for the remaining models (step 12); RICH one-step progression (proposal section 3C)
_t0 = time.perf_counter()
PHS = PH.settings(CFG)
_sv = {c: PH.svm_tune_specs(c, DEV[c], FEATS[(c, "T2")], CFG) for c in COHORTS}
EN.run_tasks([s for v in _sv.values() for s in v], CFG, CACHE, N_WORKERS, "PH: SVM tuning")
_rows = []
for c, sp in _sv.items():
    sc = PL.score_specs(sp, DEV[c], CACHE).groupby("ci").score.mean()
    grid = [s_["hp"] for s_ in sp if s_["id"].endswith("__k0")]
    HP[(c, "SVM")] = grid[int(sc.idxmin())]
    _rows += [{"cohort": c, "model": "SVM", "config": int(ci), "dev_score": v, "chosen": int(ci) == int(sc.idxmin()), "hp": json.dumps(grid[int(ci)])}
              for ci, v in sc.items()]
PH_TUNING = pd.DataFrame(_rows)
SVM_SPECS = {c: PL.cv_specs(c, "T2", FOLDS[c], FEATS[(c, "T2")], HP, LAM, CFG, models=("SVM",), variant="ph", extras=False) for c in COHORTS}
REFIT_SPECS = {c: PH.refit_specs(c, FOLDS[c], FEATS[(c, "T2")], HP, LAM, CFG) for c in COHORTS}
PHVAR_SPECS = {c: PH.variant_specs(c, FOLDS[c], FEATS[(c, "T2")], HP, LAM, CFG, EVAL[c]) for c in COHORTS}
FR["POOL_IND"] = FR["POOL"].assign(cohort_rich=(FR["POOL"].cohort == "RICH").astype(float))
FR["RICH_prog"] = PH.progression_frame(EVAL["RICH"])
EN.save_frames(CACHE, {"POOL_IND": FR["POOL_IND"], "RICH_prog": FR["RICH_prog"]})
PROG_FOLDS = D.repeated_folds(FR["RICH_prog"].y.to_numpy(), "RICH_prog", CFG.n_splits, 1)
PHPOOL_SPECS = PH.pool_indicator_specs(EVAL, FOLDS, POOLED_FEATS, HP, LAM, CFG)
PHTR_SPECS = PH.transport_specs(EVAL, FOLDS, SHARED, HP, CFG)
PROG_SPECS = PH.progression_specs(PROG_FOLDS, FEATS[("RICH", "T2")], HP, LAM, CFG)
PH_ALL_SPECS = ([s for v in _sv.values() for s in v] + [s for v in SVM_SPECS.values() for s in v] + [s for v in REFIT_SPECS.values() for s in v]
                + [s for v in PHVAR_SPECS.values() for s in v] + PHPOOL_SPECS + PHTR_SPECS + PROG_SPECS)
EN.run_tasks(PH_ALL_SPECS[len([s for v in _sv.values() for s in v]):], CFG, CACHE, N_WORKERS, "PH: fits")
PH_TUNING.to_csv(OUT / "tables" / "tablePH01_svm_tuning.csv", index=False)
print(PH_TUNING.round(5).to_string(index=False))
print(f"progression frame (RICH, baseline FPG < 5.6): {len(FR['RICH_prog']):,} rows, {int(FR['RICH_prog'].y.sum()):,} with progression")
print(f"CELL PH1 OK | post hoc fits done | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL PH2 | refit reproducibility; SVM performance (step 19); model-agnostic violation rates for every model and the BMI 24-to-35
# pair test (step 2); class-weighted versus unweighted (step 8); post hoc variants (steps 6, 20, 21, section 5); pooled training with
# a cohort indicator (step 20, M4)
_t0 = time.perf_counter()
SVMP = {c: AN.collect(SVM_SPECS[c], NE[c], CACHE)[0] for c in COHORTS}
SVMTHR = {c: AN.collect(SVM_SPECS[c], NE[c], CACHE)[1] for c in COHORTS}
rows_rep, viol_acc, pair_acc = [], {}, {}
for c in COHORTS:
    for s_ in REFIT_SPECS[c]:
        r_ = EN.load(CACHE, s_["id"])
        m, j = s_["_model"], s_["tests"][0]["idx"]
        ref = (SVMP[c][("SVM", 0, 0)]["none"] if m == "SVM" else P[c][(m, 0, 0)]["none"])[j].astype(float)
        rows_rep.append({"cohort": c, "model": m, "fold": s_["_k"], "max_abs_diff_vs_main_fit": float(np.max(np.abs(r_["preds"]["none"].astype(float) - ref)))})
        for f, v in r_["perturb"].items():
            a = viol_acc.setdefault((c, m, f), [0, 0, []])
            a[0] += v["n"]
            a[1] += v["viol"]
            if np.isfinite(v["mean_change"]):
                a[2].append(v["mean_change"])
        if "bmi24" in r_["preds"]:
            d = r_["preds"]["bmi35"].astype(float) - r_["preds"]["bmi24"].astype(float)
            a = pair_acc.setdefault((c, m), [0, 0, 0])
            a[0], a[1], a[2] = a[0] + len(d), a[1] + int((d < 0).sum()), a[2] + int((d == 0).sum())
REPRO = pd.DataFrame(rows_rep).groupby(["cohort", "model"]).max_abs_diff_vs_main_fit.max().reset_index()
rows = []
for (c, m, f), (n, k, mc) in viol_acc.items():
    lo, hi = PH.wilson(k, n)
    rows.append({"cohort": c, "model": m, "predictor": f, "n": n, "violations": k, "violation_rate": k / n if n else np.nan, "lo": lo, "hi": hi,
                 "mean_signed_logit_change": float(np.mean(mc)) if mc else np.nan})
VIOL_FEAT = pd.DataFrame(rows)
VIOL_ALL = VIOL_FEAT.groupby(["cohort", "model"]).agg(n=("n", "sum"), violations=("violations", "sum")).reset_index()
VIOL_ALL["violation_rate"] = VIOL_ALL.violations / VIOL_ALL.n
VIOL_ALL[["lo", "hi"]] = [PH.wilson(int(k), int(n)) for k, n in zip(VIOL_ALL.violations, VIOL_ALL.n)]
rows = []
for (c, m), (n, k, eq) in pair_acc.items():
    lo, hi = PH.wilson(k, n)
    rows.append({"cohort": c, "model": m, "pairs": n, "risk_lower_at_bmi35": k, "share_lower": k / n, "lo": lo, "hi": hi, "share_unchanged": eq / n})
BMIPAIRS = pd.DataFrame(rows)
# SVM performance and CKINN minus SVM (run-averaged, same bootstrap convention as table 8)
_pl = {c: {"y": Y[c], "vecs": {"SVM": [SVMP[c][("SVM", r, s)]["none"] for r, s in MAINRUNS], "CKINN": [P[c][("CKINN", r, s)]["none"] for r, s in MAINRUNS]},
           "stats": ["auroc", "brier", "slope", "citl", "ece"]} for c in COHORTS}
_b = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
rows = []
for c in COHORTS:
    pt, dr = _b[c]["point"], _b[c]["draws"]
    loss = float(np.mean([MT.auroc_w(MT.Ranked(SVMP[c][("SVM", r, s)]["none"], Y[c])) - MT.auroc_w(MT.Ranked(SVMP[c][("SVM", r, s)][PRIM], Y[c]))
                          for r, s in MAINRUNS]))
    rows.append({"cohort": c, "model": "SVM", "auroc": pt[("SVM", "auroc")], "auroc_lo": AN.ci(dr[("SVM", "auroc")])[0], "auroc_hi": AN.ci(dr[("SVM", "auroc")])[1],
                 "auprc": float(np.mean([AN.auprc(Y[c], SVMP[c][("SVM", r, s)]["none"]) for r, s in MAINRUNS])), "brier": pt[("SVM", "brier")],
                 "slope": pt[("SVM", "slope")], "citl": pt[("SVM", "citl")], "ece": pt[("SVM", "ece")], "ckinn_minus_svm_auroc": pt[("CKINN", "auroc")] - pt[("SVM", "auroc")],
                 "diff_lo": AN.ci(dr[("CKINN", "auroc")] - dr[("SVM", "auroc")])[0], "diff_hi": AN.ci(dr[("CKINN", "auroc")] - dr[("SVM", "auroc")])[1],
                 f"auroc_loss_{PRIM}": loss})
SVMPERF = pd.DataFrame(rows)
# class weights and variants (repeat 0, seed index 0): each against the same model fitted without the change
VARP = {}
for c in COHORTS:
    for s_ in PHVAR_SPECS[c]:
        VARP.setdefault((c, s_["_variant"], s_["_model"]), np.full(NE[c], np.nan))[s_["tests"][0]["idx"]] = EN.load(CACHE, s_["id"])["preds"]["none"]
_pl = {}
for (c, v, m), p in VARP.items():
    assert np.isfinite(p).all()
    _pl[f"{c}|{v}|{m}"] = {"y": Y[c], "vecs": {"reference": [P[c][(m, 0, 0)]["none"]], "variant": [p]}, "groups": GROUPS[c], "fpr": CFG.fpr_matched,
                           "stats": ["auroc", "brier", "slope", "citl", "tpr_sex_gap", "tpr_age_range"]}
_b = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
rows = []
for key, b in _b.items():
    c, v, m = key.split("|")
    row = {"cohort": c, "variant": v, "model": m}
    for st in ("auroc", "brier", "slope", "citl", "tpr_sex_gap", "tpr_age_range"):
        row[f"{st}_reference"], row[f"{st}_variant"] = b["point"][("reference", st)], b["point"][("variant", st)]
        row[f"{st}_diff"] = b["point"][("variant", st)] - b["point"][("reference", st)]
        row[f"{st}_diff_lo"], row[f"{st}_diff_hi"] = AN.ci(b["draws"][("variant", st)] - b["draws"][("reference", st)])
    rows.append(row)
PHVAR = pd.DataFrame(rows)
# pooled training with a cohort indicator (M4) against the same pooled training without it and against single-cohort training
PPI = {}
for s_ in PHPOOL_SPECS:
    r_ = EN.load(CACHE, s_["id"])
    for t in s_["tests"]:
        off = 0 if t["tag"] == "NAGALA" else NE["NAGALA"]
        PPI.setdefault((s_["_model"], t["tag"]), np.full(NE[t["tag"]], np.nan))[t["idx"] - off] = r_["preds"][t["tag"]]
_pl = {c: {"y": Y[c], "vecs": {f"{m}_{w}": [p] for m in ("M2", "CKINN") for w, p in (("single", P[c][(m, 0, 0)]["none"]), ("pooled", POOLP[(m, c)]),
                                                                                     ("pooled_indicator", PPI[(m, c)]))}, "stats": ["auroc", "brier"]}
       for c in ("NAGALA", "RICH")}
_b = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
rows = []
for c in ("NAGALA", "RICH"):
    for m in ("M2", "CKINN"):
        pt, dr = _b[c]["point"], _b[c]["draws"]
        d = dr[(f"{m}_pooled_indicator", "auroc")] - dr[(f"{m}_pooled", "auroc")]
        rows.append({"cohort": c, "model": m, "auroc_single": pt[(f"{m}_single", "auroc")], "auroc_pooled": pt[(f"{m}_pooled", "auroc")],
                     "auroc_pooled_indicator": pt[(f"{m}_pooled_indicator", "auroc")],
                     "indicator_minus_pooled": pt[(f"{m}_pooled_indicator", "auroc")] - pt[(f"{m}_pooled", "auroc")], "lo": AN.ci(d)[0], "hi": AN.ci(d)[1]})
PHPOOL = pd.DataFrame(rows)
for name, df in (("tablePH02_refit_reproducibility", REPRO), ("tablePH03_svm_performance", SVMPERF), ("tablePH04_violation_by_predictor", VIOL_FEAT),
                 ("tablePH05_violation_all_models", VIOL_ALL), ("tablePH06_bmi_24_35_pairs", BMIPAIRS), ("tablePH07_variants", PHVAR),
                 ("tablePH08_pooled_cohort_indicator", PHPOOL)):
    df.to_csv(OUT / "tables" / f"{name}.csv", index=False)
with pd.option_context("display.width", 250, "display.max_columns", 30, "display.max_rows", 200):
    print(REPRO.to_string(index=False)); print(SVMPERF.round(4).to_string(index=False)); print(VIOL_ALL.round(4).to_string(index=False))
    print(BMIPAIRS.round(4).to_string(index=False)); print(PHVAR[["cohort", "variant", "model", "auroc_diff", "auroc_diff_lo", "auroc_diff_hi", "brier_diff", "citl_diff"]].round(4).to_string(index=False))
    print(PHPOOL.round(4).to_string(index=False))
print(f"CELL PH2 OK | reproducibility, SVM, violations, variants and pooled-indicator tables | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL PH3 | train-by-test transport matrix for every model, seed 0 (step 12); intercept-only recalibration in the target with 1,000
# participants, target labels used (step 17); older-and-lean stress test across lambda (step 10); RQ7: incident diabetes among people with
# baseline FPG below 5.6 mmol/L (step 22); RICH one-step progression (proposal 3C); AUPRC and false-negative rate by group (step 16)
_t0 = time.perf_counter()
TRX0, TRREF0 = {(a, b, m): v for (a, b, m, s), v in TRX.items() if s == 0}, {(t, m): v for (t, m, s), v in TRREF.items() if s == 0}
for s_ in [x for x in PHTR_SPECS if x["id"].startswith("phtrext")]:
    TRX0[(s_["_src"], s_["_tgt"], s_["_model"])] = EN.load(CACHE, s_["id"])["preds"]["tgt"].astype(float)
for _tgt in ("RICH", "NAGALA"):
    _p = AN.collect([x for x in PHTR_SPECS if x["id"].startswith("phtrref") and x["_tgt"] == _tgt], NE[_tgt], CACHE, key=lambda s: s["_model"])[0]
    TRREF0.update({(_tgt, m): v["none"].astype(float) for m, v in _p.items()})
MATRIX_MODELS = ["LR", "cLR", "GAM", "RF", "XGB", "XGB_mono", "LGBM", "LGBM_mono", "SVM", "MI_LR", "M1", "M2", "CKINN"]
_pl = {f"{src}->{tgt}": {"y": Y[tgt], "vecs": {**{f"ext_{m}": [TRX0[(src, tgt, m)]] for m in MATRIX_MODELS},
                                                **{f"ref_{m}": [TRREF0[(tgt, m)]] for m in MATRIX_MODELS}}, "stats": ["auroc"]}
       for src, tgt in (("NAGALA", "RICH"), ("RICH", "NAGALA"))}
_b = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
rows, rows_rc = [], []
for m in MATRIX_MODELS:
    row = {"model": m}
    for src, tgt in (("NAGALA", "RICH"), ("RICH", "NAGALA")):
        pt, dr = _b[f"{src}->{tgt}"]["point"], _b[f"{src}->{tgt}"]["draws"]
        row[f"within_{tgt}_cv"] = pt[(f"ref_{m}", "auroc")]
        row[f"{src}_to_{tgt}"] = pt[(f"ext_{m}", "auroc")]
        row[f"gap_{src}_to_{tgt}"] = pt[(f"ref_{m}", "auroc")] - pt[(f"ext_{m}", "auroc")]
        row[f"gap_{src}_to_{tgt}_lo"], row[f"gap_{src}_to_{tgt}_hi"] = AN.ci(dr[(f"ref_{m}", "auroc")] - dr[(f"ext_{m}", "auroc")])
        y = Y[tgt]
        cal = np.sort(np.random.default_rng(C.seed_for("recal", tgt)).choice(len(y), CFG.recal_n, replace=False))
        rest = np.setdiff1d(np.arange(len(y)), cal)
        p = TRX0[(src, tgt, m)]
        a = MT.citl_w(y[cal], p[cal])
        prc = 1 / (1 + np.exp(-(MT.logit(MT.clip(p)) + a)))
        rows_rc.append({"source": src, "target": tgt, "model": m, "intercept_shift": a, "brier_before": MT.brier_w(y[rest], p[rest]),
                        "brier_after": MT.brier_w(y[rest], prc[rest]), "citl_before": MT.citl_w(y[rest], p[rest]), "citl_after": MT.citl_w(y[rest], prc[rest]),
                        "mean_predicted_after": float(prc[rest].mean()), "observed_rate": float(y[rest].mean()),
                        "calibration_n": int(len(cal)), "calibration_events": int(y[cal].sum()), "calibration_observed_rate": float(y[cal].mean())})
    rows.append(row)
TMATRIX, RECAL_INT = pd.DataFrame(rows), pd.DataFrame(rows_rc)
# older-and-lean stress test (age 65 or more and BMI below 20) across lambda; lambda 0 is M2 (same configuration, seed and rows)
rows, _pl = [], {}
for c in COHORTS:
    fr = EVAL[c]
    g = np.flatnonzero(((fr.age_years >= 65) & (fr.bmi_kgm2 < 20)).to_numpy())
    ev = int(Y[c][g].sum())
    lam_vecs = {"lam_0": P[c][("M2", 0, 0)]["none"][g]}
    lam_vecs.update({f"lam_{l:g}": PV[c][f"lam_{l:g}"]["none"][g] for l in CFG.lam_grid})
    ok = ev >= CFG.min_group_events and (len(g) - ev) >= CFG.min_group_events
    if ok:
        _pl[c] = {"y": Y[c][g], "vecs": {k: [v] for k, v in lam_vecs.items()}, "stats": ["auroc", "citl"]}
    for k, v in lam_vecs.items():
        rows.append({"cohort": c, "lambda": float(k[4:]), "n": len(g), "events": ev, "estimable": ok, "mean_predicted": float(v.mean()) if len(g) else np.nan,
                     "observed_rate": float(Y[c][g].mean()) if len(g) else np.nan})
STRESS = pd.DataFrame(rows)
if _pl:
    _b = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
    for col in ("auroc", "auroc_lo", "auroc_hi", "citl"):
        STRESS[col] = np.nan
    for c, b in _b.items():
        for k in b["point"]:
            if k[1] == "auroc":
                ix = STRESS.index[(STRESS.cohort == c) & (STRESS["lambda"] == float(k[0][4:]))]
                STRESS.loc[ix, "auroc"] = b["point"][k]
                STRESS.loc[ix, ["auroc_lo", "auroc_hi"]] = AN.ci(b["draws"][k])
                STRESS.loc[ix, "citl"] = b["point"][(k[0], "citl")]
# RQ7: people below the impaired-fasting-glucose threshold at baseline
RQ7_MODELS = ("LR", "XGB_mono", "M1", "M2", "CKINN")
rows, _pl = [], {}
for c in ("NAGALA", "RICH"):
    g = np.flatnonzero((EVAL[c].fpg_mmol < 5.6).to_numpy())
    _pl[f"T2|{c}"] = {"y": Y[c][g], "vecs": {m: [P[c][(m, r, s)]["none"][g] for r, s in MAINRUNS] for m in RQ7_MODELS}, "stats": ["auroc", "citl", "brier"]}
    _pl[f"T3|{c}"] = {"y": Y[c][g], "vecs": {m: [PT3[c][(m, 0, 0)]["none"][g]] for m in RQ7_MODELS}, "stats": ["auroc", "citl", "brier"]}
_b = AN.bootstrap(_pl, CFG.n_boot_desc, N_WORKERS)
for key, b in _b.items():
    tier, c = key.split("|")
    g = np.flatnonzero((EVAL[c].fpg_mmol < 5.6).to_numpy())
    for m in RQ7_MODELS:
        vec = [P[c][(m, r, s)]["none"][g] for r, s in MAINRUNS] if tier == "T2" else [PT3[c][(m, 0, 0)]["none"][g]]
        rows.append({"cohort": c, "tier": tier, "model": m, "runs": len(vec), "n": len(g), "events": int(Y[c][g].sum()), "auroc": b["point"][(m, "auroc")],
                     "auroc_lo": AN.ci(b["draws"][(m, "auroc")])[0], "auroc_hi": AN.ci(b["draws"][(m, "auroc")])[1],
                     "tpr_at_fpr_0.10": float(np.mean([MT.tpr_at_fpr_w(MT.Ranked(v, Y[c][g]), CFG.fpr_matched) for v in vec])),
                     "citl": b["point"][(m, "citl")], "brier": b["point"][(m, "brier")]})
RQ7 = pd.DataFrame(rows)
# RICH one-step progression (healthy at baseline to FPG of 5.6 mmol/L or more, or diabetes)
PROGP = AN.collect(PROG_SPECS, len(FR["RICH_prog"]), CACHE)[0]
_yp = FR["RICH_prog"].y.to_numpy().astype(float)
_b = AN.bootstrap({"prog": {"y": _yp, "vecs": {m: [PROGP[(m, 0, 0)]["none"]] for m in PH.PROGRESSION_MODELS}, "stats": ["auroc", "brier", "slope", "citl"]}},
                  CFG.n_boot_desc, N_WORKERS)["prog"]
rows = []
for m in PH.PROGRESSION_MODELS:
    row = {"model": m, "n": len(_yp), "events": int(_yp.sum()), "auroc": _b["point"][(m, "auroc")], "auroc_lo": AN.ci(_b["draws"][(m, "auroc")])[0],
           "auroc_hi": AN.ci(_b["draws"][(m, "auroc")])[1], "auprc": AN.auprc(_yp, PROGP[(m, 0, 0)]["none"]), "brier": _b["point"][(m, "brier")],
           "slope": _b["point"][(m, "slope")], "citl": _b["point"][(m, "citl")]}
    for F in ("M2", "M1"):
        if m == "CKINN":
            d = _b["draws"][("CKINN", "auroc")] - _b["draws"][(F, "auroc")]
            row[f"ckinn_minus_{F}"], (row[f"ckinn_minus_{F}_lo"], row[f"ckinn_minus_{F}_hi"]) = _b["point"][("CKINN", "auroc")] - _b["point"][(F, "auroc")], AN.ci(d)
    rows.append(row)
PROG = pd.DataFrame(rows)
# AUPRC and false-negative rate (at the common threshold) by group
rows = []
for c in COHORTS:
    fr, y = EVAL[c], Y[c]
    for gt, gv in (("sex", fr.sex.to_numpy()), ("age_band", fr.age_band.to_numpy())):
        for gname in sorted(pd.unique(gv), key=str):
            ix = np.flatnonzero(gv == gname)
            ok = y[ix].sum() >= CFG.min_group_events and (y[ix] == 0).sum() >= CFG.min_group_events
            for m in FAIR_MODELS:
                vals = [(AN.auprc(y[ix], P[c][(m, r, s)]["none"][ix]) if ok else np.nan,
                         float(((P[c][(m, r, s)]["none"][ix] < THR[c][(m, r, s)][0][ix]) & (y[ix] == 1)).sum() / max((y[ix] == 1).sum(), 1)) if ok else np.nan)
                        for r, s in MAINRUNS]
                rows.append({"cohort": c, "group_type": gt, "group": str(gname), "model": m, "n": len(ix), "events": int(y[ix].sum()), "estimable": ok,
                             "auprc": float(np.mean([v[0] for v in vals])), "fnr_common_thr": float(np.mean([v[1] for v in vals])),
                             "prevalence": float(y[ix].mean())})
GROUP_PR = pd.DataFrame(rows)
for name, df in (("tablePH09_transport_matrix", TMATRIX), ("tablePH10_intercept_recalibration", RECAL_INT), ("tablePH11_stress_older_lean", STRESS),
                 ("tablePH12_rq7_fpg_below_5_6", RQ7), ("tablePH13_rich_progression", PROG), ("tablePH14_auprc_fnr_by_group", GROUP_PR)):
    df.to_csv(OUT / "tables" / f"{name}.csv", index=False)
with pd.option_context("display.width", 250, "display.max_columns", 40, "display.max_rows", 200):
    print(TMATRIX.round(4).to_string(index=False)); print(STRESS.round(4).to_string(index=False)); print(RQ7.round(4).to_string(index=False))
    print(PROG.round(4).to_string(index=False))
print(f"CELL PH3 OK | transport matrix, recalibration, stress test, RQ7, progression and group AUPRC tables | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL PH4 | SHAP summary and agreement with the priors and with integrated gradients (step 14); patient-level reasoning text for CKINN
# (steps 15, 27; texts stay in the local cache); audit table and unavailable variables (step 5); sign table with references (step 6);
# one scorecard per cohort and model (steps 24, 27); partial-dependence and SHAP figures (step 14)
from scipy.stats import spearmanr
_t0 = time.perf_counter()
rows_imp, rows_dir, rows_ig, rows_rs = [], [], [], []
for c in COHORTS:
    task = TASK[c].task.iloc[0]
    for s_ in REFIT_SPECS[c]:
        if "shap" not in s_:
            continue
        r_ = EN.load(CACHE, s_["id"])
        sh, m = r_["shap"], s_["_model"]
        phi, x, feats = sh["phi"].astype(float), sh["x"].astype(float), sh["feats"]
        imp = np.abs(phi).mean(0)
        rows_imp += [{"cohort": c, "model": m, "feature": f, "mean_abs_shap": float(v), "rank": int(rk) + 1}
                     for f, v, rk in zip(feats, imp, np.argsort(np.argsort(-imp)))]
        for f, sgn in PL.signs_for("T2", feats).items():
            jx = feats.index(f)
            obs = ~np.isnan(x[:, jx])
            if f in C.BINARY:
                one, zero = obs & (x[:, jx] == 1), obs & (x[:, jx] == 0)
                eff = float(phi[one, jx].mean() - phi[zero, jx].mean()) if one.any() and zero.any() else np.nan
            else:
                eff = float(spearmanr(x[obs, jx], phi[obs, jx])[0]) if obs.sum() > 2 and np.std(x[obs, jx]) > 0 else np.nan
            rows_dir.append({"cohort": c, "model": m, "feature": f, "prior_sign": sgn, "shap_direction_statistic": eff,
                             "agrees_with_prior": bool(np.sign(eff) == sgn) if np.isfinite(eff) else None})
        if m in NN:
            A = EN.load(CACHE, PL.tid("cv", c, "T2", "main", "r0", "k0", m, "s0"))["attr"].astype(float)
            rows_ig.append({"cohort": c, "model": m, "spearman_mean_abs_shap_vs_mean_abs_ig": float(spearmanr(imp, np.abs(A).mean(0))[0])})
        if "reason" in r_:
            rec = r_["reason"]
            texts = [PH.reasoning_text(rec, i, task) for i in range(len(rec["p"]))]
            pd.DataFrame({"evaluation_row": s_["shap"]["idx"][:len(texts)], "text": texts}).to_csv(CACHE / "preds" / f"reasoning_{c}.csv", index=False)
            top = [rec["feats"][int(np.argmax(np.abs(rec["phi"][i])))] for i in range(len(rec["p"]))]
            drops = [np.nanmax(rec["p"][i] - rec["p_cf"][i]) if np.isfinite(rec["p_cf"][i]).any() else np.nan for i in range(len(rec["p"]))]
            rows_rs.append({"cohort": c, "patients": len(texts), "share_consistent_with_all_priors": float((~rec["viol_point"].any(1)).mean()),
                            "most_common_top_contributor": pd.Series(top).mode().iloc[0], "share_with_lowering_panel_counterfactual": float(np.mean([d > 0 for d in drops if np.isfinite(d)])) if any(np.isfinite(drops)) else np.nan,
                            "median_largest_drop_in_probability": float(np.nanmedian([d for d in drops if np.isfinite(d) and d > 0])) if any(np.isfinite(d) and d > 0 for d in drops) else np.nan})
SHAP_IMP, SHAP_DIR, SHAP_IG, REASON = pd.DataFrame(rows_imp), pd.DataFrame(rows_dir), pd.DataFrame(rows_ig), pd.DataFrame(rows_rs)
AUDIT_TBL = PH.audit_table(TASK, FEATS, list(C.ALL_FEATURES) + ["tg_hdl_ratio", "prediabetes_told", "hypertension_told"])
UNAVAIL = pd.DataFrame(PH.UNAVAILABLE, columns=["proposal_variable", "status_in_these_data"])
SIGNREF = pd.DataFrame([{"predictor": f, "prior_sign": "+" if C.SIGNS_T3.get(f, 1 if f == "dbp_mmHg" else 0) > 0 else "-",
                         "used_in": "T2 and T3" if f in C.SIGNS_T2 else ("T3 only" if f in C.SIGNS_T3 else "post hoc variant"), "pmid": pm, "reference": ref}
                        for f, (pm, ref) in PH.SIGN_CITATIONS.items()])
# scorecard: one row per cohort and model (run-averaged where five runs exist)
_sc_models = list(C.MAIN_MODELS) + ["SVM"]
_perf = pd.concat([PERF[PERF.model.isin(C.MAIN_MODELS)][["cohort", "model", "auroc", "auprc", "brier", "slope", "citl", "ece"]],
                   SVMPERF[["cohort", "model", "auroc", "auprc", "brier", "slope", "citl", "ece"]]], ignore_index=True)
rows = []
for c in COHORTS:
    y = Y[c]
    for m in _sc_models:
        PP, TT = (SVMP[c], SVMTHR[c]) if m == "SVM" else (P[c], THR[c])
        tm = [MT.threshold_metrics(y, PP[(m, r, s)]["none"], TT[(m, r, s)][0]) for r, s in MAINRUNS]
        g = GROUPS[c]
        sexgap = float(np.mean([abs(MT.tpr_at_fpr_w(MT.Ranked(PP[(m, r, s)]["none"][g["sex"][0]], y[g["sex"][0]]), CFG.fpr_matched)
                                    - MT.tpr_at_fpr_w(MT.Ranked(PP[(m, r, s)]["none"][g["sex"][1]], y[g["sex"][1]]), CFG.fpr_matched)) for r, s in MAINRUNS])) if len(g["sex"]) == 2 else np.nan
        row = {"cohort": c, "model": m, **{k: float(np.mean([t[k] for t in tm])) for k in ("sens", "spec", "ppv", "npv")}, "sex_gap_tpr_at_fpr_0.10": sexgap}
        va = VIOL_ALL[(VIOL_ALL.cohort == c) & (VIOL_ALL.model == m)]
        row["violation_rate_perturbation"] = float(va.violation_rate.iloc[0]) if len(va) else np.nan
        bp = BMIPAIRS[(BMIPAIRS.cohort == c) & (BMIPAIRS.model == m)]
        row["bmi_pair_share_lower"] = float(bp.share_lower.iloc[0]) if len(bp) else np.nan
        if c in ("NAGALA", "RICH") and m in MATRIX_MODELS:
            src = "RICH" if c == "NAGALA" else "NAGALA"
            row["transport_gap_into_this_cohort"] = float(TMATRIX.loc[TMATRIX.model == m, f"gap_{src}_to_{c}"].iloc[0])
        rows.append(row)
SCORECARD = _perf.merge(pd.DataFrame(rows), on=["cohort", "model"], how="right")
# share of constrained predictors whose SHAP dependence has the prior sign, stored so the report reads it from a table
SHAP_DIRSUM = (SHAP_DIR.assign(_a=SHAP_DIR.agrees_with_prior.map({True: 1.0, False: 0.0}))
               .groupby(["cohort", "model"])._a.agg(n_constrained="size", n_estimable="count", n_agree="sum", share_agree="mean").reset_index())
SHAP_DIRSUM["n_agree"] = SHAP_DIRSUM.n_agree.astype(int)
for name, df in (("tablePH15_shap_importance", SHAP_IMP), ("tablePH16_shap_direction_vs_prior", SHAP_DIR), ("tablePH17_shap_vs_integrated_gradients", SHAP_IG),
                 ("tablePH18_reasoning_summary", REASON), ("tablePH19_predictor_audit", AUDIT_TBL), ("tablePH20_unavailable_variables", UNAVAIL),
                 ("tablePH21_sign_table_references", SIGNREF), ("tablePH22_scorecard", SCORECARD),
                 ("tablePH23_shap_direction_summary", SHAP_DIRSUM)):
    df.to_csv(OUT / "tables" / f"{name}.csv", index=False)
# figures: partial dependence (repeat 0, averaged over folds) and mean |SHAP|
_FIG = OUT / "figures"
_col = {"M2": "#2a78d6", "CKINN": "#eb6834", "LR": "#6250d6", "XGB_mono": "#52514e"}
_mk = {"M2": "o", "CKINN": "s", "LR": "D", "XGB_mono": "v"}
_pdf = {c: [f for f in PL.signs_for("T2", FEATS[(c, "T2")]) if f not in C.BINARY] for c in COHORTS}
_nc = max(len(v) for v in _pdf.values())
fig, axs = plt.subplots(3, _nc, figsize=(2.3 * _nc, 6.6), squeeze=False)
for ri, c in enumerate(COHORTS):
    for ci in range(_nc):
        ax = axs[ri][ci]
        if ci >= len(_pdf[c]):
            ax.axis("off")
            continue
        f = _pdf[c][ci]
        for m in ("M2", "CKINN", "XGB_mono", "LR"):
            cur = [EN.load(CACHE, PL.tid("cv", c, "T2", "main", "r0", f"k{k}", m, "s0"))["pd"][f] for k in range(CFG.n_splits)]
            ax.plot(np.mean([q["grid"] for q in cur], 0), np.mean([q["pd"] for q in cur], 0), marker=_mk[m], ms=3, color=_col[m], label=m, lw=1.2)
        ax.set_title(f"{c}: {PH.LABELS.get(f, f)}", fontsize=7)
        ax.tick_params(labelsize=6)
axs[0][0].legend(fontsize=6)
fig.supylabel("mean predicted risk (partial dependence)", fontsize=8)
fig.tight_layout()
for e, kw in (("pdf", {}), ("png", {"dpi": 300})):
    fig.savefig(_FIG / f"figPH1_partial_dependence.{e}", bbox_inches="tight", **kw)
plt.close(fig)
fig, axs = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, c in zip(axs, COHORTS):
    sub = SHAP_IMP[(SHAP_IMP.cohort == c) & SHAP_IMP.model.isin(["M2", "CKINN", "XGB_mono"])]
    order = sub[sub.model == "CKINN"].sort_values("mean_abs_shap").feature.tolist()
    for k, m in enumerate(("M2", "CKINN", "XGB_mono")):
        v = sub[sub.model == m].set_index("feature").reindex(order).mean_abs_shap
        ax.barh(np.arange(len(order)) + (k - 1) * 0.27, v, height=0.25, color=_col[m], label=m)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([PH.LABELS.get(f, f) for f in order], fontsize=6)
    ax.set_title(f"{c}: mean |SHAP| (log-odds)", fontsize=8)
axs[0].legend(fontsize=6)
fig.tight_layout()
for e, kw in (("pdf", {}), ("png", {"dpi": 300})):
    fig.savefig(_FIG / f"figPH2_shap_importance.{e}", bbox_inches="tight", **kw)
plt.close(fig)
with pd.option_context("display.width", 250, "display.max_columns", 40, "display.max_rows", 120):
    print(SHAP_DIRSUM.round(3).to_string(index=False)); print(SHAP_IG.round(3).to_string(index=False))
    print(REASON.round(3).to_string(index=False)); print(SCORECARD.round(4).to_string(index=False))
print(f"CELL PH4 OK | SHAP, reasoning summary, audit, references, scorecard and two figures | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 36 | figures: vector PDF and 300 dpi PNG. Colours follow the model, never its rank: M2 blue, CKINN orange, M1 aqua, LR violet;
# every series also has its own marker and line style, so identity never rests on colour alone
_t0 = time.perf_counter()
FIG = OUT / "figures"
COL = {"M2": "#2a78d6", "CKINN": "#eb6834", "M1": "#1baf7a", "LR": "#6250d6", "XGB_mono": "#52514e"}
MRK = {"M2": "o", "CKINN": "s", "M1": "^", "LR": "D", "XGB_mono": "v"}
LST = {"M2": "-", "CKINN": "-", "M1": "--", "LR": ":", "XGB_mono": "-."}
plt.rcParams.update({"font.size": 8, "axes.edgecolor": "#52514e", "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e1", "grid.linewidth": 0.6, "legend.frameon": False, "figure.facecolor": "white",
                     "savefig.facecolor": "white", "lines.linewidth": 1.5, "axes.axisbelow": True})


def savefig(fig, name):
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(FIG / f"{name}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


SHORT = {"AUROC (non-inferiority, margin 0.01)": "AUROC, non-inferiority", "Brier score": "Brier", "|calibration slope - 1|": "|slope - 1|",
         "across-seed attribution agreement (Spearman)": "attribution agreement across seeds", "sex gap in sensitivity at FPR 0.10": "sex gap, TPR at FPR 0.10",
         "age-band range in sensitivity at FPR 0.10": "age-band range, TPR at FPR 0.10", "transport gap (target CV AUROC - external AUROC)": "transport gap",
         "|calibration-in-the-large| in the target": "|CITL| in target"}
# Figure 1: hypothesis endpoints, standardised difference (difference / bootstrap SD, oriented so that right = the predicted direction)
_pr = EP[EP.primary].copy()
_pr["z"] = np.where(_pr.predicted == "higher", 1, -1) * (_pr.difference + np.where(_pr.hypothesis == "H1", _pr.margin, 0)) / _pr.boot_sd
fig, axs = plt.subplots(1, 2, figsize=(9, 0.26 * _pr.groupby("family").size().max() + 1.2), sharey=True)
for ax, (fam, g) in zip(axs, _pr.groupby("family")):
    g = g.reset_index(drop=True)
    ax.barh(range(len(g)), g.z, color=[COL["CKINN"] if st == "predicted" else "#9b9a95" if st == "null" else "#e34948" for st in g.status], height=0.6)
    ax.axvline(0, color="#0b0b0b", lw=0.8)
    for xv in (-1.96, 1.96):
        ax.axvline(xv, color="#9b9a95", ls=":", lw=0.8)
    ax.set_yticks(range(len(g)))
    ax.set_yticklabels([f"{h} {u.replace(' (T2)', '')}: {SHORT.get(e, e)}" for h, u, e in zip(g.hypothesis, g.unit, g.endpoint)], fontsize=6)
    ax.set_title(fam, fontsize=8)
    ax.set_xscale("symlog", linthresh=2)
    ax.set_xlabel("difference / bootstrap SD, symmetric log scale\n(right = predicted direction; H1 measured from the -0.01 margin)")
axs[0].invert_yaxis()
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color=COL["CKINN"], label="predicted (Holm)"), Patch(color="#9b9a95", label="null"),
                    Patch(color="#e34948", label="opposite (Holm)")], fontsize=7, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.06))
savefig(fig, "fig1_hypothesis_endpoints")
# Figure 2: reliability curves with bands
fig, axs = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, c in zip(axs, COHORTS):
    sub = CALCURVE[CALCURVE.cohort == c]
    for m in ("M1", "M2", "CKINN"):
        g = sub[sub.model == m]
        ax.fill_between(g.mean_predicted, g.obs_lo, g.obs_hi, color=COL[m], alpha=0.15, lw=0)
        ax.plot(g.mean_predicted, g.observed, LST[m], marker=MRK[m], ms=4, color=COL[m], label=m)
    lim = float(max(sub.mean_predicted.max(), sub.observed.max()))
    ax.plot([0, lim], [0, lim], color="#9b9a95", lw=0.8)
    ax.set_title(f"{c}: calibration (run 0)")
    ax.set_xlabel("mean predicted risk")
    ax.set_ylabel("observed proportion")
    ax.legend(fontsize=7)
savefig(fig, "fig2_calibration")
# Figure 3: decision curves
fig, axs = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, c in zip(axs, COHORTS):
    sub = DCA[DCA.cohort == c]
    for m in ("M1", "M2", "CKINN"):
        g = sub[sub.model == m]
        ax.plot(g.threshold, g.net_benefit, LST[m], color=COL[m], label=m)
    g = sub[sub.model == "treat all"]
    ax.plot(g.threshold, g.net_benefit, color="#9b9a95", lw=1, label="treat all")
    ax.axhline(0, color="#0b0b0b", lw=0.8)
    top = float(sub[sub.model != "treat all"].net_benefit.max())
    ax.set_ylim(-0.25 * top, 1.1 * top)
    ax.set_title(f"{c}: decision curve")
    ax.set_xlabel("threshold probability")
    ax.set_ylabel("net benefit")
    ax.legend(fontsize=7)
savefig(fig, "fig3_decision_curves")
# Figure 4: AUROC as the laboratory panel is removed completely at random
fig, axs = plt.subplots(1, 3, figsize=(10, 3.2))
_mc = [t for t in SCEN if t.startswith("mcar")]
_x = [0.0] + [float(t[4:]) / 100 for t in _mc]
for ax, c in zip(axs, COHORTS):
    sub = MISS_AUC[MISS_AUC.cohort == c].set_index("model")
    for m in ("M1", "M2", "CKINN", "LR"):
        ax.plot(_x, [sub.loc[m, "none"]] + [sub.loc[m, t] for t in _mc], LST[m], marker=MRK[m], ms=4, color=COL[m], label=m)
    ax.set_title(f"{c}: AUROC, panel removed (MCAR)")
    ax.set_xlabel("share of people without the laboratory panel")
    ax.set_ylabel("AUROC (run-averaged)")
    ax.legend(fontsize=7)
savefig(fig, "fig4_missingness")
# Figure 5: CKINN minus M2 AUROC across seeds of repeat 0 and across the main runs (variance components)
fig, ax = plt.subplots(figsize=(6, 3))
for i, c in enumerate(COHORTS):
    y = Y[c]
    a = lambda p: MT.auroc_w(MT.Ranked(p, y))
    sd = [a(P[c][("CKINN", 0, s)]["none"]) - a(P[c][("M2", 0, s)]["none"]) for _, s in SEEDRUNS]
    rd = [a(P[c][("CKINN", r, s)]["none"]) - a(P[c][("M2", r, s)]["none"]) for r, s in MAINRUNS]
    ax.scatter(np.full(len(sd), i - 0.12), sd, marker="o", s=18, color=COL["M2"], label="seeds of repeat 0" if i == 0 else None, zorder=3)
    ax.scatter(np.full(len(rd), i + 0.12), rd, marker="s", s=18, color=COL["CKINN"], label="main runs (repeat r, seed r)" if i == 0 else None, zorder=3)
    lo, hi = AN.ci(BOOT_FAM[c]["draws"][("CKINN", "auroc")] - BOOT_FAM[c]["draws"][("M2", "auroc")])
    ax.plot([i + 0.3, i + 0.3], [lo, hi], color="#0b0b0b", lw=1.5)
    ax.plot([i + 0.3], [BOOT_FAM[c]["point"][("CKINN", "auroc")] - BOOT_FAM[c]["point"][("M2", "auroc")]], marker="D", ms=4, color="#0b0b0b",
            label="run average, 95% bootstrap interval" if i == 0 else None)
ax.axhline(0, color="#9b9a95", lw=0.8)
ax.axhline(-CFG.ni_margin, color="#9b9a95", lw=0.8, ls=":", label="non-inferiority margin")
ax.set_xticks(range(len(COHORTS)))
ax.set_xticklabels(COHORTS)
ax.set_ylabel("AUROC, CKINN minus M2")
ax.legend(fontsize=6, loc="best")
savefig(fig, "fig5_seed_and_split_variation")
# Figure 6: constraint variants, AUROC difference from CKINN (repeat 0)
fig, axs = plt.subplots(1, 3, figsize=(11, 4), sharex=True)
for ax, c in zip(axs, COHORTS):
    g = VARIANTS[VARIANTS.cohort == c].reset_index(drop=True)
    ax.errorbar(g.auroc_minus_ckinn, range(len(g)), xerr=[g.auroc_minus_ckinn - g.lo, g.hi - g.auroc_minus_ckinn], fmt="o", ms=3,
                color=COL["CKINN"], ecolor="#52514e", elinewidth=0.8)
    ax.axvline(0, color="#0b0b0b", lw=0.8)
    ax.set_yticks(range(len(g)))
    ax.set_yticklabels(g.variant, fontsize=6)
    ax.invert_yaxis()
    ax.set_title(f"{c}: variant minus CKINN (AUROC)")
savefig(fig, "fig6_constraint_variants")
assert all((FIG / f"{n}.{e}").exists() for n in ("fig1_hypothesis_endpoints", "fig2_calibration", "fig3_decision_curves", "fig4_missingness",
                                                  "fig5_seed_and_split_variation", "fig6_constraint_variants") for e in ("pdf", "png"))
print(f"CELL 36 OK | six figures saved as PDF and 300 dpi PNG | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 37 | LaTeX (booktabs) exports of the main tables; every value formatted from the stored CSV tables
_t0 = time.perf_counter()


def latex_table(df: pd.DataFrame, path: Path, caption: str, label: str, digits: int = 3) -> None:
    def f(v):
        if isinstance(v, (bool, np.bool_)):
            return "yes" if v else "no"
        if isinstance(v, (int, np.integer)):
            return f"{int(v):,}"
        if isinstance(v, (float, np.floating)):
            return "--" if not np.isfinite(v) else f"{v:.{digits}f}"
        return str(v).replace("_", r"\_").replace("%", r"\%").replace("&", r"\&").replace(">", r"$>$").replace("<", r"$<$")
    cols = [str(c).replace("_", r"\_") for c in df.columns]
    lines = [r"\begin{table}[ht]", r"\centering", r"\scriptsize", rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             r"\begin{tabular}{" + "l" * len(cols) + "}", r"\toprule", " & ".join(cols) + r" \\", r"\midrule"]
    lines += [" & ".join(f(v) for v in row) + r" \\" for row in df.itertuples(index=False)]
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    path.write_text("\n".join(lines) + "\n")


_rd = lambda n: pd.read_csv(OUT / "tables" / f"{n}.csv", keep_default_na=False, na_values=[""])
_perf = _rd("table08_performance")
latex_table(_perf[_perf.model.isin(["LR", "cLR", "GAM", "XGB_mono", "LGBM_mono", "M1", "M2", "CKINN"])][
    ["cohort", "model", "events", "auroc", "auroc_lo", "auroc_hi", "brier", "scaled_brier", "slope", "citl", "ece"]],
    OUT / "latex" / "table_performance.tex", "Run-averaged out-of-fold performance at tier T2 (5 repeats of 5-fold cross-validation).", "tab:perf")
_ep = _rd("table09_hypothesis_endpoints")
latex_table(_ep[_ep.primary.astype(str) == "True"][["family", "hypothesis", "unit", "endpoint", "difference", "ci_lo", "ci_hi", "p_pred_holm", "status"]],
            OUT / "latex" / "table_endpoints.tex", "Primary endpoints of hypothesis families A (CKINN minus M2) and B (CKINN minus M1).", "tab:endpoints", 4)
latex_table(_rd("table10_verdicts"), OUT / "latex" / "table_verdicts.tex", "Pre-specified verdicts (protocol v2, section 7).", "tab:verdicts")
latex_table(_rd("table11_h1_crosschecks")[["family", "cohort", "auroc_diff_runavg", "boot_lo", "boot_hi", "delong_p", "nb_lo", "nb_hi", "p_tost_margin_0.01"]],
            OUT / "latex" / "table_h1_crosschecks.tex", "Cross-checks of the AUROC comparison.", "tab:h1x", 4)
print(f"CELL 37 OK | LaTeX tables written | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 38 | generated results report (report.md): every number is formatted by code from a stored table
_t0 = time.perf_counter()
REP = OUT / "report"
T = {p.stem: pd.read_csv(p, keep_default_na=False, na_values=[""]) for p in sorted((OUT / "tables").glob("*.csv"))}


def _fmt(v, d=3):
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        return "--" if not np.isfinite(v) else f"{v:.{d}f}"
    return str(v)


def md_table(df: pd.DataFrame, d: int = 3) -> str:
    cols = list(df.columns)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    out += ["| " + " | ".join(_fmt(v, d) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(out)


def ci_str(est, lo, hi, d=4):
    return f"{est:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]"


_ep, _vd, _perf = T["table09_hypothesis_endpoints"], T["table10_verdicts"], T["table08_performance"]
_epp = _ep[_ep.primary.astype(str) == "True"]
verdict_lines = [f"- **{r.hypothesis} {r.verdict}** (family {r.family}): {r.units_predicted} of {r.units} units in the predicted direction after Holm "
                 f"correction, {r.units_opposite} in the opposite direction." for r in _vd.itertuples()]
h1_lines = []
for r in _ep[_ep.endpoint.str.startswith("AUROC (non")].itertuples():
    h1_lines.append(f"- {r.family}, {r.unit}: AUROC difference {ci_str(r.difference, r.ci_lo, r.ci_hi)}; Holm-adjusted one-sided p for "
                    f"non-inferiority {r.p_pred_holm:.4f}; status {r.status}.")
_main = _perf[_perf.runs == CFG.n_repeats]
best_lines = [f"- {c}: highest run-averaged AUROC {g.loc[g.auroc.idxmax(), 'model']} ({g.auroc.max():.4f}); lowest Brier "
              f"{g.loc[g.brier.idxmin(), 'model']} ({g.brier.min():.5f})." for c, g in _main.groupby("cohort")]
_x = T["table11_h1_crosschecks"]
_mde = _x[(_x.cohort == "NAGALA") & (_x.family.str.startswith("A"))].iloc[0]
mde_line = (f"For NAGALA, the bootstrap standard deviation of the CKINN minus M2 AUROC difference was {_mde.boot_sd:.4f}, so the difference detectable with "
            f"80% power at two-sided 5% is {_mde.mde_80pct_power:.4f}, and the power of the non-inferiority test (margin {CFG.ni_margin}) if the true "
            f"difference were zero is {_mde.power_noninferiority_if_true_diff_0:.3f}. Both ignore the Holm correction and the variability of model "
            f"training, so they overstate the power of the actual decision.")
_sens_cols = ["cohort", "model", "runs", "events", "auroc", "auroc_lo", "auroc_hi", "auprc", "brier", "scaled_brier", "slope", "citl", "ece"]
_dc0 = T["table01_data_card"].set_index("cohort")
LIMITS = [
    "Version 1 results were seen before every v2 decision; v2 is a re-analysis of the same development and evaluation rows, not a new test set. "
    "All amendments are listed in protocol_v2.md, section 2.",
    f"Task B labels are binary incident diabetes over follow-up windows that differ between cohorts (NAGALA {_dc0.loc['NAGALA', 'follow_up_min_years']:.1f} "
    f"to {_dc0.loc['NAGALA', 'follow_up_max_years']:.1f} years, RICH {_dc0.loc['RICH', 'follow_up_min_years']:.1f} to {_dc0.loc['RICH', 'follow_up_max_years']:.1f} "
    "years; AUDIT F15); transport differences mix population shift with design. The censoring-aware analyses address the horizon only in part.",
    "Cross-population evidence is from Japan and China; NHANES is a detection task, its extract is a non-random subset of the survey (AUDIT F14), and "
    "survey-weighted estimates do not represent the US population.",
    "RICH family history is probably under-recorded at several sites (AUDIT F16); RICH drinking status is unused (AUDIT F17).",
    "Missingness experiments remove values at test time; the training augmentation of M2, CKINN, LR_aug and XGB_aug matches the primary MCAR "
    "scenario, which favours those models against the other comparators (AUDIT F10).",
    "H4 is now measured by across-seed attribution stability, which CKINN is not trained for; the violation rates remain a manipulation check and "
    "are not evidence of clinical validity. Agreement with clinical priors is not proof that a model is causally correct.",
    "Subgroup gaps at one common threshold reflect differences in risk between groups; sensitivity at matched specificity isolates ranking within "
    "groups but subgroups with few events have wide intervals.",
    "A 'seed' changes the network initialisation, the inner validation split and the batch order together, so the seed variance component "
    "includes inner-split variance. M1, which has no augmentation, sees its mini-batches in a different order from M2 and CKINN after the first epoch.",
    "Lambda was chosen at the top of the pre-declared grid in every cohort and tier: the development score barely changed across the grid and the "
    "declared rule takes the largest lambda within one standard error, so CKINN here is a strongly constrained network.",
    "Bootstrap intervals hold the fitted models fixed; the corrected repeated-CV t-test, which allows for training variability, gives wider intervals. "
    "Intervals for the IPCW time-dependent AUROC hold the censoring distribution fixed.",
    "The post hoc analyses (label PH; section 12 of report.md) were decided after the v2 results were seen; they describe the models further and cannot support "
    "a confirmatory claim. The reasoning texts and SHAP values are model behaviour, not causal effects or advice.",
    "Outputs are risk estimates from retrospective research data; they are not diagnoses and have not been validated for clinical use.",
]
REPORT = f"""# CKINN versus M1 and M2 on real data, version 2: results report (generated)

Generated {_dt.date.today().isoformat()} in `{CFG.mode}` mode; configuration hash `{PROTOCOL_HASH[:16]}`. Every number below is formatted by code from the
stored tables in `tables/`; none was typed. Outputs are research risk estimates, not diagnoses. Labels: P1 pre-specified in v1, P2 pre-specified
in v2 (decided after seeing v1), E exploratory.

## 1. Data
{md_table(T["table01_data_card"][["cohort", "rows", "labelled", "positives", "prevalence_pct", "predictors_present"]], 2)}

NHANES label (amendment A11):

{md_table(T["table03_nhanes_label"])}

Development and evaluation rows:

{md_table(T["table04_split"])}

## 2. Pre-specified verdicts (P2)
{chr(10).join(verdict_lines)}

{md_table(_vd)}

H1 in detail (run-averaged AUROC difference, 95% paired-bootstrap interval, {CFG.n_boot:,} draws):

{chr(10).join(h1_lines)}

All primary endpoints (difference = CKINN minus comparator):

{md_table(_epp[["family", "hypothesis", "unit", "endpoint", "difference", "ci_lo", "ci_hi", "p_pred_holm", "p_opp_holm", "status"]], 4)}

## 3. Cross-checks of H1 and sources of variance (P2)
{md_table(_x[["family", "cohort", "auroc_diff_runavg", "boot_lo", "boot_hi", "delong_diff_run0", "delong_p", "nb_lo", "nb_hi", "nb_p", "p_tost_margin_0.01"]], 4)}

{mde_line}

{md_table(T["table12_variance_components"][T["table12_variance_components"].metric == "auroc"], 4)}

## 4. Discrimination and calibration of every model (P2)
{chr(10).join(best_lines)}

{md_table(_perf[_sens_cols], 4)}

CKINN minus each competitor (AUROC; descriptive, no multiplicity correction):

{md_table(T["table13_ckinn_vs_competitors"][T["table13_ckinn_vs_competitors"].metric == "auroc"], 4)}

## 5. Clinical utility (P2)
{md_table(T["table16_net_benefit_points"], 4)}

{md_table(T["table17_threshold_metrics"][T["table17_threshold_metrics"].model.isin(["M1", "M2", "CKINN", "LR", "XGB_mono"])], 3)}

## 6. Censoring-aware evaluation (P2) and 3-year IPCW models (E)
{md_table(T["table18_censoring_aware"], 4)}

{md_table(T["table19_ipcw_3y_models"], 4)}

## 7. Subgroups (E; H5 is in section 2)
{md_table(T["table21_subgroup_gaps"][T["table21_subgroup_gaps"].model.isin(["M2", "CKINN", "M1"])], 4)}

## 8. Missingness (P2)
{md_table(T["table22_missingness_auroc"], 4)}

{md_table(T["table23_missingness_loss_tests"], 4)}

RICH leave-one-site-out, pooled held-out rows:

{md_table(T["table26_rich_site_pooled"], 4)}

## 9. Constraint evidence (E; manipulation check P1)
{md_table(T["table27_manipulation_check"], 4)}

{md_table(T["table28_constraint_variants"][["cohort", "variant", "auroc_minus_ckinn", "lo", "hi", "brier_minus_ckinn", "violation_rate_v1_table", "violation_rate_own_table", "violation_rate_dropped_feature"]], 4)}

Empirical sign of each constrained predictor (adjusted logistic regression on the evaluation rows) and agreement of each model's held-out partial
dependence with the prior:

{md_table(T["table29_empirical_signs_vs_pd"][T["table29_empirical_signs_vs_pd"].model.isin(["M2", "CKINN"])][["cohort", "model", "feature", "prior_sign", "adjusted_coef", "coef_lo", "coef_hi", "adjusted_evidence", "pd_change_p10_to_p90", "pd_sign_agrees_with_prior"]], 4)}

{md_table(T["table30_violation_sparse_and_other_cohort"], 4)}

## 10. Transport (P2) and secondary transport analyses (E)
{md_table(T["table31_transport"], 4)}

{md_table(T["table32_transport_recalibration_reweighting"], 4)}

{md_table(T["table33_pooled_training"], 4)}

## 11. Sensitivity analyses (E)
{md_table(T["table34_sensitivity"], 4)}

## 12. Post hoc additions for the 28-step protocol (PH; decided after the v2 results, no verdict depends on them)
Refits that repeat a main-run fit (largest absolute difference from the stored predictions; 0 means identical):

{md_table(T["tablePH02_refit_reproducibility"], 6)}

SVM (radial-kernel approximation, Platt scaling), run-averaged, and CKINN minus SVM:

{md_table(T["tablePH03_svm_performance"], 4)}

Violation rate for every model (log-odds change against the prior after +0.25 SD or a 0-to-1 change; Wilson intervals):

{md_table(T["tablePH05_violation_all_models"], 4)}

BMI set to 24 and to 35 in the same held-out people: share whose predicted risk is lower at 35:

{md_table(T["tablePH06_bmi_24_35_pairs"], 4)}

Variants (each against the same model fitted without the change, repeat 0):

{md_table(T["tablePH07_variants"][["cohort", "variant", "model", "auroc_diff", "auroc_diff_lo", "auroc_diff_hi", "brier_diff", "citl_diff", "tpr_sex_gap_diff", "tpr_age_range_diff"]], 4)}

Pooled NAGALA plus RICH training with a cohort indicator:

{md_table(T["tablePH08_pooled_cohort_indicator"], 4)}

Train-by-test AUROC matrix (seed 0):

{md_table(T["tablePH09_transport_matrix"], 4)}

Intercept-only recalibration in the target: the intercept is estimated on a random sample of {CFG.recal_n:,} target participants (target labels used; the same sample for every model) and evaluated on the remaining target participants. With few events in that sample the estimated shift is imprecise, so calibration-in-the-large after recalibration need not be near zero; the calibration-sample columns show why.

{md_table(T["tablePH10_intercept_recalibration"], 4)}

Older and lean subgroup (age 65 or more, BMI below 20) across lambda; not estimable below 20 events:

{md_table(T["tablePH11_stress_older_lean"], 4)}

RQ7, participants with baseline fasting glucose below 5.6 mmol/L:

{md_table(T["tablePH12_rq7_fpg_below_5_6"], 4)}

RICH one-step progression (baseline FPG below 5.6 mmol/L; outcome FPG of 5.6 mmol/L or more at the final visit, or diabetes):

{md_table(T["tablePH13_rich_progression"], 4)}

SHAP direction against the priors (number and share of constrained predictors whose SHAP dependence has the prior's sign; per predictor in table PH16) and agreement with integrated gradients:

{md_table(T["tablePH23_shap_direction_summary"], 3)}

{md_table(T["tablePH17_shap_vs_integrated_gradients"], 3)}

Patient-level reasoning text for CKINN (texts are row-level and stay in the local cache):

{md_table(T["tablePH18_reasoning_summary"], 3)}

Scorecard (one row per cohort and model):

{md_table(T["tablePH22_scorecard"], 4)}

## 13. Limitations
{chr(10).join('- ' + x for x in LIMITS)}
"""
(REP / "report.md").write_text(REPORT, encoding="utf-8")
print(REPORT[:3000])
print(f"CELL 38 OK | report.md written ({len(REPORT):,} characters) | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 39 | manuscript-ready Methods and Results draft (British spelling); numbers formatted from stored tables only
_t0 = time.perf_counter()
_dc, _sp = T["table01_data_card"].set_index("cohort"), T["table04_split"].set_index("cohort")
_lam = T["table07_lambda"]
_lam_ch = _lam[_lam.chosen.astype(str) == "True"]


def _ep_sentence(fam_prefix: str, c: str) -> str:
    g = _ep[(_ep.family.str.startswith(fam_prefix)) & (_ep.unit == f"{c} (T2)")].set_index("endpoint")
    a = g.loc["AUROC (non-inferiority, margin 0.01)"]
    b = g.loc["Brier score"]
    return (f"{c}: AUROC difference {a.difference:.4f} (95% CI {a.ci_lo:.4f} to {a.ci_hi:.4f}; non-inferiority {a.status}), "
            f"Brier difference {b.difference:.5f} ({b.ci_lo:.5f} to {b.ci_hi:.5f})")


def _v(fam_prefix, h):
    r = _vd[(_vd.family.str.startswith(fam_prefix)) & (_vd.hypothesis == h)].iloc[0]
    return r.verdict


_tr = T["table31_transport"]
_tr_lines = [f"{r.source} to {r.target}, {r.model}: target cross-validated AUROC {r.auroc_target_cv:.3f}, external {r.auroc_external:.3f}, gap "
             f"{r.transport_gap:.3f}" for r in _tr[_tr.model.isin(["M1", "M2", "CKINN"])].itertuples()]
METHODS = f"""# Methods and Results (draft generated from stored tables)

_Draft for the manuscript. British (Oxford) spelling. Every number in this file is formatted by code from the tables of the v2 run (mode `{CFG.mode}`, configuration hash `{PROTOCOL_HASH[:16]}`). References marked TODO(cite) must be checked before submission._

## Methods

**Design and amendments.** This is a re-analysis (version 2) of a study whose version 1 results had been seen. All changes were fixed in a protocol before any version 2 result existed; each is listed with its reason in the amendment log (protocol_v2.md, section 2), and analyses are labelled as pre-specified in version 1, pre-specified in version 2, or exploratory.

**Data.** Three public, de-identified datasets were used: the NAGALA cohort (Japan; {int(_dc.loc['NAGALA', 'rows']):,} participants, {int(_dc.loc['NAGALA', 'positives']):,} incident diabetes events), the Rich Healthcare Group cohort (China, 15 sites; {int(_dc.loc['RICH', 'rows']):,} participants, {int(_dc.loc['RICH', 'positives']):,} events) and an NHANES extract (USA, cycles 1999 to 2018), in which undiagnosed diabetes was defined as fasting plasma glucose of at least 7.0 mmol/L after at least 8 hours of fasting or HbA1c of at least 6.5% among adults reporting no diagnosis and no glucose-lowering medication ({int(_dc.loc['NHANES', 'labelled']):,} labelled adults, {int(_dc.loc['NHANES', 'positives']):,} positive). Predictors were routine clinical and laboratory measurements without glucose or HbA1c (tier T2); a tier adding them (T3) was a sensitivity analysis in the two cohorts. Exact zeros in laboratory values were treated as not measured. In each cohort {CFG.dev_frac:.0%} of participants (stratified) formed a development subset used only for tuning; the remainder formed the evaluation subset.

**Models.** M1 is a multilayer network with per-group encoders in which missing values take the training mean; M2 is the same network with the missingness mask as input and training-time missingness augmentation; CKINN is M2 with a penalty on input derivatives whose sign contradicts a fixed clinical sign table (risk increasing with age, body-mass index, waist, systolic pressure, family history and triglycerides; decreasing with HDL cholesterol and regular exercise). Within a fold, M1, M2 and CKINN shared the seed, the inner validation split and the training rows, so that CKINN with lambda 0 is identical to M2. Competitors were L2 logistic regression, sign-constrained logistic regression, a spline additive logistic model, random forest, XGBoost and LightGBM with and without monotone constraints from the same sign table, and multiple imputation by chained equations ({CFG.mi_m} imputations) followed by logistic regression.

**Tuning.** On the development subset, M1 and M2 each evaluated {CFG.tune_mlp_configs} configurations; CKINN took M2's best configuration among the first {CFG.tune_ck_base} and evaluated {len(CFG.lam_grid)} values of lambda, the same budget of {CFG.tune_mlp_configs} evaluations. Lambda was the largest value whose development score was within one standard error of the best score among lambda 0 and the grid (declared before any version 2 result).

**Evaluation.** Each model was evaluated by {CFG.n_repeats} repeats of stratified {CFG.n_splits}-fold cross-validation on the evaluation subset, with network seed r in repeat r; in repeat 0 the networks were also trained with {CFG.n_seeds - 1} further seeds. Metrics were computed on the pooled out-of-fold predictions of each run and averaged over runs. Uncertainty came from a paired participant bootstrap ({CFG.n_boot:,} draws) applied identically to all runs and models; one-sided p-values were read from the draws. DeLong's test on one run, the corrected resampled t-test over the {CFG.n_repeats * CFG.n_splits} folds and, in the Rich cohort, a cluster bootstrap over sites were cross-checks. Two hypothesis families (CKINN minus M2, primary; CKINN minus M1) were each corrected by Holm's method across their primary endpoints: H1 AUROC non-inferiority (margin {CFG.ni_margin}); H2 Brier score and distance of the calibration slope from 1; H3 the transport gap (AUROC of the same model class cross-validated inside the target cohort minus the AUROC of the source-trained model applied unchanged) and absolute calibration-in-the-large in the target; H4 the agreement of integrated-gradient attributions across seeds; H5 the sex gap and age-band range of sensitivity at a false-positive rate of {CFG.fpr_matched}. Secondary analyses covered calibration (slope, calibration-in-the-large, expected calibration error and reliability curves), decision curves, censoring-aware metrics (Harrell's and Uno's C and the inverse-probability-of-censoring weighted AUROC at {CFG.horizon:g} years), subgroup metrics, missingness injected at test time, leave-one-site-out validation, constraint ablations with wrong-sign controls, empirical sign checks against held-out partial dependence, and transport with recalibration, reweighting and pooled training. TODO(cite) for each method.

## Results

**Verdicts.** In family A (CKINN minus M2), H1 was {_v('A', 'H1')}, H2 {_v('A', 'H2')}, H3 {_v('A', 'H3')}, H4 {_v('A', 'H4')} and H5 {_v('A', 'H5')}. In family B (CKINN minus M1), H1 was {_v('B', 'H1')}, H2 {_v('B', 'H2')}, H3 {_v('B', 'H3')}, H4 {_v('B', 'H4')} and H5 {_v('B', 'H5')}.

**CKINN versus M2.** {'; '.join(_ep_sentence('A', c) for c in COHORTS)}.

**CKINN versus M1.** {'; '.join(_ep_sentence('B', c) for c in COHORTS)}.

**Best-performing models.** {' '.join(x[2:] for x in best_lines)}

**Power.** {mde_line}

**Transport.** {'; '.join(_tr_lines)}.

## Limitations
{chr(10).join('- ' + x for x in LIMITS)}
"""
(REP / "methods_results_draft.md").write_text(METHODS, encoding="utf-8")
print(METHODS[:2500])
print(f"CELL 39 OK | Methods and Results draft written | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 40 | TRIPOD+AI checklist filled from what the code did (item wording abbreviated; check the numbering against the official
# checklist before submission: TODO(verify))
_t0 = time.perf_counter()
TRIPOD = [
    ("Title and abstract", "Identify the study as developing and evaluating prediction models; summarise", "To be written; report.md and methods_results_draft.md give the numbers"),
    ("Background and objectives", "Rationale, intended use, objectives", "Proposal and protocol_v2.md section 1; outputs are research risk estimates, not diagnoses"),
    ("Data sources", "Source and dates of data", f"NAGALA, Rich Healthcare Group, NHANES 1999-2018 extract; file hashes in manifest.json"),
    ("Participants", "Eligibility, setting, treatments", "Cell 17; NHANES exclusions (A11) in table03; no rows removed in NAGALA or RICH"),
    ("Data preparation", "Pre-processing and quality checks", "AUDIT.md; cell 17; zeros treated as missing (A12); winsorising, log and z-score fitted in training folds"),
    ("Outcome", "Definition, timing, blinding", "Incident diabetes as recorded by each cohort; NHANES laboratory definition; follow-up windows differ (AUDIT F15)"),
    ("Predictors", "Definition, timing, measurement", "table05 feature sets; T2 excludes glucose and HbA1c; T3 sensitivity"),
    ("Sample size", "Justification", "Fixed by the data; table04 events; minimum detectable effect in table11"),
    ("Missing data", "Handling", "Mask and augmentation (M2, CKINN); mean fill (M1); native handling (trees); multiple imputation baseline; tables 22-24"),
    ("Analytical methods", "Model type, building, tuning, internal validation", "protocol_v2.md sections 4-6; tables 06-07; repeated CV with paired seeds"),
    ("Class imbalance", "Methods used", "None (no resampling); output bias initialised at the training prevalence"),
    ("Fairness", "Approaches to fairness", "H5 and tables 20-21 (sex, age band, site, ethnic group); minimum-events rule"),
    ("Model output", "Form of predictions", "Probabilities; thresholds from inner validation splits for operating points"),
    ("Training versus evaluation", "Differences between data used", "Development rows tune only; evaluation rows by cross-validation; transport and site analyses external"),
    ("Performance measures", "Metrics and their uncertainty", "AUROC, AUPRC, Brier, scaled Brier, slope, calibration-in-the-large, ECE, net benefit, censoring-aware metrics; bootstrap intervals"),
    ("Model updating / recalibration", "If done", "Recalibration in the small in the target (table32, target labels used, exploratory)"),
    ("Ethics", "Approval and consent", "Public de-identified data; check each source's terms before submission: TODO(verify)"),
    ("Funding and conflicts", "Sources and roles", "To be completed by the authors"),
    ("Protocol and registration", "Where available", "protocol_v2.md with amendment log and configuration hash; not registered"),
    ("Data and code availability", "Access", "Data from public sources; this notebook and library (ckv2) reproduce every table"),
    ("Patient and public involvement", "Details", "None"),
    ("Participant flow and characteristics", "Numbers, events, missingness", "tables 01-04"),
    ("Model specification", "Full model or how to obtain it", "Configuration and chosen hyperparameters in manifest.json; code in ckv2"),
    ("Model performance", "With intervals, by subgroup", "tables 08-34; figures 1-6"),
    ("Interpretation and limitations", "Overall interpretation, limitations, usability", "report.md section 12; methods_results_draft.md"),
]
TRIPOD_MD = "# TRIPOD+AI checklist (filled from what the code did)\n\nItem wording is abbreviated and the numbering must be checked against the official TRIPOD+AI checklist before submission: TODO(verify).\n\n" + \
    md_table(pd.DataFrame(TRIPOD, columns=["Section", "Item (abbreviated)", "Where addressed in this study"])) + "\n"
(REP / "TRIPOD_AI_checklist.md").write_text(TRIPOD_MD, encoding="utf-8")
print(f"CELL 40 OK | TRIPOD+AI checklist written | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 41 | integrity scan: configuration hash; every decimal number in the report and the draft traces to a stored table or to the
# configuration; no result-like number in this notebook's markdown cells; no row-level file among the archived outputs
_t0 = time.perf_counter()
_fz = json.loads((OUT / "report" / "frozen_protocol_v2.json").read_text())
assert _fz["sha256"] == C.config_hash(CFG) == PROTOCOL_HASH, "configuration changed after it was frozen"
_vals = np.concatenate([t.select_dtypes(include=[float, int]).to_numpy(float).ravel() for t in T.values()])
_cfgv = [float(v) for v in json.loads(CFG.to_json()).values() if isinstance(v, (int, float)) and not isinstance(v, bool)]
_cfgv += [float(x) for v in json.loads(CFG.to_json()).values() if isinstance(v, list) for x in v if isinstance(x, (int, float))]
_cfgv += [float(v) for v in PH.settings(CFG).values()]                  # post hoc settings quoted in the report text
_vals = np.concatenate([_vals[np.isfinite(_vals)], np.array(_cfgv)])
_untraced = []
for name, text in (("report.md", REPORT), ("methods_results_draft.md", METHODS)):
    body = re.sub(r"`[^`]*`", "", text)
    for mm in re.finditer(r"(?<![\w.])-?\d+\.(\d{2,})(?![\w.])", body):
        tok, d = float(mm.group(0)), len(mm.group(1))
        if not np.any(np.abs(np.round(_vals, d) - tok) < 10 ** -(d + 2)):
            _untraced.append((name, mm.group(0)))
assert not _untraced, f"numbers that no stored table supports: {_untraced[:10]}"
_nb_path = next((p for p in [Path(os.environ.get("CKINN_NOTEBOOK", "")), *Path(".").glob("CKINN_vs_MLP_real_data_final*.ipynb")] if p.is_file()), None)
if _nb_path is not None:
    _md = [c for c in json.loads(_nb_path.read_text())["cells"] if c["cell_type"] == "markdown"]
    _bad = [x for c in _md for x in re.findall(r"\d\.\d{3,}", "".join(c["source"]))]
    assert not _bad, f"result-like numbers in markdown cells: {_bad[:5]}"
    print(f"markdown scan of {_nb_path.name}: {len(_md)} markdown cells, no result-like numbers")
else:
    print("notebook file not found in the working folder: markdown scan skipped (set CKINN_NOTEBOOK to its path)")
_archived = [p for p in OUT.rglob("*") if p.is_file() and CACHE not in p.parents]
assert not [p for p in _archived if p.suffix in (".npz", ".pkl", ".xlsx") or p.name.startswith("_raw_")], "row-level file outside the cache"
print(f"CELL 41 OK | {len(_untraced)} untraced numbers; configuration hash intact | {time.perf_counter() - _t0:.0f}s")

In [ ]:
# CELL 42 | runtime, manifest with package versions, input and output hashes, and the outputs archive (aggregate files only)
_t0 = time.perf_counter()
_rt = []
for stage, specs in (("tuning", [s for v in _specs.values() for s in v] + [s for v in _lspecs.values() for s in v]),
                     ("main runs and seed study", [s for c in CV_SPECS for s in CV_SPECS[c] + SEED_SPECS[c]]), ("repeat-0 analyses", _all),
                     ("transport, pooled, sites", TR_SPECS + POOL_SPECS + SITE_SPECS), ("post hoc (PH)", PH_ALL_SPECS)):
    secs = [EN.load(CACHE, s["id"])["seconds"] for s in specs]
    _rt.append({"stage": stage, "tasks": len(specs), "cpu_seconds": float(np.sum(secs)), "median_task_seconds": float(np.median(secs))})
RUNTIME = pd.DataFrame(_rt)
RUNTIME.to_csv(OUT / "tables" / "table35_runtime.csv", index=False)
print(RUNTIME.round(1).to_string(index=False))
MAN = {"created_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds"), "mode": CFG.mode, "configuration_sha256": PROTOCOL_HASH,
       "configuration": json.loads(CFG.to_json()), "inputs": {k: {"file": FILES[k].name, "sha256": FILE_HASH[k]} for k in FILES},
       "environment": VERSIONS, "workers": N_WORKERS, "wall_seconds": time.perf_counter() - T_START,
       "chosen_hyperparameters": {f"{k[0]}|{k[1]}": v for k, v in HP.items()}, "chosen_lambda": {f"{k[0]}|{k[1]}": v for k, v in LAM.items()},
       "verdicts": T["table10_verdicts"].to_dict("records"), "held_out_sites": SITES_HELD, "shared_predictors": SHARED,
       "library_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((LIB / "ckv2").glob("*.py"))}}
(REP / "manifest.json").write_text(json.dumps(MAN, indent=2, default=str))
_files = sorted(p for p in OUT.rglob("*") if p.is_file() and CACHE not in p.parents and p.suffix != ".zip" and p.name != "output_hashes.json")
(REP / "output_hashes.json").write_text(json.dumps({str(p.relative_to(OUT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in _files}, indent=1))
with zipfile.ZipFile(OUT / f"ckinn_v2_outputs_{CFG.mode}.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in _files + [REP / "output_hashes.json"]:
        z.write(p, p.relative_to(OUT))
print(f"{len(_files)} files archived in {OUT / f'ckinn_v2_outputs_{CFG.mode}.zip'} (cache with row-level data excluded)")
print(f"CELL 42 OK | manifest and archive written | total wall time {time.perf_counter() - T_START:.0f}s")

## References and citation policy
Only sources checked in PubMed or Scite are listed without a flag; anything else carries TODO(cite) and must be checked before submission. The verified list is kept in `references.md` next to this notebook.